#

<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); border-radius: 12px; padding: 16px 24px;">
  <h1 style="color: #e0aaff; font-size: 1.9em; margin: 0 0 6px;">BTCUSDT 1 jam · iTransformer, Transformer, Ridge</h1>
  <p style="color: #ddd6fe; margin: 0 0 6px;">Walk-forward 15 origin × K ∈ {1, 4, 8, 12} × 5 seed × 3 model = 900 run pada horizon 24 jam, dibandingkan dengan Naive-RW. Kode arsitektur disalin dari repositori resmi pada commit terkunci; training berjalan di Kaggle GPU T4 × 2, satu run per GPU.</p>
  <p style="color: #ddd6fe; margin: 0;">Kaggle: lampirkan dataset <code>BTCUSDT_1h.parquet</code>, pilih <b>GPU T4 × 2</b>, isi <code>WEEKLY_GPU_HOURS_REMAINING</code> di sel setup, lalu <b>Save Version → Save &amp; Run All</b>.</p>
</div>

**Daftar isi**

- [01 · Persiapan lingkungan dan konfigurasi](#section-01)
- [02 · Muat data dan audit kualitas](#section-02)
- [03 · Feature engineering dan eksplorasi](#section-03)
- [04 · Split walk-forward, scaling, dan K_eff](#section-04)
- [05 · Model, baseline, dan fungsi training](#section-05)
- [06 · Persiapan evaluasi dan eksekutor](#section-06)
- [07 · Pemeriksaan sebelum training](#section-07)
- [08 · Validasi pilot dan rencana sesi](#section-08)
- [09 · Training grid walk-forward](#section-09)
- [10 · Evaluasi model dan research questions](#section-10)
- [11 · Simpan hasil, tabel, dan figure](#section-11)
- [12 · Lampiran — sinkronisasi lokal](#section-12)

##

<a id="section-01"></a>

<div style="background: linear-gradient(135deg, #0b1021, #14213d); border-left: 4px solid #8ecae6; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #8ecae6; margin: 0 0 4px; font-size: 1.35em;">🔧 01 · Persiapan lingkungan dan konfigurasi</h2>
  <p style="color: #a8c7d8; margin: 0; font-size: 0.95em;">Sesi Kaggle GPU T4 × 2, dataset input, impor bersama, dan konfigurasi penelitian.</p>
</div>

###

<div style="background: linear-gradient(90deg, #231400, #3a2200); border-left: 3px solid #f48c06; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #ffba08; font-size: 1em; margin: 0;">🗺️ Peta artefak</h3> <p style="display: inline; color: #ffd08a; font-size: 0.9em; margin: 0;">· Sel mana menulis apa, dari metadata tiap sel langkah.</p></div>

In [1]:
_ARTIFACT_MAP = [
    (3, 'artifact_map', [], []),
    (5, 'setup', [], ['data/raw/BTCUSDT_1h.parquet']),
    (20, 'data', [], ['data/raw/BTCUSDT_1h.parquet']),
    (25, 'features', [], []),
    (34, 'keff', ['artifacts/keff_table.parquet'], []),
    (53, 'upstream', [], ['vendor/thuml_iTransformer/**']),
    (70, 'code_digest', [], []),
    (72, 'invariants', ['artifacts/naive_rw_by_origin.parquet'], []),
    (74, 'pilot', ['artifacts/validation/*.json', 'artifacts/pilot.json'], []),
    (76, 'grid', ['artifacts/preds/*.parquet', 'artifacts/meta/*.json', 'artifacts/weights/*.pt', 'artifacts/checkpoints/*.pt', 'artifacts/session_status.json'], ['data/raw/BTCUSDT_1h.parquet']),
    (79, 'evaluate', [], ['artifacts/preds/*.parquet', 'artifacts/meta/*.json']),
    (81, 'contrasts', [], []),
    (83, 'research_questions', [], []),
    (85, 'direction', [], []),
    (87, 'report', ['paper/paper_numbers.json', 'paper/tables/*.tex', 'paper/figures/*.pdf', 'paper/figures/*.png', 'paper/panels/*.parquet'], []),
    (89, 'sync_back', [], []),
]

print("ARTIFACT MAP: cell, step, writes, reads")
for _i, _slug, _w, _r in _ARTIFACT_MAP:
    print(f"  cell {_i:>3}  {_slug}")
    for _p in _w:
        print(f"            writes {_p}")
    for _p in _r:
        print(f"            reads  {_p}")


ARTIFACT MAP: cell, step, writes, reads
  cell   3  artifact_map
  cell   5  setup
            reads  data/raw/BTCUSDT_1h.parquet
  cell  20  data
            reads  data/raw/BTCUSDT_1h.parquet
  cell  25  features
  cell  34  keff
            writes artifacts/keff_table.parquet
  cell  53  upstream
            reads  vendor/thuml_iTransformer/**
  cell  70  code_digest
  cell  72  invariants
            writes artifacts/naive_rw_by_origin.parquet
  cell  74  pilot
            writes artifacts/validation/*.json
            writes artifacts/pilot.json
  cell  76  grid
            writes artifacts/preds/*.parquet
            writes artifacts/meta/*.json
            writes artifacts/weights/*.pt
            writes artifacts/checkpoints/*.pt
            writes artifacts/session_status.json
            reads  data/raw/BTCUSDT_1h.parquet
  cell  79  evaluate
            reads  artifacts/preds/*.parquet
            reads  artifacts/meta/*.json
  cell  81  contrasts
  cell  83  research_questi

###

<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af2; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #e0aaff; font-size: 1em; margin: 0;">🧰 Setup sesi</h3> <p style="display: inline; color: #cbb2e8; font-size: 0.9em; margin: 0;">· Kuota, 2 × T4, folder kerja, dan parquet input dicari di kedalaman berapa pun.</p></div>

<div style="background: #0e0e12; border-left: 3px solid #6c757d; border-radius: 0 6px 6px 0; padding: 4px 14px; font-size: 0.82em; color: #cfcfcf;"><span style="color: #9ec5fe; font-weight: 600;">MEMBACA</span> <code>data/raw/BTCUSDT_1h.parquet</code></div>

In [2]:
import os
import subprocess
import sys
import time
from pathlib import Path

# The Kaggle session wall runs from this cell, so the session budget does too.
SESSION_T0 = globals().get("SESSION_T0", time.perf_counter())

WEEKLY_GPU_HOURS_REMAINING = 30  # from Kaggle's quota meter, before Run All
SESSION_ALREADY_USED_H = 0.0       # hours this session ran before the first cell
SESSION_LIMIT_H = 11.5             # below the 12-hour session wall
SAVE_RESERVE_H = 0.75              # kept free for saving the output
ANALYSIS_ONLY = False              # True renders a complete saved grid without training

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = (Path("/kaggle/working") if ON_KAGGLE else Path.cwd()).resolve()
ARTIFACTS = WORK / "artifacts"
VENDOR = WORK / "vendor" / "thuml_iTransformer"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
for _package in ("layers", "model", "utils"):
    (VENDOR / _package).mkdir(parents=True, exist_ok=True)
    (VENDOR / _package / "__init__.py").write_text("", encoding="utf-8")
if Path.cwd().resolve() != WORK:
    os.chdir(WORK)


def ensure(module: str, pip_name: str | None = None) -> None:
    """Install a package only when the image lacks it; the image's versions win."""
    try:
        __import__(module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or module])


for _module, _pip in (("polars", None), ("pyarrow", None), ("numpy", None), ("torch", None),
                      ("sklearn", "scikit-learn"), ("scipy", None), ("statsmodels", None),
                      ("arch", None), ("matplotlib", None)):
    ensure(_module, _pip)

if not ON_KAGGLE:
    raise RuntimeError("This notebook runs on Kaggle; locally, run the pytest suite instead.")
if not ANALYSIS_ONLY:
    if WEEKLY_GPU_HOURS_REMAINING is None:
        raise ValueError("Set WEEKLY_GPU_HOURS_REMAINING from Kaggle's quota meter, then Run All.")
    if not 0 <= float(WEEKLY_GPU_HOURS_REMAINING) <= 30 or not 0 <= SESSION_ALREADY_USED_H < 12:
        raise ValueError("Use a remaining quota in [0, 30] h and an elapsed session time in [0, 12) h.")
    import torch
    _gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
    if len(_gpus) != 2 or not all("T4" in name for name in _gpus):
        raise RuntimeError(f"Select the GPU T4 x2 accelerator before training. Detected: {_gpus}")


def looks_like_parquet(path: Path) -> bool:
    """True only for a file that starts and ends with the parquet magic ``PAR1``."""
    try:
        with path.open("rb") as handle:
            if handle.read(4) != b"PAR1":
                return False
            handle.seek(-4, 2)
            return handle.read(4) == b"PAR1"
    except OSError:
        return False


def find_parquet() -> Path:
    """Find BTCUSDT_1h.parquet under the inputs at any depth, data/raw/ copies first.

    Discovery is by file name, never by dataset slug, and the file is never
    downloaded here: a new download would be a different input.
    """
    patterns = ("BTCUSDT_1h.parquet", "*/data/raw/BTCUSDT_1h.parquet",
                "BTCUSDT_1h.parquet", "*/BTCUSDT_1h.parquet", "*/*/BTCUSDT_1h.parquet")
    roots = [WORK, Path("/kaggle/input")] if ON_KAGGLE else [WORK, WORK.parent]
    rejected: list[str] = []

    def accept(candidate: Path):
        if candidate.is_file() and looks_like_parquet(candidate):
            return candidate.resolve()
        rejected.append(str(candidate))
        return None

    for root in roots:
        if not root.exists():
            continue
        for pattern in patterns:
            for hit in sorted(root.glob(pattern)):
                if (found := accept(hit)) is not None:
                    return found
    for root in roots:
        if not root.exists():
            continue
        hits = [h for h in sorted(root.rglob("BTCUSDT_1h.parquet")) if h.is_file()]
        valid = [h for h in hits if looks_like_parquet(h)]
        rejected += [str(h) for h in hits if h not in valid]
        if valid:
            preferred = [h for h in valid if h.parent.name == "raw"]
            chosen = (preferred or valid)[0]
            if len(valid) > 1:
                print(f"note: {len(valid)} copies of BTCUSDT_1h.parquet under {root}; using {chosen}")
            return chosen.resolve()
    if rejected:
        raise FileNotFoundError(
            "found candidates but none passed the PAR1 magic check at both ends: "
            f"{rejected}. A truncated upload, a Git LFS pointer, or the wrong file under the right name.")
    raise FileNotFoundError(
        f"BTCUSDT_1h.parquet not found under {[str(r) for r in roots]}. Attach data/raw/ as a "
        "Kaggle Dataset; it is not downloaded here, because a new download is a different input.")


PARQUET = find_parquet()
os.environ["ITBTC_PARQUET"] = str(PARQUET)

import numpy as np
import polars as pl
import torch

print(f"work      {WORK}")
print(f"parquet   {PARQUET}  ({PARQUET.stat().st_size / 1e6:.1f} MB)")
print(f"polars {pl.__version__} | torch {torch.__version__} | numpy {np.__version__}")
print(f"CUDA devices: {torch.cuda.device_count()}")
for _i in range(torch.cuda.device_count()):
    _cap = torch.cuda.get_device_capability(_i)
    print(f"  cuda:{_i}  {torch.cuda.get_device_name(_i)}  sm_{_cap[0]}{_cap[1]}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 47.6 MB/s eta 0:00:00
work      /kaggle/working
parquet   /kaggle/input/datasets/akmaleyzal/btcusdt-1h/BTCUSDT_1h.parquet  (6.7 MB)
polars 1.35.2 | torch 2.10.0+cu128 | numpy 2.0.2
CUDA devices: 2
  cuda:0  Tesla T4  sm_75
  cuda:1  Tesla T4  sm_75


###

<div style="background: linear-gradient(90deg, #101010, #1c1c1c); border-left: 3px solid #9e9e9e; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #d0d0d0; font-size: 1em; margin: 0;">📚 Library</h3> <p style="display: inline; color: #a8a8a8; font-size: 0.9em; margin: 0;">· Semua impor tingkat modul, sekali untuk seluruh notebook.</p></div>

In [3]:
from __future__ import annotations
import argparse
import ast
import contextlib
import gc
import hashlib
import importlib
import json
import math
import numpy as np
import os
import platform
import polars as pl
import queue
import random
import re
import shutil
import subprocess
import sys
import threading
import time
import torch
import warnings
from collections import OrderedDict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timedelta, timezone
from pathlib import Path
from torch import Tensor, nn
from types import SimpleNamespace
from typing import Final, Literal, Protocol


###

<div style="background: linear-gradient(90deg, #0b1021, #14213d); border-left: 3px solid #8ecae6; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #8ecae6; font-size: 1em; margin: 0;">📐 <code>config.py</code></h3> <p style="display: inline; color: #a8c7d8; font-size: 0.9em; margin: 0;">· Kontrak data, protokol walk-forward, 15 origin, dan sumber tiap algoritma.</p></div>

In [4]:
"""Design constants, the walk-forward origin grid, and where each algorithm came from.

Every number here is fixed before any model runs, so no magic number is buried
in pipeline code. The origin grid is derived from the constants rather than
written out.
"""

# -- data window ---------------------------------------------------------------

DATA_START: Final = datetime(2018, 1, 1, tzinfo=timezone.utc)
DATA_END: Final = datetime(2026, 8, 1, tzinfo=timezone.utc)  # EXCLUSIVE

BARS_EXPECTED: Final = 75_216
BARS_ACTUAL: Final = 75_094
MISSING_BARS: Final = 122
GAP_BLOCKS: Final = 27

#: sha256 of ``data/raw/BTCUSDT_1h.parquet``, the one input the study reads.
INPUT_SHA256: Final = "8270a84b07c2923bc885782a8ba4e1898133d18ee3b260f157fcee3fd6923b4e"

# -- window geometry -----------------------------------------------------------

SEQ_LEN: Final = 96   # L: four days of lookback
PRED_LEN: Final = 24  # H: one day ahead

#: A window spans ``L + H`` bars, so one break removes ``L + H - 1`` start positions.
WINDOW_SPAN: Final = SEQ_LEN + PRED_LEN         # 120
STARTS_LOST_PER_BREAK: Final = WINDOW_SPAN - 1  # 119

# -- walk-forward protocol -----------------------------------------------------

TRAIN_MONTHS: Final = 24      # fixed rolling window, never expanding
VAL_MONTHS: Final = 3         # final 3 months of the training window
TRAIN_SUB_MONTHS: Final = TRAIN_MONTHS - VAL_MONTHS  # 21: where the scaler is fitted
TEST_BLOCKS: Final = 6
BLOCK_DAYS: Final = 30
BLOCK_HOURS: Final = BLOCK_DAYS * 24  # 720 forecast origins per block

#: Five months between origins. A spacing coprime to 12 keeps the test block
#: index from tracking the calendar month; 5 gives the most origins among those.
ORIGIN_SPACING_MONTHS: Final = 5

FIRST_ORIGIN: Final = datetime(2020, 1, 1, tzinfo=timezone.utc)

# -- ladder, seeds and the training sample ------------------------------------------

K_LADDER: Final = (1, 4, 8, 12)
SEEDS: Final = (42, 43, 44, 45, 46)

#: Training windows drawn per run, identical for all three models at a cell.
TRAIN_WINDOW_LIMIT: Final = 11_500
SELECTION_SEED: Final = 1729


def add_months(when: datetime, months: int) -> datetime:
    """Shift a first-of-month datetime by whole calendar months.

    Raises:
        ValueError: If ``when`` is not on the first of a month; clamping a day
            would move a split silently.
    """
    if when.day != 1:
        raise ValueError(
            f"add_months is only used on month boundaries in this study; got "
            f"day={when.day}. Clamping rules would silently move a split."
        )
    total = when.month - 1 + months
    return when.replace(year=when.year + total // 12, month=total % 12 + 1)


@dataclass(frozen=True, slots=True)
class Origin:
    """One walk-forward origin and every boundary derived from it.

    Boundaries are half-open ``[start, end)``. The origin ends validation and
    starts testing: a forecaster at ``o`` has seen everything before ``o``.
    """

    index: int
    origin: datetime

    @property
    def train_start(self) -> datetime:
        """Start of the 24-month rolling training window."""
        return add_months(self.origin, -TRAIN_MONTHS)

    @property
    def train_sub_end(self) -> datetime:
        """End of the 21-month training sub-block, which is also ``val_start``."""
        return add_months(self.origin, -VAL_MONTHS)

    @property
    def val_start(self) -> datetime:
        return self.train_sub_end

    @property
    def val_end(self) -> datetime:
        return self.origin

    @property
    def test_start(self) -> datetime:
        return self.origin

    @property
    def test_end(self) -> datetime:
        return self.origin + timedelta(days=BLOCK_DAYS * TEST_BLOCKS)

    def block(self, b: int) -> tuple[datetime, datetime]:
        """Half-open bounds of test block ``b``, one-indexed."""
        if not 1 <= b <= TEST_BLOCKS:
            raise ValueError(f"block index must be in 1..{TEST_BLOCKS}, got {b}")
        start = self.origin + timedelta(days=BLOCK_DAYS * (b - 1))
        return start, start + timedelta(days=BLOCK_DAYS)

    def blocks(self) -> list[tuple[int, datetime, datetime]]:
        """Every test block as ``(label, start, end)``, label one-indexed."""
        return [(b, *self.block(b)) for b in range(1, TEST_BLOCKS + 1)]

    @property
    def label(self) -> str:
        """``YYYY-MM``, the form used in every table and figure."""
        return self.origin.strftime("%Y-%m")


#: What :func:`itransformer_btc.splits.build_origin_tensors` accepts.
OriginLike = Origin


def origin_grid(
    first: datetime = FIRST_ORIGIN,
    spacing_months: int = ORIGIN_SPACING_MONTHS,
    data_start: datetime = DATA_START,
    data_end: datetime = DATA_END,
) -> list[Origin]:
    """Every origin whose 24-month training window and six test blocks fit the data.

    Under the constants above this yields 15 origins, 2020-01 to 2025-11.

    Raises:
        ValueError: If the first origin would need data from before the window.
    """
    grid: list[Origin] = []
    candidate = first
    while True:
        origin = Origin(index=len(grid) + 1, origin=candidate)
        if origin.train_start < data_start:
            raise ValueError(
                f"origin {origin.label} needs training data from "
                f"{origin.train_start.date()}, before the data window opens at "
                f"{data_start.date()}"
            )
        if origin.test_end > data_end:
            break
        grid.append(origin)
        candidate = add_months(candidate, spacing_months)
    return grid


#: Materialised once; import this rather than rebuilding the grid.
ORIGINS: Final = origin_grid()


# -- provenance of every algorithm the study runs -------------------------------


@dataclass(frozen=True, slots=True)
class Upstream:
    """Where one algorithm in this package came from, and what was changed.

    Attributes:
        component: The names in this package the row accounts for.
        module: The ``src/itransformer_btc`` file they live in.
        status: ``copied`` (the authors' code, unchanged), ``library`` (imported
            and called) or ``own`` (written here from the published description).
        reference: IEEE-style citation.
        repo: Official code, empty when there is none.
        licence: Upstream licence, empty when there is no upstream code.
        accessed: ISO date the repository was last opened.
        adapted: Every deliberate departure, or how the code is called.
        verified: True when the repository at the stated revision was checked.
    """

    component: str
    module: str
    status: str
    reference: str
    repo: str = ""
    licence: str = ""
    accessed: str = ""
    adapted: str = ""
    verified: bool = False


#: Printed by the notebook's provenance step and bound to the module docstrings
#: by ``tests/test_provenance.py``.
SOURCE_PROVENANCE: Final[tuple[Upstream, ...]] = (
    Upstream(
        component="ITransformerForecaster (Model in model/iTransformer.py)",
        module="model.py",
        status="copied",
        reference=(
            "Y. Liu, T. Hu, H. Zhang, H. Wu, S. Wang, L. Ma, and M. Long, "
            '"iTransformer: Inverted transformers are effective for time series '
            'forecasting," in Proc. 12th Int. Conf. Learn. Represent. (ICLR), '
            "2024. arXiv:2310.06625."
        ),
        repo="https://github.com/thuml/iTransformer",
        licence="MIT",
        accessed="2026-09-23",
        adapted=(
            "Copied unchanged at commit c2426e68ca13f74aaec08045c5c724d8ad328124. "
            "The adapter passes x_mark=None, reads the target channel, and uses "
            "d_model 128 and d_ff 256 for the sample size; use_norm stays on."
        ),
        verified=True,
    ),
    Upstream(
        component="VanillaForecaster (Model in model/Transformer.py)",
        module="model.py",
        status="copied",
        reference=(
            "A. Vaswani, N. Shazeer, N. Parmar, J. Uszkoreit, L. Jones, A. N. "
            'Gomez, L. Kaiser, and I. Polosukhin, "Attention is all you need," in '
            "Adv. Neural Inf. Process. Syst. 30 (NeurIPS), 2017; implementation "
            "from the iTransformer repository (Liu et al., 2024)."
        ),
        repo="https://github.com/thuml/iTransformer",
        licence="MIT",
        accessed="2026-09-23",
        adapted=(
            "Copied unchanged at the same commit, with the official code's "
            "defaults: 2 encoder and 1 decoder layer, gelu, label_len 48, "
            "x_mark=None. The decoder reads the last 48 lookback hours plus 24 "
            "zero placeholders and decodes in one pass; the target channel is "
            "read from the all-channel head."
        ),
        verified=True,
    ),
    Upstream(
        component="RidgeForecaster (sklearn.linear_model.Ridge)",
        module="baselines.py",
        status="library",
        reference=(
            "A. E. Hoerl and R. W. Kennard, "
            '"Ridge regression: Biased estimation for nonorthogonal problems," '
            "Technometrics, vol. 12, no. 1, pp. 55-67, 1970; F. Pedregosa et al., "
            '"Scikit-learn: Machine learning in Python," J. Mach. Learn. Res., '
            "vol. 12, pp. 2825-2830, 2011."
        ),
        repo="https://github.com/scikit-learn/scikit-learn",
        licence="BSD-3-Clause",
        accessed="2026-09-23",
        adapted=(
            "Ridge(fit_intercept=True, solver='cholesky') on float64 flattened "
            "K x 96 windows, one fit per alpha; alpha is chosen on validation MSE "
            "and the fitted coefficients are copied into a torch module."
        ),
    ),
    Upstream(
        component="Naive-RW benchmark (OriginTensors.naive_rw_z)",
        module="splits.py",
        status="own",
        reference=(
            "R. J. Hyndman and G. Athanasopoulos, Forecasting: Principles and "
            "Practice, 3rd ed. Melbourne, Australia: OTexts, 2021."
        ),
        adapted=(
            "A random walk in price predicts a zero log-return. In scaler space "
            "that is -mu_g/sigma_g, not 0, which would be the training drift."
        ),
    ),
    Upstream(
        component="walk-forward with purging (Origin, build_origin_tensors)",
        module="splits.py",
        status="own",
        reference=(
            "M. Lopez de Prado, Advances in Financial Machine Learning. "
            "Hoboken, NJ: Wiley, 2018, ch. 7; L. J. Tashman, "
            '"Out-of-sample tests of forecasting accuracy: An analysis and '
            'review," Int. J. Forecast., vol. 16, no. 4, pp. 437-450, 2000; '
            "C. Bergmeir and J. M. Benitez, "
            '"On the use of cross-validation for time series predictor '
            'evaluation," Information Sciences, vol. 191, pp. 192-213, 2012.'
        ),
        adapted=(
            "Purging at both boundaries (train/validation and train/test); no "
            "embargo, because every feature is per-bar; origins five months apart."
        ),
    ),
    Upstream(
        component="Scaler",
        module="splits.py",
        status="own",
        reference="No upstream publication: a per-channel z-score.",
        adapted="Fitted on the 21-month training sub-block only, at every origin.",
    ),
    Upstream(
        component="Adam optimiser and StepLR schedule (train_one)",
        module="train.py",
        status="library",
        reference=(
            "D. P. Kingma and J. Ba, "
            '"Adam: A method for stochastic optimization," in Proc. 3rd Int. '
            "Conf. Learn. Represent. (ICLR), 2015. arXiv:1412.6980."
        ),
        repo="https://docs.pytorch.org/docs/stable/optim.html",
        licence="BSD-3-Clause (PyTorch)",
        accessed="2026-09-03",
        adapted=(
            "Adam at lr 1e-4; StepLR halves every four epochs so the 30-epoch cap "
            "can bind. The loop is written here: GPU-resident splits, index "
            "slicing, no Dataset or DataLoader."
        ),
    ),
    Upstream(
        component="dm_test, clark_west_test (HLN correction, rectangular LRV)",
        module="metrics.py",
        status="own",
        reference=(
            "F. X. Diebold and R. S. Mariano, "
            '"Comparing predictive accuracy," J. Bus. Econ. Statist., vol. 13, '
            "no. 3, pp. 253-263, 1995; D. Harvey, S. Leybourne, and P. Newbold, "
            '"Testing the equality of prediction mean squared errors," Int. J. '
            "Forecast., vol. 13, no. 2, pp. 281-291, 1997; T. E. Clark and "
            'K. D. West, "Approximately normal tests for equal predictive '
            'accuracy in nested models," J. Econometrics, vol. 138, no. 1, '
            "pp. 291-311, 2007."
        ),
        adapted=(
            "Written on numpy. The long-run variance is the truncated rectangular "
            "estimator at lag h-1, not Bartlett, which would shrink the high-lag "
            "autocovariances. Clark-West is applied only against Naive-RW."
        ),
    ),
    Upstream(
        component="wild cluster restricted bootstrap (WCR) for beta1",
        module="metrics.py",
        status="own",
        reference=(
            "A. C. Cameron, J. B. Gelbach, and D. L. Miller, "
            '"Bootstrap-based improvements for inference with clustered errors," '
            "Rev. Econ. Statist., vol. 90, no. 3, pp. 414-427, 2008; "
            "J. G. MacKinnon, M. O. Nielsen, and M. D. Webb, "
            '"Cluster-robust inference: A guide to empirical practice," '
            "J. Econometrics, vol. 232, no. 2, pp. 272-299, 2023."
        ),
        adapted=(
            "Restricted (the null is imposed), bootstrapping the cluster-robust t, "
            "with Rademacher and Webb weights both reported and p = (1 + count)/(1 + B)."
        ),
    ),
    Upstream(
        component="romano_wolf",
        module="comparisons.py",
        status="own",
        reference=(
            "J. P. Romano and M. Wolf, "
            '"Stepwise multiple testing as formalized data snooping," '
            "Econometrica, vol. 73, no. 4, pp. 1237-1282, 2005."
        ),
        adapted="Stepdown over all pairs and within each declared family.",
    ),
    Upstream(
        component="model_confidence_set, mcs_table",
        module="comparisons.py",
        status="own",
        reference=(
            "P. R. Hansen, A. Lunde, and J. M. Nason, "
            '"The model confidence set," Econometrica, vol. 79, no. 2, '
            "pp. 453-497, 2011."
        ),
        adapted="Reported at 90% and 75% as a membership column of Table 6.",
    ),
    Upstream(
        component="variance_ratio and adf inside efficiency_table",
        module="efficiency.py",
        status="library",
        reference=(
            "A. W. Lo and A. C. MacKinlay, "
            '"Stock market prices do not follow random walks: Evidence from a '
            'simple specification test," Rev. Financial Stud., vol. 1, no. 1, '
            "pp. 41-66, 1988."
        ),
        repo="https://github.com/bashtage/arch",
        licence="NCSA (arch), BSD-3-Clause (statsmodels)",
        accessed="2026-09-03",
        adapted=(
            "arch.unitroot.VarianceRatio and statsmodels.tsa.stattools.adfuller "
            "are called directly, imported inside the function that needs them."
        ),
    ),
    Upstream(
        component="hurst_rs",
        module="efficiency.py",
        status="own",
        reference=(
            "H. E. Hurst, "
            '"Long-term storage capacity of reservoirs," Trans. Amer. Soc. Civil '
            "Eng., vol. 116, no. 1, pp. 770-799, 1951."
        ),
        adapted="Rescaled-range estimator written here on numpy.",
    ),
    Upstream(
        component="participation_ratio, stable_rank, lookback_correlation_pr",
        module="keff.py",
        status="own",
        reference=(
            "L. Laloux, P. Cizeau, J.-P. Bouchaud, and M. Potters, "
            '"Noise dressing of financial correlation matrices," Phys. Rev. '
            "Lett., vol. 83, no. 7, pp. 1467-1470, 1999; V. Plerou, "
            "P. Gopikrishnan, B. Rosenow, L. A. N. Amaral, T. Guhr, and "
            'H. E. Stanley, "Random matrix approach to cross correlations in '
            'financial data," Phys. Rev. E, vol. 65, no. 6, 066126, 2002.'
        ),
        adapted=(
            "PR on correlation matrices, never covariance, because the variates "
            "do not share units; measured per origin on the training sub-block."
        ),
    ),
    Upstream(
        component="parkinson, garman_klass, rogers_satchell (family F2)",
        module="features.py",
        status="own",
        reference=(
            "M. Parkinson, "
            '"The extreme value method for estimating the variance of the rate '
            'of return," J. Business, vol. 53, no. 1, pp. 61-65, 1980; '
            'M. B. Garman and M. J. Klass, "On the estimation of security price '
            'volatilities from historical data," J. Business, vol. 53, no. 1, '
            "pp. 67-78, 1980; L. C. G. Rogers and S. E. Satchell, "
            '"Estimating variance from high, low and closing prices," Ann. Appl. '
            "Probab., vol. 1, no. 4, pp. 504-512, 1991."
        ),
        adapted=(
            "Per-bar, never smoothed. Rogers-Satchell vanishes on shadowless "
            "bars, so it is taken as log(RS + 1e-9)."
        ),
    ),
)


###

<div style="background: linear-gradient(90deg, #101010, #1c1c1c); border-left: 3px solid #9e9e9e; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #d0d0d0; font-size: 1em; margin: 0;">🧾 <code>__init__.py</code></h3> <p style="display: inline; color: #a8a8a8; font-size: 0.9em; margin: 0;">· Nama publik paket.</p></div>

In [5]:
"""Tested projection of notebooks/btc_walkforward_3model.ipynb.

Edit the notebook and export through its final cell; this package is what the
tests import and what ``code_sha256`` hashes.
"""

__all__ = [
    "ORIGINS",
    "Origin",
    "PRED_LEN",
    "SEQ_LEN",
    "STARTS_LOST_PER_BREAK",
    "WINDOW_SPAN",
    "origin_grid",
]


##

<a id="section-02"></a>

<div style="background: linear-gradient(135deg, #1a1200, #2b1d00); border-left: 4px solid #ffb703; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #ffd60a; margin: 0 0 4px; font-size: 1.35em;">📥 02 · Muat data dan audit kualitas</h2>
  <p style="color: #ffca7a; margin: 0; font-size: 0.95em;">Gap memutus deret tanpa imputasi; jendela divalidasi lewat timestamp; anggaran jendela dicocokkan per origin.</p>
</div>

###

<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 3px solid #ffb703; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #ffd60a; font-size: 1em; margin: 0;">✂️ <code>segments.py</code></h3> <p style="display: inline; color: #ffca7a; font-size: 0.9em; margin: 0;">· Gap dan bar tak layak memutus deret; tidak ada imputasi.</p></div>

In [6]:
"""The segment law: what breaks the series, and where.

A segment is a maximal run of contiguous, usable hourly bars. The series breaks
at every missing bar, because no price formed and imputation is undefined, and
at every zero-volume or ``high == low`` bar, which carries no trade information
either. Returns and windows are computed inside segments only.
"""

HOUR_MS: Final = 3_600_000

DEFAULT_PARQUET: Final = Path("data/raw/BTCUSDT_1h.parquet")


@dataclass(frozen=True, slots=True)
class Segment:
    """A maximal run of contiguous usable bars, as half-open row indices.

    Attributes:
        start_row: First row index into the usable frame, inclusive.
        end_row: One past the last row index, exclusive.
        start_ts: Epoch ms of the first bar.
        end_ts: Epoch ms of the last bar, inclusive.
    """

    start_row: int
    end_row: int
    start_ts: int
    end_ts: int

    @property
    def n_bars(self) -> int:
        return self.end_row - self.start_row

    def window_starts(self, span: int) -> int:
        """How many ``span``-bar windows start inside this segment; never negative."""
        return max(0, self.n_bars - span + 1)


def load_bars(path: Path | str = DEFAULT_PARQUET) -> pl.DataFrame:
    """Load the immutable input parquet, sorted, with an epoch-ms ``ts_ms`` column.

    Raises:
        FileNotFoundError: If the file is absent.
        ValueError: If the frame is empty, has duplicate timestamps, or reaches
            outside the half-open data window.
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. The Stage 1 artifacts live in data/raw/; "
            f"regenerate with "
            f"`python spot_klines_btc.py --rebuild-only --outdir ./data/raw`."
        )

    frame = (
        pl.read_parquet(path)
        .with_columns(pl.col("open_time").dt.epoch("ms").alias("ts_ms"))
        .sort("ts_ms")
    )

    if frame.height == 0:
        raise ValueError(f"{path} is empty")

    n_unique = frame.select(pl.col("ts_ms").n_unique()).item()
    if n_unique != frame.height:
        raise ValueError(
            f"{path} carries {frame.height - n_unique} duplicate timestamps; "
            f"Stage 1 de-duplicates, so this is not Stage 1 output"
        )

    lo, hi = frame.select(
        pl.col("ts_ms").min().alias("lo"), pl.col("ts_ms").max().alias("hi")
    ).row(0)
    window_lo = int(DATA_START.timestamp() * 1000)
    window_hi = int(DATA_END.timestamp() * 1000)
    if lo < window_lo or hi >= window_hi:
        raise ValueError(
            f"{path} reaches outside the half-open data window "
            f"[{DATA_START.isoformat()}, {DATA_END.isoformat()}): "
            f"first={lo} last={hi}. Re-emit with --rebuild-only, which "
            f"applies clip_to_window()."
        )
    return frame


def usable_mask(frame: pl.DataFrame) -> pl.DataFrame:
    """Flag each bar usable or not.

    A bar is unusable when it has zero volume or ``high == low``. ``zero_trades`` is
    recorded for the data audit but does not by itself exclude a bar.

    Returns:
        The input plus boolean ``zero_volume``, ``flat_bar``, ``zero_trades`` and
        ``usable``.
    """
    return frame.with_columns(
        (pl.col("volume") <= 0).alias("zero_volume"),
        (pl.col("high") <= pl.col("low")).alias("flat_bar"),
        (pl.col("trades") <= 0).alias("zero_trades"),
    ).with_columns(
        (~pl.col("zero_volume") & ~pl.col("flat_bar")).alias("usable")
    )


def build_segments(
    frame: pl.DataFrame,
    start: datetime | None = None,
    end: datetime | None = None,
) -> list[Segment]:
    """Split the usable bars of ``[start, end)`` into contiguous segments.

    A new segment starts wherever the previous usable bar is not exactly one hour
    earlier, which covers missing and excluded bars alike.
    """
    if "usable" not in frame.columns:
        frame = usable_mask(frame)

    span = frame.filter(pl.col("usable"))
    if start is not None:
        span = span.filter(pl.col("ts_ms") >= int(start.timestamp() * 1000))
    if end is not None:
        span = span.filter(pl.col("ts_ms") < int(end.timestamp() * 1000))
    if span.height == 0:
        return []

    ts = span.get_column("ts_ms").to_list()
    segments: list[Segment] = []
    seg_start = 0
    for i in range(1, len(ts)):
        if ts[i] - ts[i - 1] != HOUR_MS:
            segments.append(Segment(seg_start, i, ts[seg_start], ts[i - 1]))
            seg_start = i
    segments.append(Segment(seg_start, len(ts), ts[seg_start], ts[-1]))
    return segments


@dataclass(frozen=True, slots=True)
class BreakSummary:
    """Measured break profile of a span; every field is counted."""

    calendar_hours: int
    bars_present: int
    bars_usable: int
    missing_bars: int
    zero_volume_bars: int
    flat_bars: int
    zero_trade_bars: int
    excluded_positions: int
    break_runs: int

    @property
    def segments(self) -> int:
        """Segments the span splits into."""
        return max(1, self.break_runs + 1)

    @property
    def window_starts_lost(self) -> int:
        """``119 x break_runs + excluded_positions`` window starts lost to breaks."""
        return STARTS_LOST_PER_BREAK * self.break_runs + self.excluded_positions


def break_summary(
    frame: pl.DataFrame,
    start: datetime,
    end: datetime,
) -> BreakSummary:
    """Measure every break-inducing condition in ``[start, end)``.

    A break run is a maximal stretch of excluded calendar hours, missing or
    unusable. Each run costs 119 window starts, so adjacent exclusions are counted
    once.
    """
    if "usable" not in frame.columns:
        frame = usable_mask(frame)

    lo = int(start.timestamp() * 1000)
    hi = int(end.timestamp() * 1000)
    span = frame.filter((pl.col("ts_ms") >= lo) & (pl.col("ts_ms") < hi))

    calendar_hours = (hi - lo) // HOUR_MS
    usable = int(span.select(pl.col("usable").sum()).item())
    counts = span.select(
        pl.col("zero_volume").sum().alias("zv"),
        pl.col("flat_bar").sum().alias("fb"),
        pl.col("zero_trades").sum().alias("zt"),
    ).row(0)

    # Walk the calendar, not the rows: a missing bar has no row to inspect, and
    # a run mixing missing with unusable positions must count once.
    usable_ts = set(span.filter(pl.col("usable")).get_column("ts_ms").to_list())
    break_runs = 0
    in_run = False
    for t in range(lo, hi, HOUR_MS):
        if t in usable_ts:
            in_run = False
        else:
            if not in_run:
                break_runs += 1
            in_run = True

    return BreakSummary(
        calendar_hours=calendar_hours,
        bars_present=span.height,
        bars_usable=usable,
        missing_bars=calendar_hours - span.height,
        zero_volume_bars=int(counts[0]),
        flat_bars=int(counts[1]),
        zero_trade_bars=int(counts[2]),
        excluded_positions=calendar_hours - usable,
        break_runs=break_runs,
    )


###

<div style="background: linear-gradient(90deg, #1a1200, #2b1d00); border-left: 3px solid #fb8500; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #ffb703; font-size: 1em; margin: 0;">🪟 <code>windows.py</code></h3> <p style="display: inline; color: #ffca7a; font-size: 0.9em; margin: 0;">· Jendela sah divalidasi lewat timestamp, bukan indeks.</p></div>

In [7]:
"""Window enumeration, validated by timestamp rather than by position.

A window ``[s, s+L+H)`` is valid only if ``t[s+L+H-1] - t[s] == (L+H-1)`` hours.
Positional sliding after a row drop would close a gap silently, so every emitted
window is checked. Enumerating only windows that lie wholly inside a span also
applies the purge: the last target ends at the span boundary.
"""

def enumerate_windows(
    frame: pl.DataFrame,
    start: datetime | None = None,
    end: datetime | None = None,
    seq_len: int = SEQ_LEN,
    pred_len: int = PRED_LEN,
) -> list[int]:
    """Every valid window start inside ``[start, end)``, as epoch ms.

    Windows are built inside segments and never across a break.

    Args:
        frame: Bars carrying ``ts_ms``; ``usable`` is derived if absent.
        start: Inclusive lower bound of the span.
        end: Exclusive upper bound of the span.
        seq_len: Lookback ``L``.
        pred_len: Horizon ``H``.

    Returns:
        Window-start timestamps in ascending order.

    Raises:
        ValueError: If an emitted window fails the timestamp identity, which means
            the segment builder is wrong.
    """
    span = seq_len + pred_len
    segments = build_segments(frame, start, end)
    if not segments:
        return []

    rows = (frame if "usable" in frame.columns else usable_mask(frame)).filter(
        pl.col("usable")
    )
    if start is not None:
        rows = rows.filter(pl.col("ts_ms") >= int(start.timestamp() * 1000))
    if end is not None:
        rows = rows.filter(pl.col("ts_ms") < int(end.timestamp() * 1000))
    ts = rows.get_column("ts_ms").to_list()

    starts: list[int] = []
    for seg in segments:
        for s in range(seg.start_row, seg.end_row - span + 1):
            last = s + span - 1
            if ts[last] - ts[s] != (span - 1) * HOUR_MS:
                raise ValueError(
                    f"window at ts={ts[s]} spans a break: "
                    f"t[{last}] - t[{s}] = {ts[last] - ts[s]} ms, expected "
                    f"{(span - 1) * HOUR_MS} ms. The segment builder is wrong; "
                    f"do not relax this check."
                )
            starts.append(ts[s])
    return starts


def count_windows(
    segments: list[Segment],
    seq_len: int = SEQ_LEN,
    pred_len: int = PRED_LEN,
) -> int:
    """Total window starts across segments, ``max(0, n - span + 1)`` each.

    The closed form ``(bars - 119) - (119 x breaks + missing)`` agrees only while
    every segment is longer than one window; this count is exact either way.
    """
    span = seq_len + pred_len
    return sum(seg.window_starts(span) for seg in segments)


###

<div style="background: linear-gradient(90deg, #231400, #3a2200); border-left: 3px solid #f48c06; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #ffba08; font-size: 1em; margin: 0;">🧮 <code>budget.py</code></h3> <p style="display: inline; color: #ffd08a; font-size: 0.9em; margin: 0;">· Anggaran jendela per origin, dicocokkan persis.</p></div>

In [8]:
"""Per-origin window accounting, checked against a table fixed before any run.

The committed table holds what the input data must yield per origin. This module
measures the same quantities from the artifact, and the budget test asserts exact
equality per origin: a pooled comparison would hide positional drift.
"""

#: ``(break_runs, excluded_positions, windows_kept)`` per origin's 21-month training
#: sub-block, measured from the input data before any model ran. Pinned rather
#: than recomputed, so the test catches drift instead of agreeing with itself.
COMMITTED_TRAIN_BUDGET: Final[dict[str, tuple[int, int, int]]] = {
    "2020-01": (11, 87, 13_934),
    "2020-06": (13, 63, 13_701),
    "2020-11": (12, 48, 13_741),
    "2021-04": (13, 41, 13_716),
    "2021-09": (14, 30, 13_560),
    "2022-02": (14, 32, 13_558),
    "2022-07": (9, 20, 14_165),
    "2022-12": (8, 19, 14_285),
    "2023-05": (2, 6, 15_021),
    "2023-10": (1, 2, 15_072),
    "2024-03": (1, 2, 15_120),
    "2024-08": (1, 2, 15_096),
    "2025-01": (1, 2, 15_096),
    "2025-06": (0, 0, 15_217),
    "2025-11": (0, 0, 15_217),
}


@dataclass(frozen=True, slots=True)
class OriginBudget:
    """Measured window accounting for one origin's training sub-block."""

    origin: Origin
    summary: BreakSummary
    windows_measured: int
    windows_closed_form: int
    test_block_starts: tuple[int, ...]

    @property
    def label(self) -> str:
        return self.origin.label

    @property
    def loss_pct(self) -> float:
        """Window starts lost to breaks, as a percentage of a gap-free span."""
        ceiling = self.summary.calendar_hours - STARTS_LOST_PER_BREAK
        return 100.0 * (1.0 - self.windows_measured / ceiling) if ceiling else 0.0

    @property
    def closed_form_agrees(self) -> bool:
        """Whether the closed-form count matches the segment-wise count.

        A disagreement means a segment is shorter than one window.
        """
        return self.windows_measured == self.windows_closed_form


def surviving_block_starts(frame: pl.DataFrame, origin: Origin, b: int) -> int:
    """Window starts surviving inside test block ``b``, out of 720.

    Every hour of a test block is an admissible forecast origin, because a lookback
    that reaches back across the block boundary is information a forecaster has.
    An hour is lost only when a break falls inside the 120 bars its window spans.
    """
    lo, hi = origin.block(b)
    lo_ms = int(lo.timestamp() * 1000)
    hi_ms = int(hi.timestamp() * 1000)
    span_ms = (WINDOW_SPAN - 1) * HOUR_MS

    usable = set(
        frame.filter(pl.col("usable")).get_column("ts_ms").to_list()
    )
    survivors = 0
    for start in range(lo_ms, hi_ms, HOUR_MS):
        # The window is contiguous exactly when every hour it spans is usable.
        if all((start + (k - SEQ_LEN) * HOUR_MS) in usable for k in range(WINDOW_SPAN)):
            survivors += 1
    return survivors


def origin_budget(frame: pl.DataFrame, origin: Origin) -> OriginBudget:
    """Measure one origin's 21-month training sub-block and its six test blocks."""
    summary = break_summary(frame, origin.train_start, origin.train_sub_end)
    segments = build_segments(frame, origin.train_start, origin.train_sub_end)

    ceiling = summary.calendar_hours - STARTS_LOST_PER_BREAK
    blocks = [surviving_block_starts(frame, origin, b) for b in range(1, TEST_BLOCKS + 1)]

    return OriginBudget(
        origin=origin,
        summary=summary,
        windows_measured=count_windows(segments, SEQ_LEN, PRED_LEN),
        windows_closed_form=ceiling - summary.window_starts_lost,
        test_block_starts=tuple(blocks),
    )


def budget_table(frame: pl.DataFrame) -> list[OriginBudget]:
    """Measure every origin in the grid."""
    return [origin_budget(frame, origin) for origin in ORIGINS]


def format_markdown(budgets: list[OriginBudget]) -> str:
    """Render the measured table in the layout of ``docs/ORIGIN_WINDOW_BUDGET.md``."""
    head = (
        "| # | Origin | Training sub-block | Breaks | Excluded | Windows kept "
        "| Loss | Test-block starts B1…B6 |\n"
        "|---:|---|---|---:|---:|---:|---:|---|\n"
    )
    rows = [
        f"| {b.origin.index:>2} | {b.origin.origin:%Y-%m-%d} "
        f"| {b.origin.train_start:%Y-%m-%d} → {b.origin.train_sub_end:%Y-%m-%d} "
        f"| {b.summary.break_runs} | {b.summary.excluded_positions} "
        f"| {b.windows_measured:,} | {b.loss_pct:.1f}% "
        f"| {' / '.join(str(n) for n in b.test_block_starts)} |"
        for b in budgets
    ]
    return head + "\n".join(rows) + "\n"


###

<div style="background: linear-gradient(90deg, #231400, #3a2200); border-left: 3px solid #f48c06; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #ffba08; font-size: 1em; margin: 0;">📊 Muat bar dan audit anggaran</h3> <p style="display: inline; color: #ffd08a; font-size: 0.9em; margin: 0;">· sha256 input dicocokkan, bar tak layak ditandai, anggaran jendela per origin harus sama persis.</p></div>

<div style="background: #0e0e12; border-left: 3px solid #6c757d; border-radius: 0 6px 6px 0; padding: 4px 14px; font-size: 0.82em; color: #cfcfcf;"><span style="color: #9ec5fe; font-weight: 600;">MEMBACA</span> <code>data/raw/BTCUSDT_1h.parquet</code></div>

In [9]:
_digest = hashlib.sha256(PARQUET.read_bytes()).hexdigest()
if _digest != INPUT_SHA256:
    raise RuntimeError(f"{PARQUET.name} has sha256 {_digest[:12]}, not the study's input "
                       f"{INPUT_SHA256[:12]}; attach the original data/raw/BTCUSDT_1h.parquet.")

bars = usable_mask(load_bars(PARQUET))
print(f"bars {bars.height:,}  usable {int(bars['usable'].sum()):,}  unusable {int((~bars['usable']).sum())}")
print(bars.filter(~pl.col("usable")).select(["open_time", "zero_volume", "flat_bar", "zero_trades"]))

budgets = budget_table(bars)
drift = [(b.label, b.summary.break_runs, b.summary.excluded_positions, b.windows_measured,
          COMMITTED_TRAIN_BUDGET[b.label])
         for b in budgets
         if (b.summary.break_runs, b.summary.excluded_positions, b.windows_measured)
         != COMMITTED_TRAIN_BUDGET[b.label]]
assert not drift, f"window budget differs from docs/ORIGIN_WINDOW_BUDGET.md: {drift}"
print(pl.DataFrame([
    {"origin": b.label, "train_windows": b.windows_measured, "loss_pct": round(b.loss_pct, 2),
     "closed_form_agrees": b.closed_form_agrees,
     **{f"B{i}": n for i, n in enumerate(b.test_block_starts, start=1)}}
    for b in budgets
]))


bars 75,094  usable 75,091  unusable 3
shape: (3, 4)
┌─────────────────────────┬─────────────┬──────────┬─────────────┐
│ open_time               ┆ zero_volume ┆ flat_bar ┆ zero_trades │
│ ---                     ┆ ---         ┆ ---      ┆ ---         │
│ datetime[ms, UTC]       ┆ bool        ┆ bool     ┆ bool        │
╞═════════════════════════╪═════════════╪══════════╪═════════════╡
│ 2019-06-07 21:00:00 UTC ┆ true        ┆ true     ┆ true        │
│ 2021-02-11 03:00:00 UTC ┆ true        ┆ true     ┆ true        │
│ 2023-03-24 12:00:00 UTC ┆ true        ┆ true     ┆ true        │
└─────────────────────────┴─────────────┴──────────┴─────────────┘
shape: (15, 10)
┌─────────┬───────────────┬──────────┬────────────────────┬───┬─────┬─────┬─────┬─────┐
│ origin  ┆ train_windows ┆ loss_pct ┆ closed_form_agrees ┆ … ┆ B3  ┆ B4  ┆ B5  ┆ B6  │
│ ---     ┆ ---           ┆ ---      ┆ ---                ┆   ┆ --- ┆ --- ┆ --- ┆ --- │
│ str     ┆ i64           ┆ f64      ┆ bool               ┆   ┆ 

##

<a id="section-03"></a>

<div style="background: linear-gradient(135deg, #001a1a, #002b2b); border-left: 4px solid #48cae4; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #90e0ef; margin: 0 0 4px; font-size: 1.35em;">🧪 03 · Feature engineering dan eksplorasi</h2>
  <p style="color: #ade8f4; margin: 0; font-size: 0.95em;">Dua belas variat per-bar F1–F5 tanpa rolling window, dan diagnostik efisiensi pasar.</p>
</div>

###

<div style="background: linear-gradient(90deg, #001a1a, #002b2b); border-left: 3px solid #48cae4; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #90e0ef; font-size: 1em; margin: 0;">🔬 <code>features.py</code></h3> <p style="display: inline; color: #ade8f4; font-size: 0.9em; margin: 0;">· Dua belas variat per-bar F1–F5 dan tangga K.</p></div>

In [10]:
"""The twelve variates and the K ladder cut over them.

Every variate is a per-bar function of the current bar (``r`` also reads the
previous close), so no feature uses a rolling window and no test bar can reach a
training feature. The ladder is cumulative: rung K is the first K columns and the
target ``r`` is channel 0 at every rung. Nothing outside families F1-F5 enters.

Upstream:
    Written here in polars from the published closed forms. F2 volatility:
    M. Parkinson, J. Business 53(1), 1980; M. B. Garman and M. J. Klass,
    J. Business 53(1), 1980; L. C. G. Rogers and S. E. Satchell, Ann. Appl.
    Probab. 1(4), 1991. Rogers-Satchell vanishes on shadowless bars, so it is
    taken as ``log(RS + 1e-9)``.
"""

VARIATE_ORDER: Final[tuple[str, ...]] = (
    # F1 price trajectory — K=1 is `r` alone
    "r",
    "upper_shadow",
    "lower_shadow",
    # F3 intensity, first member — completes K=4
    "log_quote_volume",
    # K=8: intensity, order flow, intrabar location
    "log_trade_count",
    "taker_buy_ratio",
    "signed_flow",
    "vwap_location",
    # K=12: the F2 volatility estimators + the dependent intensity member
    "log_parkinson",
    "log_garman_klass",
    "log_rogers_satchell",
    "log_mean_trade_size",
)

TARGET: Final = "r"
TARGET_INDEX: Final = 0

#: Parkinson's normaliser, ``1 / (4 ln 2)``.
_PARKINSON_C: Final = 1.0 / (4.0 * math.log(2.0))
#: Garman–Klass's second-term coefficient, ``2 ln 2 - 1`` ≈ 0.386. Strictly
#: below 0.5, which is what keeps the estimator positive: ``|ln(C/O)| <=
#: ln(H/L)`` because C and O both lie in ``[L, H]``, so GK >= 0.114 (ln H/L)^2.
_GK_C: Final = 2.0 * math.log(2.0) - 1.0

_RS_STABILISER: Final = 1e-9


def ladder_columns(k: int) -> list[str]:
    """The variate names at rung ``k`` (1, 4, 8 or 12)."""
    if k not in (1, 4, 8, 12):
        raise ValueError(f"K must be a documented rung 1/4/8/12, got {k}")
    return list(VARIATE_ORDER[:k])


def build_features(frame: pl.DataFrame) -> pl.DataFrame:
    """Compute all twelve variates per segment, dropping each segment's first bar.

    ``r`` is computed within each segment, so no return spans a gap.

    Returns:
        ``ts_ms``, ``usable`` and the twelve variates in ladder order as Float64.

    Raises:
        ValueError: If any variate is null or non-finite, which means the segment
            law did not run.
    """
    if "usable" not in frame.columns:
        frame = usable_mask(frame)

    rows = frame.filter(pl.col("usable")).sort("ts_ms")

    # Segment identity from the timestamp alone. Excluded bars are already gone,
    # so they show up here as jumps, exactly as downtime does.
    rows = rows.with_columns(
        (pl.col("ts_ms").diff().fill_null(HOUR_MS) != HOUR_MS).cum_sum().alias("_seg")
    )

    log_h_l = (pl.col("high") / pl.col("low")).log()
    log_c_o = (pl.col("close") / pl.col("open")).log()
    vwap = pl.col("quote_volume") / pl.col("volume")

    out = rows.with_columns(
        # -- F1 price trajectory, 3 dof -------------------------------------
        (pl.col("close").log() - pl.col("close").log().shift(1).over("_seg")).alias("r"),
        (pl.col("high") / pl.max_horizontal("open", "close")).log().alias("upper_shadow"),
        (pl.min_horizontal("open", "close") / pl.col("low")).log().alias("lower_shadow"),

        # -- F3 intensity, 2 dof — the third is the difference of the first two
        pl.col("quote_volume").log().alias("log_quote_volume"),
        pl.col("trades").log().alias("log_trade_count"),
        (pl.col("quote_volume") / pl.col("trades")).log().alias("log_mean_trade_size"),

        # Base-denominated: the buyer-initiated share of traded volume.
        (pl.col("taker_buy_base") / pl.col("volume")).alias("taker_buy_ratio"),

        # Total, because H == L bars break the series.
        ((vwap - pl.col("close")) / (pl.col("high") - pl.col("low"))).alias("vwap_location"),

        # Per-bar, never smoothed; only Rogers-Satchell needs the stabiliser.
        (_PARKINSON_C * log_h_l.pow(2)).log().alias("log_parkinson"),
        (0.5 * log_h_l.pow(2) - _GK_C * log_c_o.pow(2)).log().alias("log_garman_klass"),
        (
            (pl.col("high") / pl.col("close")).log() * (pl.col("high") / pl.col("open")).log()
            + (pl.col("low") / pl.col("close")).log() * (pl.col("low") / pl.col("open")).log()
            + _RS_STABILISER
        ).log().alias("log_rogers_satchell"),
    ).with_columns(
        # The product of two K=8 members; the dependence is disclosed.
        (
            (2.0 * pl.col("taker_buy_ratio") - 1.0) * pl.col("log_quote_volume")
        ).alias("signed_flow"),
    )

    # The first bar of each segment has no predecessor inside its segment.
    out = out.filter(pl.col("r").is_not_null())

    out = out.select(["ts_ms", "usable", *VARIATE_ORDER]).with_columns(
        [pl.col(c).cast(pl.Float64) for c in VARIATE_ORDER]
    )

    offenders = {
        name: n
        for name in VARIATE_ORDER
        if (
            n := int(
                out.select(
                    (~pl.col(name).is_finite() | pl.col(name).is_null()).sum()
                ).item()
            )
        )
    }
    if offenders:
        raise ValueError(
            f"non-finite variate values: {offenders}. Every variate is total "
            f"once zero-volume and H == L bars are excluded by the segment law, "
            f"so this means the exclusion did not run."
        )
    return out


###

<div style="background: linear-gradient(90deg, #001a1a, #002b2b); border-left: 3px solid #48cae4; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #90e0ef; font-size: 1em; margin: 0;">🔬 Bangun frame fitur</h3> <p style="display: inline; color: #ade8f4; font-size: 0.9em; margin: 0;">· Satu baris per segmen jatuh karena r butuh close sebelumnya; setiap variat harus finite.</p></div>

In [11]:
features = build_features(bars)
print(f"feature frame {features.height:,} rows x {len(VARIATE_ORDER)} variates; "
      f"{int(bars['usable'].sum()) - features.height} rows dropped, one per segment")
for k in K_LADDER:
    print(f"  K={k:>2}: {ladder_columns(k)}")
print(features.select(VARIATE_ORDER).describe().filter(
    pl.col("statistic").is_in(["mean", "std", "min", "max"])))
_rs = features["log_rogers_satchell"]
print(f"log_rogers_satchell at the log(1e-9) floor: {int((_rs <= np.log(1e-9) + 1e-9).sum())} bars")
assert all(features.select([pl.col(c).is_finite().all() for c in VARIATE_ORDER]).row(0)), \
    "a variate is non-finite"


feature frame 75,062 rows x 12 variates; 29 rows dropped, one per segment
  K= 1: ['r']
  K= 4: ['r', 'upper_shadow', 'lower_shadow', 'log_quote_volume']
  K= 8: ['r', 'upper_shadow', 'lower_shadow', 'log_quote_volume', 'log_trade_count', 'taker_buy_ratio', 'signed_flow', 'vwap_location']
  K=12: ['r', 'upper_shadow', 'lower_shadow', 'log_quote_volume', 'log_trade_count', 'taker_buy_ratio', 'signed_flow', 'vwap_location', 'log_parkinson', 'log_garman_klass', 'log_rogers_satchell', 'log_mean_trade_size']
shape: (4, 13)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ r         ┆ upper_sha ┆ lower_sha ┆ … ┆ log_parki ┆ log_garma ┆ log_roger ┆ log_mean │
│ ---       ┆ ---       ┆ dow       ┆ dow       ┆   ┆ nson      ┆ n_klass   ┆ s_satchel ┆ _trade_s │
│ str       ┆ f64       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ l         ┆ ize      │
│           ┆           ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64   

###

<div style="background: linear-gradient(90deg, #002200, #003300); border-left: 3px solid #7ae582; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #95d5b2; font-size: 1em; margin: 0;">📉 <code>efficiency.py</code></h3> <p style="display: inline; color: #b7e4c7; font-size: 0.9em; margin: 0;">· ADF, variance ratio, dan Hurst.</p></div>

In [12]:
"""Market-efficiency diagnostics for the data section (Table 2).

Reported for the whole sample and for each origin's 21-month training sub-block,
so the claim that efficiency varies over time is shown rather than assumed. The
test blocks are never read. ``arch`` and ``statsmodels`` take numpy arrays, so
pandas never enters.

Upstream:
    Variance ratio: ``arch.unitroot.VarianceRatio`` (https://github.com/bashtage/arch,
    NCSA), after A. W. Lo and A. C. MacKinlay, Rev. Financial Stud. 1(1), 1988.
    ADF: ``statsmodels.tsa.stattools.adfuller`` (BSD-3-Clause), after D. A. Dickey
    and W. A. Fuller, J. Amer. Statist. Assoc. 74(366), 1979.
    ``hurst_rs``: written here, after H. E. Hurst, Trans. Amer. Soc. Civil Eng.
    116(1), 1951.
"""

#: Lags for the Lo-MacKinlay variance ratio. Powers of two spanning two hours to
#: two thirds of a day, which brackets the horizons this study forecasts.
VR_LAGS: Final[tuple[int, ...]] = (2, 4, 8, 16)

#: Smallest R/S block. Below ~16 points the rescaled range is dominated by its
#: own small-sample bias and the log-log slope bends upward on white noise.
HURST_MIN_N: Final = 16

#: Blocks needed for the log-log regression to mean anything. Two points define a
#: line exactly and would report a slope with no residual to doubt it.
HURST_MIN_BLOCK_SIZES: Final = 3

VR_TREND: Final = "c"


@dataclass(frozen=True, slots=True)
class VarianceRatioRow:
    """One Lo-MacKinlay variance ratio; ``vr`` near 1 is consistent with a random walk."""

    lag: int
    vr: float
    statistic: float
    p_value: float


@dataclass(frozen=True, slots=True)
class ADFRow:
    """Augmented Dickey-Fuller on log-returns."""

    statistic: float
    p_value: float
    used_lag: int
    n_obs: int


def hurst_rs(x: np.ndarray, min_n: int = HURST_MIN_N, max_n: int | None = None) -> float:
    """Hurst exponent by rescaled range: the OLS slope of log(R/S) on log(n).

    Block sizes are dyadic; at each size the mean R/S over non-overlapping blocks
    enters the regression.

    Args:
        x: The series, normally log-returns.
        min_n: Smallest block.
        max_n: Largest block, by default ``len(x) // 4``.

    Raises:
        ValueError: If fewer than three usable block sizes fit.
    """
    x = np.asarray(x, dtype=np.float64)
    n_total = len(x)
    if max_n is None:
        max_n = n_total // 4
    sizes = [n for n in (min_n * 2**i for i in range(64)) if n <= max_n]
    if len(sizes) < HURST_MIN_BLOCK_SIZES:
        raise ValueError(
            f"R/S needs at least {HURST_MIN_BLOCK_SIZES} block sizes between "
            f"{min_n} and {max_n}; got {len(sizes)} at n={n_total}"
        )

    logs_n: list[float] = []
    logs_rs: list[float] = []
    for n in sizes:
        blocks = x[: (n_total // n) * n].reshape(-1, n)
        deviate = np.cumsum(blocks - blocks.mean(axis=1, keepdims=True), axis=1)
        spread = deviate.max(axis=1) - deviate.min(axis=1)
        sd = blocks.std(axis=1, ddof=1)
        # A constant block has no scale to rescale by. Dropping it is not
        # imputation -- nothing is invented, the block simply carries no R/S.
        keep = sd > 0
        if not keep.any():
            continue
        logs_n.append(float(np.log(n)))
        logs_rs.append(float(np.log(float((spread[keep] / sd[keep]).mean()))))

    if len(logs_n) < HURST_MIN_BLOCK_SIZES:
        raise ValueError(
            f"only {len(logs_n)} block sizes carried a non-zero standard deviation"
        )
    slope, _ = np.polyfit(np.asarray(logs_n), np.asarray(logs_rs), 1)
    return float(slope)


def variance_ratios(
    r: np.ndarray, lags: tuple[int, ...] = VR_LAGS
) -> list[VarianceRatioRow]:
    """Lo-MacKinlay variance ratio at each lag, from a return series.

    ``VarianceRatio`` expects a level series and differences it itself, so the
    returns are cumulated first; passing returns would report ``VR = 1/lag``. The
    returns are computed per segment, so the cumulation crosses no gap.
    """
    from arch.unitroot import VarianceRatio

    level = np.cumsum(np.asarray(r, dtype=np.float64))
    rows: list[VarianceRatioRow] = []
    for lag in lags:
        ratio = VarianceRatio(level, lags=lag, trend=VR_TREND, overlap=True)
        rows.append(
            VarianceRatioRow(
                lag=lag,
                vr=float(ratio.vr),
                statistic=float(ratio.stat),
                p_value=float(ratio.pvalue),
            )
        )
    return rows


def adf(r: np.ndarray) -> ADFRow:
    """Augmented Dickey-Fuller with AIC lag selection, on log-returns."""
    from statsmodels.tsa.stattools import adfuller

    stat, p_value, used_lag, n_obs, *_ = adfuller(
        np.asarray(r, dtype=np.float64), autolag="AIC"
    )
    return ADFRow(
        statistic=float(stat),
        p_value=float(p_value),
        used_lag=int(used_lag),
        n_obs=int(n_obs),
    )


def _row(span: str, r: np.ndarray) -> dict[str, float | str | int]:
    """One Table 2 row: ADF, Hurst and the variance ratio at every lag."""
    unit_root = adf(r)
    row: dict[str, float | str | int] = {
        "span": span,
        "n": int(len(r)),
        "adf_stat": unit_root.statistic,
        "adf_p": unit_root.p_value,
        "hurst": hurst_rs(r),
    }
    for ratio in variance_ratios(r):
        row[f"vr_{ratio.lag}"] = ratio.vr
        row[f"vr_p_{ratio.lag}"] = ratio.p_value
    return row


def _training_returns(features: pl.DataFrame, origin: OriginLike) -> np.ndarray:
    """Log-returns of one origin's 21-month training sub-block."""
    lo = int(origin.train_start.timestamp() * 1000)
    hi = int(origin.train_sub_end.timestamp() * 1000)
    return (
        features.filter((pl.col("ts_ms") >= lo) & (pl.col("ts_ms") < hi))
        .get_column("r")
        .to_numpy()
    )


#: Shortest sub-block the R/S regression can describe: enough rows for
#: :data:`HURST_MIN_BLOCK_SIZES` dyadic sizes, each averaged over four blocks.
MIN_SPAN_ROWS: Final = HURST_MIN_N * 2 ** (HURST_MIN_BLOCK_SIZES - 1) * 4


def efficiency_table(
    features: pl.DataFrame, origins: list[OriginLike] | None = None
) -> pl.DataFrame:
    """Table 2: one row for the whole sample, then one per origin's training sub-block.

    Returns:
        ``span, n, adf_stat, adf_p, hurst`` plus ``vr_{lag}`` / ``vr_p_{lag}`` per lag.
        A sub-block shorter than ``MIN_SPAN_ROWS`` is skipped.
    """
    grid = list(origins if origins is not None else ORIGINS)
    rows = [_row("full", features.get_column("r").to_numpy())]
    for origin in grid:
        returns = _training_returns(features, origin)
        if len(returns) < MIN_SPAN_ROWS:
            continue
        rows.append(_row(origin.label, returns))
    return pl.DataFrame(rows)


##

<a id="section-04"></a>

<div style="background: linear-gradient(135deg, #001233, #001845); border-left: 4px solid #4cc9f0; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #8ecae6; margin: 0 0 4px; font-size: 1.35em;">🪟 04 · Split walk-forward, scaling, dan K_eff</h2>
  <p style="color: #a9d6e5; margin: 0; font-size: 0.95em;">21 bulan latih dan 3 bulan validasi, purge 24 jam di kedua batas, scaler dari data latih saja, K_eff sebelum training.</p>
</div>

###

<div style="background: linear-gradient(90deg, #001233, #001845); border-left: 3px solid #4cc9f0; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #8ecae6; font-size: 1em; margin: 0;">🗂️ <code>splits.py</code></h3> <p style="display: inline; color: #a9d6e5; font-size: 0.9em; margin: 0;">· Split, purge, scaler dari data latih saja, dan tensor per origin.</p></div>

In [13]:
"""Per-origin splits, the scaler, and the tensors the training loop slices.

Training and validation windows lie wholly inside their spans, so no target
crosses into the next split: the purge holds at both boundaries. A test window
is indexed by its forecast origin, the first target hour, so every hour of a test
block is an admissible origin and its lookback may reach back into validation,
which is information a forecaster has at that moment.

The scaler is fitted on the rows of the 21-month training sub-block and on
nothing else, at every origin.

Upstream:
    Rolling-origin evaluation after L. J. Tashman, Int. J. Forecast. 16(4), 2000,
    and C. Bergmeir and J. M. Benitez, Information Sciences 191, 2012. Purging
    after M. Lopez de Prado, Advances in Financial Machine Learning, Wiley, 2018,
    ch. 7, applied at both boundaries; no embargo, because every feature is
    per-bar. The scaler is written here: two lines of arithmetic kept inside the
    per-origin tensor build.
"""

Semantics = Literal["contained", "origin"]


def window_starts(
    ts: np.ndarray, start: datetime, end: datetime, semantics: Semantics,
    span: int = WINDOW_SPAN, *, seq_len: int = SEQ_LEN,
) -> np.ndarray:
    """Input-start indices of the contiguous windows that belong to ``[start, end)``.

    ``"contained"`` keeps windows lying wholly inside the span (training and
    validation). ``"origin"`` keeps windows whose first target hour falls inside
    it (test). A forecast at t reads ``[t-L, t)`` and predicts ``[t, t+H)``.
    """
    if semantics not in ("contained", "origin") or span < 2:
        raise ValueError("invalid window semantics or span")
    if semantics == "origin" and not 0 < seq_len < span:
        raise ValueError("origin semantics require 0 < seq_len < span")
    ts = np.asarray(ts)
    if ts.ndim != 1 or np.any(np.diff(ts) <= 0) or np.any(ts % HOUR_MS != 0):
        raise ValueError("timestamps must be unique, increasing UTC hour boundaries")
    lo, hi = int(start.timestamp() * 1000), int(end.timestamp() * 1000)
    if hi <= lo:
        raise ValueError("window end must follow start")
    first = np.arange(max(0, len(ts) - span + 1), dtype=np.int64)
    contiguous = (ts[first + span - 1] - ts[first]) == (span - 1) * HOUR_MS
    if semantics == "contained":
        inside = (ts[first] >= lo) & (ts[first + span - 1] < hi)
    else:
        issued = ts[first + seq_len]
        inside = (issued >= lo) & (issued < hi)
    return first[contiguous & inside]


@dataclass(frozen=True, slots=True)
class Scaler:
    """Per-channel standardiser fitted on the training sub-block only.

    Under ``use_norm`` the iTransformer cancels any per-channel affine scaling, so
    for it this sets only the reporting scale; Ridge and the vanilla Transformer
    learn in this scale directly.
    """

    mean: np.ndarray
    std: np.ndarray
    columns: tuple[str, ...]

    @classmethod
    def fit(cls, values: np.ndarray, columns: list[str]) -> "Scaler":
        std = values.std(axis=0, ddof=0)
        if not np.all(np.isfinite(std)) or np.any(std <= 0):
            raise ValueError(
                f"degenerate channel std in the training sub-block: "
                f"{dict(zip(columns, std))}"
            )
        return cls(values.mean(axis=0), std, tuple(columns))

    def transform(self, values: np.ndarray) -> np.ndarray:
        return (values - self.mean) / self.std

    @property
    def target_mu_over_sigma(self) -> float:
        """``mu_g / sigma_g`` on the target channel, the Naive-RW offset.

        A random walk predicts ``r = 0``. In scaler space that is
        ``-mu_g / sigma_g``, not 0, which would mean predicting the training drift.
        """
        return float(self.mean[TARGET_INDEX] / self.std[TARGET_INDEX])


@dataclass(frozen=True, slots=True)
class SplitTensors:
    """One split's standardised windows."""

    x: np.ndarray      # (n, L, K) float32 inputs
    y: np.ndarray      # (n, H)    float32 target channel
    ts: np.ndarray     # (n,) int64 forecast origin = first target bar open (UTC)

    def __len__(self) -> int:
        return len(self.ts)


@dataclass(frozen=True, slots=True)
class OriginTensors:
    """Everything one run consumes for an (origin, K) cell, already standardised."""

    origin: OriginLike
    k: int
    scaler: Scaler
    train: SplitTensors
    val: SplitTensors
    test_blocks: tuple[SplitTensors, ...]
    #: One-indexed label of each entry of ``test_blocks``.
    block_labels: tuple[int, ...]
    training_selection: dict | None = None

    @property
    def naive_rw_z(self) -> float:
        """The Naive-RW forecast in scaler space."""
        return -self.scaler.target_mu_over_sigma


def _gather(
    values: np.ndarray, starts: np.ndarray, ts: np.ndarray, seq_len: int, pred_len: int
) -> SplitTensors:
    """Slice windows out of a standardised array by index arithmetic, with no data loader."""
    if len(starts) == 0:
        return SplitTensors(
            x=np.empty((0, seq_len, values.shape[1]), np.float32),
            y=np.empty((0, pred_len), np.float32),
            ts=np.empty(0, np.int64),
        )
    rows = starts[:, None] + np.arange(seq_len)[None, :]
    tgt = starts[:, None] + seq_len + np.arange(pred_len)[None, :]
    return SplitTensors(
        x=values[rows].astype(np.float32, copy=False),
        y=np.ascontiguousarray(values[tgt, TARGET_INDEX].astype(np.float32, copy=False)),
        ts=ts[starts + seq_len],
    )


def build_origin_tensors(
    features: pl.DataFrame,
    origin: OriginLike,
    k: int,
    seq_len: int = SEQ_LEN,
    pred_len: int = PRED_LEN,
    train_window_limit: int | None = None,
    selection_seed: int = 1729,
) -> OriginTensors:
    """Build every split for one (origin, K) cell.

    The scaler is fitted on the purged training rows before any window is cut or
    subsampled, then applied to every split. With ``train_window_limit`` the
    training windows are drawn without replacement with ``selection_seed``, and
    the draw is recorded in ``training_selection``.

    Raises:
        ValueError: If the training split is empty, if a training target reaches
            validation (the purge failed), or if fewer windows exist than the
            limit asks for.
    """
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    values = features.select(columns).to_numpy()
    span = seq_len + pred_len

    train_idx = window_starts(ts, origin.train_start, origin.train_sub_end,
                              "contained", span)
    val_idx = window_starts(ts, origin.val_start, origin.val_end, "contained", span)
    if len(train_idx) == 0:
        raise ValueError(f"origin {origin.label}: empty training split")

    last_train_target = ts[train_idx[-1] + span - 1]
    if last_train_target >= int(origin.val_start.timestamp() * 1000):
        raise ValueError(
            f"origin {origin.label}: a training target reaches into validation "
            f"({last_train_target}); the purge did not hold"
        )

    scaler = Scaler.fit(values[train_idx[0] : train_idx[-1] + span], columns)
    scaled = scaler.transform(values)
    n_available = len(train_idx)
    if train_window_limit is not None:
        if train_window_limit < 1 or n_available < train_window_limit:
            raise ValueError(f"{origin.label}: {n_available} training windows < required {train_window_limit}")
        selected = np.random.default_rng(selection_seed).choice(n_available, train_window_limit, replace=False)
        train_idx = train_idx[np.sort(selected)]
    selection = {"available": n_available, "selected": len(train_idx),
                 "limit": train_window_limit, "seed": selection_seed,
                 "forecast_times_sha256": hashlib.sha256(ts[train_idx + seq_len].tobytes()).hexdigest(),
                 "scaler_fit": "all purged training rows before subsampling"}

    blocks = origin.blocks()
    return OriginTensors(
        origin=origin,
        k=k,
        scaler=scaler,
        train=_gather(scaled, train_idx, ts, seq_len, pred_len),
        val=_gather(scaled, val_idx, ts, seq_len, pred_len),
        test_blocks=tuple(
            _gather(scaled, window_starts(ts, lo, hi, "origin", span, seq_len=seq_len),
                    ts, seq_len, pred_len)
            for _, lo, hi in blocks
        ),
        block_labels=tuple(label for label, _, _ in blocks),
        training_selection=selection,
    )


###

<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 3px solid #9d4edd; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #e0aaff; font-size: 1em; margin: 0;">📏 <code>keff.py</code></h3> <p style="display: inline; color: #c8a2e0; font-size: 0.9em; margin: 0;">· Dimensionalitas efektif (participation ratio) per origin.</p></div>

In [14]:
"""Effective dimensionality (K_eff), measured before any model trains.

K_eff is RQ1's regressor. Every value that enters inference is computed per
origin on that origin's 21-month training sub-block, so the regressor never
reads the test period. Four variants are reported side by side, because the
model consumes a K x 96 block rather than one bar: the contemporaneous PR, the
PR after per-window normalisation (what ``use_norm`` feeds the embedding), the
stable rank of each window block, and the PR of the lookback correlation.

Upstream:
    Written here on numpy, after L. Laloux, P. Cizeau, J.-P. Bouchaud and
    M. Potters, Phys. Rev. Lett. 83(7), 1999, and V. Plerou et al., Phys. Rev. E
    65(6), 066126, 2002. PR is taken on correlation matrices, never covariance,
    because the variates do not share units.
"""

#: The gate's floor, fixed before measuring anything.
GATE_PR_FLOOR: float = 5.0

#: Windows sampled for the lookback measures, by a fixed stride.
LOOKBACK_SAMPLE: int = 2_000


def participation_ratio(eigenvalues: np.ndarray) -> float:
    """``PR = (sum lambda)^2 / sum lambda^2``, in ``[1, K]``.

    Negative eigenvalues from rounding are clipped to zero, not dropped.

    Raises:
        ValueError: If the spectrum sums to zero.
    """
    lam = np.clip(np.asarray(eigenvalues, dtype=np.float64), 0.0, None)
    total = lam.sum()
    if total <= 0:
        raise ValueError("degenerate spectrum: eigenvalues sum to zero")
    return float(total * total / np.square(lam).sum())


def contemporaneous_pr(values: np.ndarray) -> float:
    """PR of the ``K x K`` correlation matrix of ``(n, K)`` bar-level values."""
    corr = np.atleast_2d(
        np.corrcoef(np.asarray(values, dtype=np.float64), rowvar=False)
    )
    return participation_ratio(np.linalg.eigvalsh(corr))


def window_normalised_pr(windows: np.ndarray) -> float:
    """PR after standardising each channel within its window, as ``use_norm`` does.

    A gap between this and the raw PR at K=12 would mean the normalisation, not the
    data, drives the apparent redundancy of the volatility family.
    """
    x = np.asarray(windows, dtype=np.float64)
    mean = x.mean(axis=1, keepdims=True)
    std = np.sqrt(x.var(axis=1, keepdims=True) + 1e-12)
    return contemporaneous_pr(((x - mean) / std).reshape(-1, x.shape[2]))


def stable_rank(matrix: np.ndarray) -> float:
    """``||M||_F^2 / ||M||_2^2``, in ``[1, min(rows, cols)]``."""
    singular = np.linalg.svd(np.asarray(matrix, dtype=np.float64), compute_uv=False)
    if singular[0] <= 0:
        raise ValueError("degenerate window block: largest singular value is 0")
    return float(np.square(singular).sum() / (singular[0] ** 2))


def lookback_stable_rank(windows: np.ndarray, sample: int = LOOKBACK_SAMPLE) -> float:
    """Mean stable rank of each window's ``K x L`` block, standardised within the window.

    Windows are sampled by a fixed stride, so the value is deterministic. After
    standardisation the stable rank equals ``K / lambda_1`` of the within-window
    correlation matrix, so it is on the same ``[1, K]`` scale as the PR.
    """
    x = np.asarray(windows, dtype=np.float64)
    if len(x) == 0:
        raise ValueError("no windows to measure")
    stride = max(1, len(x) // sample)
    blocks = np.transpose(x[::stride][:sample], (0, 2, 1))   # (m, K, L)
    blocks = blocks - blocks.mean(axis=2, keepdims=True)
    blocks = blocks / np.sqrt(np.square(blocks).mean(axis=2, keepdims=True) + 1e-12)
    return float(np.mean([stable_rank(b) for b in blocks]))


def lookback_correlation_pr(windows: np.ndarray) -> float:
    """PR of the ``K*L x K*L`` correlation spectrum of the flattened windows.

    The only variant that sees cross-lag structure. Its ceiling is ``K*L``, so rungs
    are compared through ``KeffRow.pr_lookback_ratio``.
    """
    x = np.asarray(windows, dtype=np.float64)
    flat = x.reshape(len(x), -1)
    flat = flat - flat.mean(axis=0, keepdims=True)
    scale = np.sqrt(np.square(flat).mean(axis=0, keepdims=True) + 1e-24)
    flat = flat / scale
    gram = (flat.T @ flat) / max(1, len(flat) - 1)
    return participation_ratio(np.linalg.eigvalsh(gram))


@dataclass(frozen=True, slots=True)
class KeffRow:
    """One (origin, rung) cell of Table 2b."""

    origin: str
    origin_index: int
    k: int
    n_rows: int
    n_windows: int
    pr_raw: float
    pr_window_norm: float
    stable_rank_lookback: float
    pr_lookback_corr: float

    @property
    def pr_lookback_ratio(self) -> float:
        """``pr_lookback_corr / (K * L)``: the cross-lag PR as a share of its ceiling."""
        return self.pr_lookback_corr / (self.k * SEQ_LEN)

    @property
    def divergence(self) -> float:
        """``stable_rank_lookback - pr_raw``: what the lookback adds to the bar-level view."""
        return self.stable_rank_lookback - self.pr_raw


def _training_windows(
    features: pl.DataFrame, origin: OriginLike, k: int, seq_len: int = SEQ_LEN
) -> tuple[np.ndarray, np.ndarray]:
    """Rows and windows of one origin's 21-month training sub-block, the span the scaler sees."""
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    values = features.select(columns).to_numpy()

    lo = int(origin.train_start.timestamp() * 1000)
    hi = int(origin.train_sub_end.timestamp() * 1000)
    rows = values[(ts >= lo) & (ts < hi)]

    starts = window_starts(
        ts, origin.train_start, origin.train_sub_end, "contained", WINDOW_SPAN
    )
    if len(starts) == 0:
        raise ValueError(f"origin {origin.label}: no training window to measure")
    idx = starts[:, None] + np.arange(seq_len)[None, :]
    return rows, values[idx]


def keff_row(features: pl.DataFrame, origin: OriginLike, k: int) -> KeffRow:
    """Every K_eff variant for one (origin, rung) cell."""
    rows, windows = _training_windows(features, origin, k)
    return KeffRow(
        origin=origin.label,
        origin_index=origin.index,
        k=k,
        n_rows=len(rows),
        n_windows=len(windows),
        pr_raw=contemporaneous_pr(rows),
        pr_window_norm=window_normalised_pr(windows),
        stable_rank_lookback=lookback_stable_rank(windows),
        pr_lookback_corr=lookback_correlation_pr(windows),
    )


def keff_table(
    features: pl.DataFrame,
    origins: list[OriginLike] | None = None,
    rungs: tuple[int, ...] = K_LADDER,
) -> pl.DataFrame:
    """Table 2b: every rung at every origin, on training spans only.

    K=1 stays in although its PR is 1 by definition, so the RQ1 panel is balanced.
    """
    grid = list(origins if origins is not None else ORIGINS)
    return pl.DataFrame(
        [
            {
                "origin": row.origin,
                "origin_index": row.origin_index,
                "k": row.k,
                "n_rows": row.n_rows,
                "n_windows": row.n_windows,
                "pr_raw": row.pr_raw,
                "pr_window_norm": row.pr_window_norm,
                "stable_rank_lookback": row.stable_rank_lookback,
                "pr_lookback_corr": row.pr_lookback_corr,
                "pr_lookback_ratio": row.pr_lookback_ratio,
                "divergence": row.divergence,
            }
            for origin in grid
            for k in rungs
            for row in (keff_row(features, origin, k),)
        ]
    )


def corr_k_keff(table: pl.DataFrame, column: str = "pr_raw") -> float:
    """``corr(K, K_eff)`` across rungs; near 1 means K and K_eff are hard to tell apart."""
    means = table.group_by("k").agg(pl.col(column).mean().alias("keff")).sort("k")
    k = means.get_column("k").to_numpy().astype(np.float64)
    keff = means.get_column("keff").to_numpy()
    return float(np.corrcoef(k, keff)[0, 1])


def gate_pr(features: pl.DataFrame, k: int = 8) -> float:
    """The gate value: PR at K=8 on ``[2018-01, 2020-01)``, a span before every origin."""
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    lo = int(DATA_START.timestamp() * 1000)
    hi = int(FIRST_ORIGIN.timestamp() * 1000)
    rows = features.select(columns).to_numpy()[(ts >= lo) & (ts < hi)]
    if len(rows) == 0:
        raise ValueError("the pre-first-origin span holds no usable bar")
    return contemporaneous_pr(rows)


def gate_verdict(measured: float, floor: float = GATE_PR_FLOOR) -> str:
    """The gate's action. Below the floor the result is disclosed, not re-cut.

    F1-F5 admit only one consistent ladder, so there is no alternative cut to take.
    """
    if measured >= floor:
        return (
            f"PASS: measured PR at K=8 is {measured:.3f} >= {floor:.1f}. "
            f"Proceed; report the value in Table 2b."
        )
    return (
        f"DISCLOSE: measured PR at K=8 is {measured:.3f} < {floor:.1f}. "
        f"Proceed unchanged and disclose the value; the ladder is not re-cut, "
        f"because F1-F5 leave no second consistent cut."
    )


#: Rolling window: long enough for an 8 x 8 correlation, short enough to see regimes.
ROLLING_WINDOW_DAYS: Final = 90

#: One day between consecutive rolling windows.
ROLLING_STEP_DAYS: Final = 1


def _rolling_spans(
    ts: np.ndarray, window_days: int, step_days: int
) -> list[tuple[int, int, int]]:
    """``(window_end_ms, lo, hi)`` per window, sliced by time rather than by position.

    Position slicing would stretch a window over gaps, and gaps cluster early in the
    sample.
    """
    window_ms = window_days * 24 * HOUR_MS
    step_ms = step_days * 24 * HOUR_MS
    spans: list[tuple[int, int, int]] = []
    for end in range(int(ts[0]) + window_ms, int(ts[-1]) + 1, step_ms):
        lo = int(np.searchsorted(ts, end - window_ms, side="left"))
        hi = int(np.searchsorted(ts, end, side="left"))
        spans.append((end, lo, hi))
    return spans


def rolling_pr(
    features: pl.DataFrame,
    k: int = 8,
    window_days: int = ROLLING_WINDOW_DAYS,
    step_days: int = ROLLING_STEP_DAYS,
) -> pl.DataFrame:
    """Rolling PR over the full sample (Figure 2b), descriptive only.

    It reads the test period, so it never enters a regression or the gate.

    Returns:
        ``window_end_ms, n_rows, pr``, one row per window.
    """
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    values = features.select(columns).to_numpy()

    rows = []
    for end, lo, hi in _rolling_spans(ts, window_days, step_days):
        block = values[lo:hi]
        if len(block) <= len(columns):
            continue        # a correlation needs more rows than columns
        rows.append(
            {"window_end_ms": end, "n_rows": len(block), "pr": contemporaneous_pr(block)}
        )
    return pl.DataFrame(rows)


def rolling_ols_r2(
    features: pl.DataFrame,
    k: int = 8,
    window_days: int = ROLLING_WINDOW_DAYS,
    step_days: int = ROLLING_STEP_DAYS,
) -> pl.DataFrame:
    """In-window R^2 of ``r_{t+1}`` on the K features at t (Figure 2b), descriptive only.

    It asks whether the feature-return relation moves over time, not whether it
    forecasts. Pairs that straddle a gap are dropped.

    Returns:
        ``window_end_ms, n_pairs, r2``, one row per window.
    """
    columns = ladder_columns(k)
    ts = features.get_column("ts_ms").to_numpy()
    values = features.select(columns).to_numpy()
    target = features.get_column("r").to_numpy()

    contiguous = np.zeros(len(ts), dtype=bool)
    contiguous[:-1] = np.diff(ts) == HOUR_MS

    rows = []
    for end, lo, hi in _rolling_spans(ts, window_days, step_days):
        # Clamped so the last window cannot ask for a successor that does not
        # exist; the mask and both slices then have one length by construction
        # rather than by a length check that fires after an IndexError would.
        n = min(hi, len(ts) - 1) - lo
        if n <= 0:
            continue
        keep = contiguous[lo : lo + n]
        x = values[lo : lo + n][keep]
        y = target[lo + 1 : lo + 1 + n][keep]
        if len(x) <= len(columns) + 1:
            continue
        design = np.column_stack([np.ones(len(x)), x])
        coefficients, *_ = np.linalg.lstsq(design, y, rcond=None)
        residual = y - design @ coefficients
        total = float(((y - y.mean()) ** 2).sum())
        rows.append({
            "window_end_ms": end,
            "n_pairs": len(y),
            "r2": float(1.0 - (residual ** 2).sum() / total) if total > 0 else 0.0,
        })
    return pl.DataFrame(rows)


###

<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 3px solid #9d4edd; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #e0aaff; font-size: 1em; margin: 0;">📐 Ukur K_eff</h3> <p style="display: inline; color: #c8a2e0; font-size: 0.9em; margin: 0;">· Gerbang PR pada rentang sebelum origin pertama, lalu K_eff per origin pada sub-blok latih.</p></div>

<div style="background: #0e0e12; border-left: 3px solid #6c757d; border-radius: 0 6px 6px 0; padding: 4px 14px; font-size: 0.82em; color: #cfcfcf;"><span style="color: #ffd166; font-weight: 600;">MENULIS</span> <code>artifacts/keff_table.parquet</code></div>

In [15]:
gate = gate_pr(features, k=8)
print(gate_verdict(gate))

t0 = time.perf_counter()
keff_tbl = keff_table(features)
print(f"\nmeasured in {time.perf_counter() - t0:.0f}s on each origin's training sub-block")
print(keff_tbl.group_by("k").agg(
    pl.col("pr_raw").mean().alias("PR_raw"),
    pl.col("pr_raw").std().alias("PR_raw_sd"),
    pl.col("pr_window_norm").mean().alias("PR_windownorm"),
    pl.col("stable_rank_lookback").mean().alias("stable_rank"),
    pl.col("pr_lookback_ratio").mean().alias("crosslag_share"),
).sort("k"))
print(f"corr(K, K_eff) = {corr_k_keff(keff_tbl):.4f}")
keff_tbl.write_parquet(ARTIFACTS / "keff_table.parquet")


DISCLOSE: measured PR at K=8 is 4.393 < 5.0. Proceed unchanged and disclose the value; the ladder is not re-cut, because F1-F5 leave no second consistent cut.

measured in 36s on each origin's training sub-block
shape: (4, 6)
┌─────┬──────────┬───────────┬───────────────┬─────────────┬────────────────┐
│ k   ┆ PR_raw   ┆ PR_raw_sd ┆ PR_windownorm ┆ stable_rank ┆ crosslag_share │
│ --- ┆ ---      ┆ ---       ┆ ---           ┆ ---         ┆ ---            │
│ i64 ┆ f64      ┆ f64       ┆ f64           ┆ f64         ┆ f64            │
╞═════╪══════════╪═══════════╪═══════════════╪═════════════╪════════════════╡
│ 1   ┆ 1.0      ┆ 0.0       ┆ 1.0           ┆ 1.0         ┆ 0.979891       │
│ 4   ┆ 3.327497 ┆ 0.10339   ┆ 3.316502      ┆ 2.355108    ┆ 0.078479       │
│ 8   ┆ 4.269219 ┆ 0.116815  ┆ 4.013051      ┆ 2.70075     ┆ 0.04731        │
│ 12  ┆ 3.983529 ┆ 0.306842  ┆ 3.654499      ┆ 2.172962    ┆ 0.020133       │
└─────┴──────────┴───────────┴───────────────┴─────────────┴────────────

##

<a id="section-05"></a>

<div style="background: linear-gradient(135deg, #150029, #22003d); border-left: 4px solid #c77dff; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #e0aaff; margin: 0 0 4px; font-size: 1.35em;">🧠 05 · Model, baseline, dan fungsi training</h2>
  <p style="color: #cbb2e8; margin: 0; font-size: 0.95em;">Kode arsitektur disalin dari thuml/iTransformer pada commit terkunci dan diverifikasi sha256; Ridge dari scikit-learn; satu loop latih untuk kedua Transformer.</p>
</div>

###

<div style="background: linear-gradient(90deg, #1f1300, #332000); border-left: 3px solid #e9c46a; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #f4d58d; font-size: 1em; margin: 0;">🔐 <code>upstream.py</code></h3> <p style="display: inline; color: #f0dfb0; font-size: 0.9em; margin: 0;">· Verifikasi sha256 dan pemuatan terisolasi kode thuml/iTransformer.</p></div>

In [16]:
"""The authors' iTransformer code, copied unchanged and loaded on demand.

``itr`` and ``vtr`` are the ``Model`` classes of ``model/iTransformer.py`` and
``model/Transformer.py`` in the official iTransformer repository. The files sit
under ``vendor/thuml_iTransformer`` as published, normalised only in whitespace:
LF line endings, one final newline, and whitespace-only lines emptied, which is
the form a notebook ``%%writefile`` cell produces. ``layers/SelfAttention_Family.py`` keeps
only ``FullAttention`` and ``AttentionLayer``; its imports of ``reformer_pytorch``
and ``einops`` serve attention variants this study does not use, and are the
only lines removed. :func:`derive` states that rule as code, so a reader can
rebuild every copy from the published files.

Both files define a class named ``Model`` and import siblings as ``layers.*``,
``model.*`` and ``utils.*``. :func:`load_upstream` therefore imports them in a
private module namespace and leaves ``sys.modules`` as it found it.

Upstream:
    Y. Liu, T. Hu, H. Zhang, H. Wu, S. Wang, L. Ma, and M. Long, "iTransformer:
    Inverted transformers are effective for time series forecasting," ICLR 2024.
    Code: https://github.com/thuml/iTransformer (MIT), commit
    c2426e68ca13f74aaec08045c5c724d8ad328124.
"""

UPSTREAM_REPO: Final = "https://github.com/thuml/iTransformer"
UPSTREAM_COMMIT: Final = "c2426e68ca13f74aaec08045c5c724d8ad328124"
UPSTREAM_LICENSE: Final = "MIT"

#: sha256 of each file as GitHub serves it at the commit.
UPSTREAM_RAW_SHA256: Final[dict[str, str]] = {
    "LICENSE": "29d2a4c09fa577780522219dc248466977f77cd420354c5e9a0e86550be2b849",
    "layers/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "layers/Embed.py": "2de7e5049a696d320a23a8bdb6e08e823cf7fa3f246a0a49555fcd729c5c8398",
    "layers/SelfAttention_Family.py": "38ac67428d528b8e145e4944ca059b40629a8446c891142602d5aaadc38dfd9e",
    "layers/Transformer_EncDec.py": "985c31f4ae187b08afaa9663dc960a40544b2ec33ab55389ceab3930edc1ce02",
    "model/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "model/iTransformer.py": "6f34777a5a12c253a293f79e7e1fd5b6f79ac100b443951e051e269e7e2542db",
    "model/Transformer.py": "7a816cc3971b99ee21d87479f25cc15498607d34539c95e5d53889588653ff4b",
    "utils/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "utils/masking.py": "02f453e52cfaec65e34132923add80e7132f188706680fa86d7bd884976e09ef",
}

#: Files that keep a subset of their top-level definitions, and the imports dropped
#: with the rest. Every other file is copied whole.
UPSTREAM_TRIMMED: Final[dict[str, tuple[tuple[str, ...], tuple[str, ...]]]] = {
    "layers/SelfAttention_Family.py": (
        ("FullAttention", "AttentionLayer"),
        ("reformer_pytorch", "einops"),
    ),
}

#: sha256 of each copy under ``vendor/thuml_iTransformer``, as :func:`derive` builds it.
UPSTREAM_FILES: Final[dict[str, str]] = {
    "LICENSE": "29d2a4c09fa577780522219dc248466977f77cd420354c5e9a0e86550be2b849",
    "layers/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "layers/Embed.py": "91f50ec4e47a5fbce5c1a064614140b0d8ac1d01bb2500b0d7afe3d2f0a0885b",
    "layers/SelfAttention_Family.py": "d6643899c34a69d59b4ea4194944533f5d47d14b2ac4841e4a41047c4589b471",
    "layers/Transformer_EncDec.py": "985c31f4ae187b08afaa9663dc960a40544b2ec33ab55389ceab3930edc1ce02",
    "model/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "model/iTransformer.py": "48b424b0e3d77eec3610ca79f083b415c8ba96032ee598ad3d05383880e768e4",
    "model/Transformer.py": "7a816cc3971b99ee21d87479f25cc15498607d34539c95e5d53889588653ff4b",
    "utils/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "utils/masking.py": "02f453e52cfaec65e34132923add80e7132f188706680fa86d7bd884976e09ef",
}

#: Top-level package names the upstream files import each other through.
_NAMESPACES: Final = ("layers", "model", "utils")

#: The two classes, once :func:`load_upstream` has run.
_LOADED: dict[str, type] = {}


def permalink(path: str) -> str:
    """The file's page on GitHub at the pinned commit."""
    return f"{UPSTREAM_REPO}/blob/{UPSTREAM_COMMIT}/{path}"


def raw_url(path: str) -> str:
    """The file's raw bytes at the pinned commit."""
    return f"https://raw.githubusercontent.com/thuml/iTransformer/{UPSTREAM_COMMIT}/{path}"


def canonical(text: str) -> str:
    """LF line endings, whitespace-only lines emptied, one trailing newline.

    This is the form a file keeps after a notebook ``%%writefile`` cell: IPython
    dedents a cell before running it, which empties whitespace-only lines, and a
    cell cannot end without a newline. An empty file stays empty.
    """
    text = re.sub(r"^[ \t]+$", "", text.replace("\r\n", "\n"), flags=re.M)
    return text.rstrip("\n") + "\n" if text.strip() else ""


def derive(path: str, raw: str) -> str:
    """The copy of ``path`` this study uses, built from the published text.

    A whole file is only canonicalised. A trimmed file keeps its imports, minus
    the dropped modules, and the named top-level definitions with the blank lines
    that precede them. Lines are only ever deleted, never edited or added.
    """
    text = canonical(raw)
    if path not in UPSTREAM_TRIMMED:
        return text
    keep, dropped = UPSTREAM_TRIMMED[path]
    lines = text.splitlines(keepends=True)
    kept: set[int] = set()
    for node in ast.parse(text).body:
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            roots = ([node.module or ""] if isinstance(node, ast.ImportFrom)
                     else [alias.name for alias in node.names])
            if not any(root.split(".")[0] in dropped for root in roots):
                kept.update(range(node.lineno - 1, node.end_lineno))
        elif isinstance(node, (ast.ClassDef, ast.FunctionDef)) and node.name in keep:
            first = node.lineno - 1
            while first > 0 and not lines[first - 1].strip():
                first -= 1
            kept.update(range(first, node.end_lineno))
    return canonical("".join(lines[i] for i in sorted(kept)))


def vendor_root() -> Path:
    """Where the copies live in a source checkout."""
    return Path(__file__).resolve().parent / "vendor" / "thuml_iTransformer"


def verify_upstream(root: Path) -> None:
    """Raise unless every copy under ``root`` matches its pinned digest."""
    bad = []
    for path, expected in UPSTREAM_FILES.items():
        file = Path(root) / path
        actual = (hashlib.sha256(canonical(file.read_text(encoding="utf-8")).encode("utf-8")).hexdigest()
                  if file.is_file() else "missing")
        if actual != expected:
            bad.append(f"{path}: {actual}")
    if bad:
        raise ValueError(f"upstream copies differ from commit {UPSTREAM_COMMIT[:8]}: {bad}")


def load_upstream(root: Path) -> dict[str, type]:
    """Verify the copies under ``root``, then import both ``Model`` classes.

    Any module already registered under ``layers``, ``model`` or ``utils`` is set
    aside and restored afterwards, and the upstream modules are unregistered, so
    no other library can later import them by accident.
    """
    root = Path(root)
    verify_upstream(root)
    owned = lambda name: name.split(".")[0] in _NAMESPACES  # noqa: E731
    saved = {name: module for name, module in sys.modules.items() if owned(name)}
    for name in saved:
        del sys.modules[name]
    sys.path.insert(0, str(root))
    try:
        loaded = {
            "itr": importlib.import_module("model.iTransformer").Model,
            "vtr": importlib.import_module("model.Transformer").Model,
        }
    finally:
        sys.path.remove(str(root))
        for name in [name for name in sys.modules if owned(name)]:
            del sys.modules[name]
        sys.modules.update(saved)
    _LOADED.update(loaded)
    return dict(loaded)


def upstream_model(name: str) -> type:
    """``"itr"`` or ``"vtr"``: the upstream class, loading the checkout's copies if needed."""
    if not _LOADED:
        load_upstream(vendor_root())
    return _LOADED[name]


###

<div style="background: linear-gradient(90deg, #1f1300, #332000); border-left: 3px solid #e9c46a; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #f4d58d; font-size: 1em; margin: 0;">📦 <code>LICENSE</code></h3> <p style="display: inline; color: #f0dfb0; font-size: 0.9em; margin: 0;">· <a href="https://github.com/thuml/iTransformer/blob/c2426e68ca13f74aaec08045c5c724d8ad328124/LICENSE" style="color: #f4d58d;">thuml/iTransformer@c2426e6</a> · MIT · lisensi repositori, disertakan bersama salinan</p></div>

In [17]:
%%writefile vendor/thuml_iTransformer/LICENSE
MIT License

Copyright (c) 2022 THUML @ Tsinghua University

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

Writing vendor/thuml_iTransformer/LICENSE


###

<div style="background: linear-gradient(90deg, #1f1300, #332000); border-left: 3px solid #e9c46a; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #f4d58d; font-size: 1em; margin: 0;">📦 <code>layers/Embed.py</code></h3> <p style="display: inline; color: #f0dfb0; font-size: 0.9em; margin: 0;">· <a href="https://github.com/thuml/iTransformer/blob/c2426e68ca13f74aaec08045c5c724d8ad328124/layers/Embed.py" style="color: #f4d58d;">thuml/iTransformer@c2426e6</a> · MIT · tidak diubah</p></div>

In [18]:
%%writefile vendor/thuml_iTransformer/layers/Embed.py
import torch
import torch.nn as nn
import math


class PositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEmbedding, self).__init__()
        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model).float()
        pe.require_grad = False

        position = torch.arange(0, max_len).float().unsqueeze(1)
        div_term = (torch.arange(0, d_model, 2).float()
                    * -(math.log(10000.0) / d_model)).exp()

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return self.pe[:, :x.size(1)]


class TokenEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(TokenEmbedding, self).__init__()
        padding = 1 if torch.__version__ >= '1.5.0' else 2
        self.tokenConv = nn.Conv1d(in_channels=c_in, out_channels=d_model,
                                   kernel_size=3, padding=padding, padding_mode='circular', bias=False)
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(
                    m.weight, mode='fan_in', nonlinearity='leaky_relu')

    def forward(self, x):
        x = self.tokenConv(x.permute(0, 2, 1)).transpose(1, 2)
        return x


class FixedEmbedding(nn.Module):
    def __init__(self, c_in, d_model):
        super(FixedEmbedding, self).__init__()

        w = torch.zeros(c_in, d_model).float()
        w.require_grad = False

        position = torch.arange(0, c_in).float().unsqueeze(1)
        div_term = (torch.arange(0, d_model, 2).float()
                    * -(math.log(10000.0) / d_model)).exp()

        w[:, 0::2] = torch.sin(position * div_term)
        w[:, 1::2] = torch.cos(position * div_term)

        self.emb = nn.Embedding(c_in, d_model)
        self.emb.weight = nn.Parameter(w, requires_grad=False)

    def forward(self, x):
        return self.emb(x).detach()


class TemporalEmbedding(nn.Module):
    def __init__(self, d_model, embed_type='fixed', freq='h'):
        super(TemporalEmbedding, self).__init__()

        minute_size = 4
        hour_size = 24
        weekday_size = 7
        day_size = 32
        month_size = 13

        Embed = FixedEmbedding if embed_type == 'fixed' else nn.Embedding
        if freq == 't':
            self.minute_embed = Embed(minute_size, d_model)
        self.hour_embed = Embed(hour_size, d_model)
        self.weekday_embed = Embed(weekday_size, d_model)
        self.day_embed = Embed(day_size, d_model)
        self.month_embed = Embed(month_size, d_model)

    def forward(self, x):
        x = x.long()
        minute_x = self.minute_embed(x[:, :, 4]) if hasattr(
            self, 'minute_embed') else 0.
        hour_x = self.hour_embed(x[:, :, 3])
        weekday_x = self.weekday_embed(x[:, :, 2])
        day_x = self.day_embed(x[:, :, 1])
        month_x = self.month_embed(x[:, :, 0])

        return hour_x + weekday_x + day_x + month_x + minute_x


class TimeFeatureEmbedding(nn.Module):
    def __init__(self, d_model, embed_type='timeF', freq='h'):
        super(TimeFeatureEmbedding, self).__init__()

        freq_map = {'h': 4, 't': 5, 's': 6,
                    'm': 1, 'a': 1, 'w': 2, 'd': 3, 'b': 3}
        d_inp = freq_map[freq]
        self.embed = nn.Linear(d_inp, d_model, bias=False)

    def forward(self, x):
        return self.embed(x)


class DataEmbedding(nn.Module):
    def __init__(self, c_in, d_model, embed_type='fixed', freq='h', dropout=0.1):
        super(DataEmbedding, self).__init__()

        self.value_embedding = TokenEmbedding(c_in=c_in, d_model=d_model)
        self.position_embedding = PositionalEmbedding(d_model=d_model)
        self.temporal_embedding = TemporalEmbedding(d_model=d_model, embed_type=embed_type,
                                                    freq=freq) if embed_type != 'timeF' else TimeFeatureEmbedding(
            d_model=d_model, embed_type=embed_type, freq=freq)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, x_mark):
        if x_mark is None:
            x = self.value_embedding(x) + self.position_embedding(x)
        else:
            x = self.value_embedding(
                x) + self.temporal_embedding(x_mark) + self.position_embedding(x)
        return self.dropout(x)


class DataEmbedding_inverted(nn.Module):
    def __init__(self, c_in, d_model, embed_type='fixed', freq='h', dropout=0.1):
        super(DataEmbedding_inverted, self).__init__()
        self.value_embedding = nn.Linear(c_in, d_model)
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x, x_mark):
        x = x.permute(0, 2, 1)
        # x: [Batch Variate Time]
        if x_mark is None:
            x = self.value_embedding(x)
        else:
            # the potential to take covariates (e.g. timestamps) as tokens
            x = self.value_embedding(torch.cat([x, x_mark.permute(0, 2, 1)], 1)) 
        # x: [Batch Variate d_model]
        return self.dropout(x)

Writing vendor/thuml_iTransformer/layers/Embed.py


###

<div style="background: linear-gradient(90deg, #1f1300, #332000); border-left: 3px solid #e9c46a; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #f4d58d; font-size: 1em; margin: 0;">📦 <code>layers/SelfAttention_Family.py</code></h3> <p style="display: inline; color: #f0dfb0; font-size: 0.9em; margin: 0;">· <a href="https://github.com/thuml/iTransformer/blob/c2426e68ca13f74aaec08045c5c724d8ad328124/layers/SelfAttention_Family.py" style="color: #f4d58d;">thuml/iTransformer@c2426e6</a> · MIT · hanya FullAttention, AttentionLayer dipertahankan; impor reformer_pytorch, einops dihapus</p></div>

In [19]:
%%writefile vendor/thuml_iTransformer/layers/SelfAttention_Family.py
import torch
import torch.nn as nn
import numpy as np
from math import sqrt
from utils.masking import TriangularCausalMask, ProbMask


class FullAttention(nn.Module):
    def __init__(self, mask_flag=True, factor=5, scale=None, attention_dropout=0.1, output_attention=False):
        super(FullAttention, self).__init__()
        self.scale = scale
        self.mask_flag = mask_flag
        self.output_attention = output_attention
        self.dropout = nn.Dropout(attention_dropout)

    def forward(self, queries, keys, values, attn_mask, tau=None, delta=None):
        B, L, H, E = queries.shape
        _, S, _, D = values.shape
        scale = self.scale or 1. / sqrt(E)

        scores = torch.einsum("blhe,bshe->bhls", queries, keys)

        if self.mask_flag:
            if attn_mask is None:
                attn_mask = TriangularCausalMask(B, L, device=queries.device)

            scores.masked_fill_(attn_mask.mask, -np.inf)

        A = self.dropout(torch.softmax(scale * scores, dim=-1))
        V = torch.einsum("bhls,bshd->blhd", A, values)

        if self.output_attention:
            return (V.contiguous(), A)
        else:
            return (V.contiguous(), None)


class AttentionLayer(nn.Module):
    def __init__(self, attention, d_model, n_heads, d_keys=None,
                 d_values=None):
        super(AttentionLayer, self).__init__()

        d_keys = d_keys or (d_model // n_heads)
        d_values = d_values or (d_model // n_heads)

        self.inner_attention = attention
        self.query_projection = nn.Linear(d_model, d_keys * n_heads)
        self.key_projection = nn.Linear(d_model, d_keys * n_heads)
        self.value_projection = nn.Linear(d_model, d_values * n_heads)
        self.out_projection = nn.Linear(d_values * n_heads, d_model)
        self.n_heads = n_heads

    def forward(self, queries, keys, values, attn_mask, tau=None, delta=None):
        B, L, _ = queries.shape
        _, S, _ = keys.shape
        H = self.n_heads

        queries = self.query_projection(queries).view(B, L, H, -1)
        keys = self.key_projection(keys).view(B, S, H, -1)
        values = self.value_projection(values).view(B, S, H, -1)

        out, attn = self.inner_attention(
            queries,
            keys,
            values,
            attn_mask,
            tau=tau,
            delta=delta
        )
        out = out.view(B, L, -1)

        return self.out_projection(out), attn

Writing vendor/thuml_iTransformer/layers/SelfAttention_Family.py


###

<div style="background: linear-gradient(90deg, #1f1300, #332000); border-left: 3px solid #e9c46a; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #f4d58d; font-size: 1em; margin: 0;">📦 <code>layers/Transformer_EncDec.py</code></h3> <p style="display: inline; color: #f0dfb0; font-size: 0.9em; margin: 0;">· <a href="https://github.com/thuml/iTransformer/blob/c2426e68ca13f74aaec08045c5c724d8ad328124/layers/Transformer_EncDec.py" style="color: #f4d58d;">thuml/iTransformer@c2426e6</a> · MIT · tidak diubah</p></div>

In [20]:
%%writefile vendor/thuml_iTransformer/layers/Transformer_EncDec.py
import torch.nn as nn
import torch.nn.functional as F


class ConvLayer(nn.Module):
    def __init__(self, c_in):
        super(ConvLayer, self).__init__()
        self.downConv = nn.Conv1d(in_channels=c_in,
                                  out_channels=c_in,
                                  kernel_size=3,
                                  padding=2,
                                  padding_mode='circular')
        self.norm = nn.BatchNorm1d(c_in)
        self.activation = nn.ELU()
        self.maxPool = nn.MaxPool1d(kernel_size=3, stride=2, padding=1)

    def forward(self, x):
        x = self.downConv(x.permute(0, 2, 1))
        x = self.norm(x)
        x = self.activation(x)
        x = self.maxPool(x)
        x = x.transpose(1, 2)
        return x


class EncoderLayer(nn.Module):
    def __init__(self, attention, d_model, d_ff=None, dropout=0.1, activation="relu"):
        super(EncoderLayer, self).__init__()
        d_ff = d_ff or 4 * d_model
        self.attention = attention
        self.conv1 = nn.Conv1d(in_channels=d_model, out_channels=d_ff, kernel_size=1)
        self.conv2 = nn.Conv1d(in_channels=d_ff, out_channels=d_model, kernel_size=1)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == "relu" else F.gelu

    def forward(self, x, attn_mask=None, tau=None, delta=None):
        new_x, attn = self.attention(
            x, x, x,
            attn_mask=attn_mask,
            tau=tau, delta=delta
        )
        x = x + self.dropout(new_x)

        y = x = self.norm1(x)
        y = self.dropout(self.activation(self.conv1(y.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))

        return self.norm2(x + y), attn


class Encoder(nn.Module):
    def __init__(self, attn_layers, conv_layers=None, norm_layer=None):
        super(Encoder, self).__init__()
        self.attn_layers = nn.ModuleList(attn_layers)
        self.conv_layers = nn.ModuleList(conv_layers) if conv_layers is not None else None
        self.norm = norm_layer

    def forward(self, x, attn_mask=None, tau=None, delta=None):
        # x [B, L, D]
        attns = []
        if self.conv_layers is not None:
            for i, (attn_layer, conv_layer) in enumerate(zip(self.attn_layers, self.conv_layers)):
                delta = delta if i == 0 else None
                x, attn = attn_layer(x, attn_mask=attn_mask, tau=tau, delta=delta)
                x = conv_layer(x)
                attns.append(attn)
            x, attn = self.attn_layers[-1](x, tau=tau, delta=None)
            attns.append(attn)
        else:
            for attn_layer in self.attn_layers:
                x, attn = attn_layer(x, attn_mask=attn_mask, tau=tau, delta=delta)
                attns.append(attn)

        if self.norm is not None:
            x = self.norm(x)

        return x, attns


class DecoderLayer(nn.Module):
    def __init__(self, self_attention, cross_attention, d_model, d_ff=None,
                 dropout=0.1, activation="relu"):
        super(DecoderLayer, self).__init__()
        d_ff = d_ff or 4 * d_model
        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.conv1 = nn.Conv1d(in_channels=d_model, out_channels=d_ff, kernel_size=1)
        self.conv2 = nn.Conv1d(in_channels=d_ff, out_channels=d_model, kernel_size=1)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = F.relu if activation == "relu" else F.gelu

    def forward(self, x, cross, x_mask=None, cross_mask=None, tau=None, delta=None):
        x = x + self.dropout(self.self_attention(
            x, x, x,
            attn_mask=x_mask,
            tau=tau, delta=None
        )[0])
        x = self.norm1(x)

        x = x + self.dropout(self.cross_attention(
            x, cross, cross,
            attn_mask=cross_mask,
            tau=tau, delta=delta
        )[0])

        y = x = self.norm2(x)
        y = self.dropout(self.activation(self.conv1(y.transpose(-1, 1))))
        y = self.dropout(self.conv2(y).transpose(-1, 1))

        return self.norm3(x + y)


class Decoder(nn.Module):
    def __init__(self, layers, norm_layer=None, projection=None):
        super(Decoder, self).__init__()
        self.layers = nn.ModuleList(layers)
        self.norm = norm_layer
        self.projection = projection

    def forward(self, x, cross, x_mask=None, cross_mask=None, tau=None, delta=None):
        for layer in self.layers:
            x = layer(x, cross, x_mask=x_mask, cross_mask=cross_mask, tau=tau, delta=delta)

        if self.norm is not None:
            x = self.norm(x)

        if self.projection is not None:
            x = self.projection(x)
        return x

Writing vendor/thuml_iTransformer/layers/Transformer_EncDec.py


###

<div style="background: linear-gradient(90deg, #1f1300, #332000); border-left: 3px solid #e9c46a; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #f4d58d; font-size: 1em; margin: 0;">📦 <code>utils/masking.py</code></h3> <p style="display: inline; color: #f0dfb0; font-size: 0.9em; margin: 0;">· <a href="https://github.com/thuml/iTransformer/blob/c2426e68ca13f74aaec08045c5c724d8ad328124/utils/masking.py" style="color: #f4d58d;">thuml/iTransformer@c2426e6</a> · MIT · tidak diubah</p></div>

In [21]:
%%writefile vendor/thuml_iTransformer/utils/masking.py
import torch


class TriangularCausalMask():
    def __init__(self, B, L, device="cpu"):
        mask_shape = [B, 1, L, L]
        with torch.no_grad():
            self._mask = torch.triu(torch.ones(mask_shape, dtype=torch.bool), diagonal=1).to(device)

    @property
    def mask(self):
        return self._mask


class ProbMask():
    def __init__(self, B, H, L, index, scores, device="cpu"):
        _mask = torch.ones(L, scores.shape[-1], dtype=torch.bool).to(device).triu(1)
        _mask_ex = _mask[None, None, :].expand(B, H, L, scores.shape[-1])
        indicator = _mask_ex[torch.arange(B)[:, None, None],
                    torch.arange(H)[None, :, None],
                    index, :].to(device)
        self._mask = indicator.view(scores.shape).to(device)

    @property
    def mask(self):
        return self._mask

Writing vendor/thuml_iTransformer/utils/masking.py


###

<div style="background: linear-gradient(90deg, #1f1300, #332000); border-left: 3px solid #e9c46a; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #f4d58d; font-size: 1em; margin: 0;">📦 <code>model/iTransformer.py</code></h3> <p style="display: inline; color: #f0dfb0; font-size: 0.9em; margin: 0;">· <a href="https://github.com/thuml/iTransformer/blob/c2426e68ca13f74aaec08045c5c724d8ad328124/model/iTransformer.py" style="color: #f4d58d;">thuml/iTransformer@c2426e6</a> · MIT · tidak diubah, kecuali 2 baris berisi spasi saja dikosongkan</p></div>

In [22]:
%%writefile vendor/thuml_iTransformer/model/iTransformer.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from layers.Transformer_EncDec import Encoder, EncoderLayer
from layers.SelfAttention_Family import FullAttention, AttentionLayer
from layers.Embed import DataEmbedding_inverted
import numpy as np


class Model(nn.Module):
    """
    Paper link: https://arxiv.org/abs/2310.06625
    """

    def __init__(self, configs):
        super(Model, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.output_attention = configs.output_attention
        self.use_norm = configs.use_norm
        # Embedding
        self.enc_embedding = DataEmbedding_inverted(configs.seq_len, configs.d_model, configs.embed, configs.freq,
                                                    configs.dropout)
        self.class_strategy = configs.class_strategy
        # Encoder-only architecture
        self.encoder = Encoder(
            [
                EncoderLayer(
                    AttentionLayer(
                        FullAttention(False, configs.factor, attention_dropout=configs.dropout,
                                      output_attention=configs.output_attention), configs.d_model, configs.n_heads),
                    configs.d_model,
                    configs.d_ff,
                    dropout=configs.dropout,
                    activation=configs.activation
                ) for l in range(configs.e_layers)
            ],
            norm_layer=torch.nn.LayerNorm(configs.d_model)
        )
        self.projector = nn.Linear(configs.d_model, configs.pred_len, bias=True)

    def forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):
        if self.use_norm:
            # Normalization from Non-stationary Transformer
            means = x_enc.mean(1, keepdim=True).detach()
            x_enc = x_enc - means
            stdev = torch.sqrt(torch.var(x_enc, dim=1, keepdim=True, unbiased=False) + 1e-5)
            x_enc /= stdev

        _, _, N = x_enc.shape # B L N
        # B: batch_size;    E: d_model; 
        # L: seq_len;       S: pred_len;
        # N: number of variate (tokens), can also includes covariates

        # Embedding
        # B L N -> B N E                (B L N -> B L E in the vanilla Transformer)
        enc_out = self.enc_embedding(x_enc, x_mark_enc) # covariates (e.g timestamp) can be also embedded as tokens

        # B N E -> B N E                (B L E -> B L E in the vanilla Transformer)
        # the dimensions of embedded time series has been inverted, and then processed by native attn, layernorm and ffn modules
        enc_out, attns = self.encoder(enc_out, attn_mask=None)

        # B N E -> B N S -> B S N 
        dec_out = self.projector(enc_out).permute(0, 2, 1)[:, :, :N] # filter the covariates

        if self.use_norm:
            # De-Normalization from Non-stationary Transformer
            dec_out = dec_out * (stdev[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))
            dec_out = dec_out + (means[:, 0, :].unsqueeze(1).repeat(1, self.pred_len, 1))

        return dec_out, attns


    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):
        dec_out, attns = self.forecast(x_enc, x_mark_enc, x_dec, x_mark_dec)

        if self.output_attention:
            return dec_out[:, -self.pred_len:, :], attns
        else:
            return dec_out[:, -self.pred_len:, :]  # [B, L, D]

Writing vendor/thuml_iTransformer/model/iTransformer.py


###

<div style="background: linear-gradient(90deg, #1f1300, #332000); border-left: 3px solid #e9c46a; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #f4d58d; font-size: 1em; margin: 0;">📦 <code>model/Transformer.py</code></h3> <p style="display: inline; color: #f0dfb0; font-size: 0.9em; margin: 0;">· <a href="https://github.com/thuml/iTransformer/blob/c2426e68ca13f74aaec08045c5c724d8ad328124/model/Transformer.py" style="color: #f4d58d;">thuml/iTransformer@c2426e6</a> · MIT · tidak diubah</p></div>

In [23]:
%%writefile vendor/thuml_iTransformer/model/Transformer.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from layers.Transformer_EncDec import Decoder, DecoderLayer, Encoder, EncoderLayer, ConvLayer
from layers.SelfAttention_Family import FullAttention, AttentionLayer
from layers.Embed import DataEmbedding
import numpy as np


class Model(nn.Module):
    """
    Vanilla Transformer
    with O(L^2) complexity
    Paper link: https://proceedings.neurips.cc/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf
    """

    def __init__(self, configs):
        super(Model, self).__init__()
        self.pred_len = configs.pred_len
        self.output_attention = configs.output_attention

        if configs.channel_independence:
            self.enc_in = 1
            self.dec_in = 1
            self.c_out = 1
        else:
            self.enc_in = configs.enc_in
            self.dec_in = configs.dec_in
            self.c_out = configs.c_out

        # Embedding
        self.enc_embedding = DataEmbedding(self.enc_in, configs.d_model, configs.embed, configs.freq,
                                           configs.dropout)
        # Encoder
        self.encoder = Encoder(
            [
                EncoderLayer(
                    AttentionLayer(
                        FullAttention(False, configs.factor, attention_dropout=configs.dropout,
                                      output_attention=configs.output_attention), configs.d_model, configs.n_heads),
                    configs.d_model,
                    configs.d_ff,
                    dropout=configs.dropout,
                    activation=configs.activation
                ) for l in range(configs.e_layers)
            ],
            norm_layer=torch.nn.LayerNorm(configs.d_model)
        )
        # Decoder
        self.dec_embedding = DataEmbedding(self.dec_in, configs.d_model, configs.embed, configs.freq,
                                           configs.dropout)
        self.decoder = Decoder(
            [
                DecoderLayer(
                    AttentionLayer(
                        FullAttention(True, configs.factor, attention_dropout=configs.dropout,
                                      output_attention=False),
                        configs.d_model, configs.n_heads),
                    AttentionLayer(
                        FullAttention(False, configs.factor, attention_dropout=configs.dropout,
                                      output_attention=False),
                        configs.d_model, configs.n_heads),
                    configs.d_model,
                    configs.d_ff,
                    dropout=configs.dropout,
                    activation=configs.activation,
                )
                for l in range(configs.d_layers)
            ],
            norm_layer=torch.nn.LayerNorm(configs.d_model),
            projection=nn.Linear(configs.d_model, configs.c_out, bias=True)
        )

    def forecast(self, x_enc, x_mark_enc, x_dec, x_mark_dec):
        # Embedding
        enc_out = self.enc_embedding(x_enc, x_mark_enc)
        enc_out, attns = self.encoder(enc_out, attn_mask=None)

        dec_out = self.dec_embedding(x_dec, x_mark_dec)
        dec_out = self.decoder(dec_out, enc_out, x_mask=None, cross_mask=None)
        return dec_out

    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):
        dec_out = self.forecast(x_enc, x_mark_enc, x_dec, x_mark_dec)
        return dec_out[:, -self.pred_len:, :]  # [B, L, D]

Writing vendor/thuml_iTransformer/model/Transformer.py


###

<div style="background: linear-gradient(90deg, #1f1300, #332000); border-left: 3px solid #e9c46a; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #f4d58d; font-size: 1em; margin: 0;">🔐 Verifikasi dan muat kode upstream</h3> <p style="display: inline; color: #f0dfb0; font-size: 0.9em; margin: 0;">· Setiap salinan harus cocok dengan sha256 yang dipin sebelum model dibangun.</p></div>

<div style="background: #0e0e12; border-left: 3px solid #6c757d; border-radius: 0 6px 6px 0; padding: 4px 14px; font-size: 0.82em; color: #cfcfcf;"><span style="color: #9ec5fe; font-weight: 600;">MEMBACA</span> <code>vendor/thuml_iTransformer/**</code></div>

In [24]:
verify_upstream(VENDOR)
UPSTREAM = load_upstream(VENDOR)
print(f"{UPSTREAM_REPO} at {UPSTREAM_COMMIT} ({UPSTREAM_LICENSE})")
for _path in sorted(UPSTREAM_FILES):
    _state = "imports trimmed" if _path in UPSTREAM_TRIMMED else "unchanged"
    print(f"  {_path:32s} sha256 {UPSTREAM_FILES[_path][:12]}  {_state}")
print({tag: f"{cls.__module__}.{cls.__qualname__}" for tag, cls in UPSTREAM.items()})


https://github.com/thuml/iTransformer at c2426e68ca13f74aaec08045c5c724d8ad328124 (MIT)
  LICENSE                          sha256 29d2a4c09fa5  unchanged
  layers/Embed.py                  sha256 91f50ec4e47a  unchanged
  layers/SelfAttention_Family.py   sha256 d6643899c34a  imports trimmed
  layers/Transformer_EncDec.py     sha256 985c31f4ae18  unchanged
  layers/__init__.py               sha256 e3b0c44298fc  unchanged
  model/Transformer.py             sha256 7a816cc3971b  unchanged
  model/__init__.py                sha256 e3b0c44298fc  unchanged
  model/iTransformer.py            sha256 48b424b0e3d7  unchanged
  utils/__init__.py                sha256 e3b0c44298fc  unchanged
  utils/masking.py                 sha256 02f453e52cfa  unchanged
{'itr': 'model.iTransformer.Model', 'vtr': 'model.Transformer.Model'}


###

<div style="background: linear-gradient(90deg, #150029, #22003d); border-left: 3px solid #c77dff; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #e0aaff; font-size: 1em; margin: 0;">🧠 <code>model.py</code></h3> <p style="display: inline; color: #cbb2e8; font-size: 0.9em; margin: 0;">· Adapter iTransformer dan Transformer vanilla ke kelas upstream.</p></div>

In [25]:
"""The two transformer models: the authors' code behind the study's training interface.

``itr`` and ``vtr`` are the ``Model`` classes of the official iTransformer
repository, copied unchanged (see ``upstream.py``). This module only adapts them.
It builds the ``configs`` namespace the upstream constructors read, passes
``x_mark=None`` because no calendar feature enters the study, gives the vanilla
decoder its start token, and reads the target channel ``r`` from the all-channel
output. The loss is MSE on that channel at every rung.

Upstream:
    Code: https://github.com/thuml/iTransformer (MIT), commit
    c2426e68ca13f74aaec08045c5c724d8ad328124. Y. Liu et al., "iTransformer:
    Inverted transformers are effective for time series forecasting," ICLR 2024;
    A. Vaswani et al., "Attention is all you need," NeurIPS 2017. Settings follow
    the official defaults (2 encoder layers, 1 decoder layer, gelu, label_len 48,
    8 heads, dropout 0.1), with d_model 128 and d_ff 256 for the sample size.
"""

def _count(module: nn.Module) -> int:
    return sum(p.numel() for p in module.parameters() if p.requires_grad)


@dataclass(frozen=True, slots=True)
class ITransformerConfig:
    """iTransformer settings, named as the upstream ``configs`` fields.

    Fixed before any run and identical at every rung. The parameter count does
    not depend on K: K changes the number of tokens, not a weight shape.
    """

    seq_len: int = SEQ_LEN
    pred_len: int = PRED_LEN
    d_model: int = 128
    d_ff: int = 256
    e_layers: int = 2
    n_heads: int = 8
    dropout: float = 0.1
    activation: str = "gelu"
    use_norm: bool = True
    factor: int = 1
    embed: str = "timeF"
    freq: str = "h"
    class_strategy: str = "projection"

    def upstream_configs(self) -> SimpleNamespace:
        """The ``configs`` object ``model/iTransformer.py`` reads."""
        return SimpleNamespace(**asdict(self), output_attention=False)

    def build(self) -> "ITransformerForecaster":
        return ITransformerForecaster(self)

    def schedule(self) -> "TrainSchedule":

        return TrainSchedule()

    def fit(self, tensors, spec, *, device=None):
        """Train one cell; nothing is selected, so the config comes back unchanged."""

        model, outcome = train_one(tensors, spec, self, device=device)
        return model, self, outcome


class ITransformerForecaster(nn.Module):
    """``(B, L, K) -> (B, H)``: the upstream iTransformer, read at the target channel."""

    def __init__(self, cfg: ITransformerConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.model = upstream_model("itr")(cfg.upstream_configs())

    def forward(self, x: Tensor) -> Tensor:
        """Every channel's forecast, ``(B, H, K)``."""
        return self.model(x, None, None, None)

    def forecast_target(self, x: Tensor) -> Tensor:
        return self.forward(x)[:, :, TARGET_INDEX]

    def n_parameters(self) -> int:
        return _count(self)


@dataclass(frozen=True, slots=True)
class VanillaConfig:
    """Vanilla Transformer settings, named as the upstream ``configs`` fields.

    ``k`` is a field because the embeddings and the output head are K channels wide.
    """

    seq_len: int = SEQ_LEN
    pred_len: int = PRED_LEN
    label_len: int = 48
    k: int = 8
    d_model: int = 128
    d_ff: int = 256
    e_layers: int = 2
    d_layers: int = 1
    n_heads: int = 8
    dropout: float = 0.1
    activation: str = "gelu"
    factor: int = 1
    embed: str = "timeF"
    freq: str = "h"

    def upstream_configs(self) -> SimpleNamespace:
        """The ``configs`` object ``model/Transformer.py`` reads."""
        return SimpleNamespace(
            **asdict(self), enc_in=self.k, dec_in=self.k, c_out=self.k,
            channel_independence=False, output_attention=False,
        )

    def build(self) -> "VanillaForecaster":
        return VanillaForecaster(self)

    def schedule(self) -> "TrainSchedule":

        return TrainSchedule()

    def fit(self, tensors, spec, *, device=None):
        """Train one cell; nothing is selected, so the config comes back unchanged."""

        model, outcome = train_one(tensors, spec, self, device=device)
        return model, self, outcome


class VanillaForecaster(nn.Module):
    """``(B, L, K) -> (B, H)``: the upstream vanilla Transformer, read at the target channel.

    The decoder input is the last ``label_len`` lookback hours followed by
    ``pred_len`` zero placeholders, so every hour it reads is known at the
    forecast origin and all 24 hours are decoded in one pass.
    """

    def __init__(self, cfg: VanillaConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.model = upstream_model("vtr")(cfg.upstream_configs())

    def decoder_input(self, x: Tensor) -> Tensor:
        """``(B, label_len + pred_len, K)``: the known start token, then zeros."""
        placeholders = x.new_zeros(x.shape[0], self.cfg.pred_len, x.shape[2])
        return torch.cat([x[:, -self.cfg.label_len:, :], placeholders], dim=1)

    def forward(self, x: Tensor) -> Tensor:
        """Every channel's forecast, ``(B, H, K)``."""
        return self.model(x, None, self.decoder_input(x), None)

    def forecast_target(self, x: Tensor) -> Tensor:
        return self.forward(x)[:, :, TARGET_INDEX]

    def n_parameters(self) -> int:
        return _count(self)


###

<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af2; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #e0aaff; font-size: 1em; margin: 0;">🏋️ <code>train.py</code></h3> <p style="display: inline; color: #cbb2e8; font-size: 0.9em; margin: 0;">· Loop latih, checkpoint per epoch, digest kode, dan artefak run.</p></div>

In [26]:
"""Training loop, run identity, and the files every run leaves behind.

Each split is loaded onto the device once and batched by index slicing, with no
``Dataset`` or ``DataLoader``. Shuffling permutes an index tensor on the device.
Every run persists its raw predictions, its best weights and a metadata record
naming the code, input, configuration and environment that produced them.

Upstream:
    ``torch.optim.Adam`` (D. P. Kingma and J. Ba, ICLR 2015) at ``lr = 1e-4`` and
    ``torch.optim.lr_scheduler.StepLR`` halving every four epochs, from PyTorch
    (https://docs.pytorch.org/docs/stable/optim.html, BSD-3-Clause). Halving every
    epoch would make the 30-epoch cap unreachable. The loop around them is
    written here.
"""

ARTIFACTS: Path = Path("artifacts")

DEFAULT_PARQUET: Path = Path("data/raw/BTCUSDT_1h.parquet")

#: Environment variable naming the input parquet a process consumed.
INPUT_PARQUET_ENV: str = "ITBTC_PARQUET"

#: Code digest pinned by a launcher that has no package files to hash (the notebook).
CODE_SHA256_OVERRIDE: str | None = None

#: Serialises seeding and model construction, the only steps that share the CPU
#: generator across the per-GPU workers.
SEED_LOCK = threading.Lock()


def set_seed(seed: int, device: torch.device | None = None) -> None:
    """Seed Python, NumPy, the CPU generator and one CUDA device; force deterministic cuDNN.

    Seeding only the given device keeps two concurrent workers from resetting
    each other's CUDA stream.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.default_generator.manual_seed(seed)
    if torch.cuda.is_available():
        if device is not None and torch.device(device).type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.manual_seed(seed)
        else:
            torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def pick_device() -> torch.device:
    """Prefer CUDA; fall back to CPU."""
    return torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")


def supports_native_bf16(device: torch.device) -> bool:
    """True only on sm_80+; ``is_bf16_supported()`` also reports emulated support."""
    if device.type != "cuda":
        return False
    return torch.cuda.get_device_capability(device.index or 0)[0] >= 8


@dataclass(frozen=True, slots=True)
class RunSpec:
    """Run identity: ``{model}_o{origin:02d}_K{K:02d}_H{H:03d}_s{seed}``."""

    model: str
    origin_index: int
    k: int
    pred_len: int
    seed: int

    @property
    def run_id(self) -> str:
        return (
            f"{self.model}_o{self.origin_index:02d}_K{self.k:02d}"
            f"_H{self.pred_len:03d}_s{self.seed}"
        )


@dataclass(frozen=True, slots=True)
class TrainOutcome:
    """What one completed run produced, beyond its files."""

    run_id: str
    epochs_run: int
    best_val_mse: float
    train_loss: float
    wall_time_s: float
    n_parameters: int
    device: str


@dataclass(frozen=True, slots=True)
class TrainSchedule:
    """The optimisation budget shared by the two neural models."""

    max_epochs: int = 30
    patience: int = 5
    lr: float = 1e-4
    lr_halve_every: int = 4


class Architecture(Protocol):
    """What the trainer, the runner and the artifact writer need of a config.

    Behaviour lives in methods, not fields, because ``asdict(cfg)`` is written to
    every ``meta/*.json``.
    """

    seq_len: int
    pred_len: int

    def build(self) -> nn.Module:
        """A fresh, untrained module."""

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple[nn.Module, "Architecture", TrainOutcome]:
        """Fit one cell; return the model, the resolved config and the outcome."""


class Forecaster(Protocol):
    """What the artifact writer needs of a fitted model."""

    cfg: Architecture

    def eval(self) -> "Forecaster":
        """Inference mode, dropout off."""

    def forecast_target(self, x: Tensor) -> Tensor:
        """``(B, L, K) -> (B, H)`` on the target channel."""

    def n_parameters(self) -> int:
        """Trainable parameters, or fitted coefficients for a closed-form model."""


def _to_device(split: SplitTensors, device: torch.device) -> tuple[Tensor, Tensor]:
    """Move one split's inputs and target channel to the device."""
    return (
        torch.from_numpy(split.x).to(device, non_blocking=True),
        torch.from_numpy(split.y).to(device, non_blocking=True),
    )


@torch.no_grad()
def _mean_loss(model: nn.Module, x: Tensor, y: Tensor, batch: int = 512) -> float:
    """Mean squared error over a split, batched to bound memory, synchronised once."""
    if len(x) == 0:
        return float("nan")
    model.eval()
    total = torch.zeros((), dtype=torch.float64, device=x.device)
    for i in range(0, len(x), batch):
        total += nn.functional.mse_loss(
            model.forecast_target(x[i : i + batch]), y[i : i + batch], reduction="sum"
        ).double()
    return float(total.item()) / (len(x) * int(np.prod(y.shape[1:])))


@torch.no_grad()
def predict(model: Forecaster, x: Tensor, batch: int = 512) -> np.ndarray:
    """The target channel's H-step forecasts for every window, batched."""
    model.eval()
    if len(x) == 0:
        return np.empty((0, model.cfg.pred_len), np.float32)
    return np.concatenate(
        [
            model.forecast_target(x[i : i + batch]).cpu().numpy()
            for i in range(0, len(x), batch)
        ]
    )


_TRAINING_CONTEXT = threading.local()


class SessionBudgetExhausted(RuntimeError):
    """A recoverable pause at a checkpoint boundary, not a failed run."""


class TrainingSession:
    """Per-worker checkpoint roots and a monotonic deadline."""

    def __init__(self, out_root: Path, roots: list[Path], deadline: float):
        self.state = (Path(out_root), [Path(r) for r in roots], deadline)

    def __enter__(self):
        self.previous = getattr(_TRAINING_CONTEXT, "state", None)
        _TRAINING_CONTEXT.state = self.state
        return self

    def __exit__(self, *exc):
        _TRAINING_CONTEXT.state = self.previous


def train_one(
    tensors: OriginTensors, spec: RunSpec, cfg: Architecture, *,
    device: torch.device | None = None, max_epochs: int | None = None,
    patience: int | None = None, lr: float | None = None,
    lr_halve_every: int | None = None, batch_size: int = 32,
) -> tuple[nn.Module, TrainOutcome]:
    """Train on one device with early stopping on validation MSE, checkpointing every epoch.

    A checkpoint holds the optimiser, scheduler, best weights, epoch, patience
    counter and device RNG, and is reused only when its identity (spec, config,
    code, input, schedule and training sample) matches. An epoch interrupted by
    the session deadline is replayed from its last boundary. Loss is accumulated
    on the device and checked once per epoch, so no step waits on the GPU.
    """
    device = device or pick_device()
    active = getattr(_TRAINING_CONTEXT, "state", None)
    if active is not None and time.perf_counter() >= active[2]:
        raise SessionBudgetExhausted(f"{spec.run_id}: session budget exhausted before fitting")
    protocol = cfg.schedule() if hasattr(cfg, "schedule") else TrainSchedule()
    max_epochs = protocol.max_epochs if max_epochs is None else max_epochs
    patience = protocol.patience if patience is None else patience
    lr = protocol.lr if lr is None else lr
    lr_halve_every = protocol.lr_halve_every if lr_halve_every is None else lr_halve_every
    if min(max_epochs, patience, lr_halve_every, batch_size) < 1 or lr <= 0:
        raise ValueError("positive training schedule and batch size required")
    if not len(tensors.train) or not len(tensors.val):
        raise ValueError("training and validation must be nonempty")
    with SEED_LOCK:
        set_seed(spec.seed, device)
        model = cfg.build().to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    schedule = torch.optim.lr_scheduler.StepLR(optimiser, step_size=lr_halve_every, gamma=.5)
    x_tr, y_tr = _to_device(tensors.train, device)
    x_va, y_va = _to_device(tensors.val, device)
    best_val, best_state, stale = float("inf"), None, 0
    epochs_run, train_loss, prior_seconds = 0, float("nan"), 0.
    started = time.perf_counter()
    context = getattr(_TRAINING_CONTEXT, "state", None)
    checkpoint = None
    identity = None
    deadline = float("inf")
    if context is not None:
        out_root, roots, deadline = context
        checkpoint = out_root / "checkpoints" / f"{spec.run_id}.pt"
        checkpoint.parent.mkdir(parents=True, exist_ok=True)
        identity = json.loads(json.dumps({
            "spec": asdict(spec), "config": asdict(cfg), "code": code_sha256(),
            "input": _input_sha256()[0], "torch": str(torch.__version__),
            "device_type": device.type, "batch_size": batch_size,
            "schedule": [max_epochs, patience, lr, lr_halve_every],
            "train_times": hashlib.sha256(tensors.train.ts.tobytes()).hexdigest(),
        }))
        for root in dict.fromkeys([out_root, *roots]):
            candidate = root / "checkpoints" / f"{spec.run_id}.pt"
            if not candidate.exists():
                continue
            try:
                saved = torch.load(candidate, map_location="cpu", weights_only=True)
            except (OSError, RuntimeError, EOFError) as exc:
                raise ValueError(f"{candidate}: unreadable training checkpoint") from exc
            if saved.get("identity") != identity:
                continue
            model.load_state_dict(saved["model"])
            optimiser.load_state_dict(saved["optimizer"])
            schedule.load_state_dict(saved["scheduler"])
            best_state, best_val = saved["best_state"], saved["best_val"]
            epochs_run, stale = saved["epoch"], saved["stale"]
            train_loss, prior_seconds = saved["train_loss"], saved["wall_time_s"]
            if device.type == "cuda":
                torch.cuda.set_rng_state(saved["rng"], device=device)
            else:
                torch.set_rng_state(saved["rng"])
            break

    def save_boundary():
        if checkpoint is None:
            return
        staging = checkpoint.with_suffix(".pt.tmp")
        torch.save({
            "identity": identity, "model": model.state_dict(),
            "optimizer": optimiser.state_dict(), "scheduler": schedule.state_dict(),
            "best_state": best_state, "best_val": best_val, "epoch": epochs_run,
            "stale": stale, "train_loss": train_loss,
            "wall_time_s": prior_seconds + time.perf_counter() - started,
            "rng": torch.cuda.get_rng_state(device) if device.type == "cuda" else torch.get_rng_state(),
        }, staging)
        staging.replace(checkpoint)

    # Epoch zero is recoverable even if the first epoch hits the deadline.
    if epochs_run == 0:
        save_boundary()
    for epoch in range(epochs_run + 1, max_epochs + 1):
        if stale >= patience:
            break
        if time.perf_counter() >= deadline:
            raise SessionBudgetExhausted(f"{spec.run_id}: resume from epoch {epochs_run}")
        model.train()
        order = torch.randperm(len(x_tr), device=device)
        running = torch.zeros((), dtype=torch.float64, device=device)
        for i in range(0, len(order), batch_size):
            if time.perf_counter() >= deadline:
                raise SessionBudgetExhausted(f"{spec.run_id}: resume from epoch {epochs_run}")
            idx = order[i:i+batch_size]
            optimiser.zero_grad(set_to_none=True)
            loss = nn.functional.mse_loss(model.forecast_target(x_tr[idx]), y_tr[idx])
            loss.backward()
            optimiser.step()
            running += loss.detach().double() * len(idx)
        schedule.step()
        epochs_run = epoch
        train_loss = float(running.item()) / len(x_tr)
        if not np.isfinite(train_loss):
            raise ValueError(f"{spec.run_id}: non-finite training loss in epoch {epoch}")
        val = _mean_loss(model, x_va, y_va)
        if not np.isfinite(val):
            raise ValueError(f"{spec.run_id}: non-finite validation loss")
        if val < best_val - 1e-9:
            best_val, stale = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
        save_boundary()
    if best_state is None:
        raise ValueError(f"{spec.run_id}: no finite validation checkpoint")
    model.load_state_dict(best_state)
    return model, TrainOutcome(
        run_id=spec.run_id, epochs_run=epochs_run, best_val_mse=best_val,
        train_loss=train_loss, wall_time_s=prior_seconds + time.perf_counter()-started,
        n_parameters=model.n_parameters(), device=str(device),
    )


def scale_invariance_check(
    model: Forecaster, x: Tensor, y: Tensor, c: float = 100.0
) -> tuple[float, float]:
    """``(MSE(x), MSE(c x) / c^2)``: equal while instance normalisation is active.

    Scaling the input by ``c`` scales the denormalised forecast by ``c`` as well,
    so the invariant is ``MSE(c x) / c^2 == MSE(x)``, not ``MSE(c x) == MSE(x)``.
    """
    return _mean_loss(model, x, y), _mean_loss(model, x * c, y * c) / (c * c)


def _git_sha() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        return "unknown"


def code_sha256() -> str:
    """Digest of the package source, including the upstream copies.

    Off-repo (Kaggle) there is no git sha, so this names the code that ran.
    Line endings are normalised, so Windows and Linux checkouts agree.
    ``CODE_SHA256_OVERRIDE`` supplies the value where no files exist.
    """
    if CODE_SHA256_OVERRIDE is not None:
        return CODE_SHA256_OVERRIDE
    return tree_digest(Path(__file__).resolve().parent)


def tree_digest(root: Path) -> str:
    """sha256 over every ``*.py`` under ``root``, including the upstream copies.

    Files are taken in order of their relative POSIX path, a string order that is
    the same on Windows and Linux, and line endings are normalised to LF.
    """
    digest = hashlib.sha256()
    for path in sorted(root.rglob("*.py"), key=lambda p: p.relative_to(root).as_posix()):
        digest.update(path.relative_to(root).as_posix().encode("utf-8"))
        digest.update(path.read_bytes().replace(b"\r\n", b"\n"))
    return digest.hexdigest()


def resolve_input_parquet(parquet: Path | str | None = None) -> Path:
    """The input parquet this process consumed: argument, then environment, then default."""
    if parquet is not None:
        return Path(parquet)
    from_env = os.environ.get(INPUT_PARQUET_ENV)
    return Path(from_env) if from_env else DEFAULT_PARQUET


def _input_sha256(parquet: Path | str | None = None) -> tuple[str, str]:
    """Hash the actual input bytes."""
    try:
        return hashlib.sha256(resolve_input_parquet(parquet).read_bytes()).hexdigest(), "file-digest"
    except OSError:
        return "unknown", "unresolved"


def environment(device: torch.device | None = None) -> dict:
    """Library versions, GPU and upstream commit, recorded with every run."""
    versions = {"python": platform.python_version(), "torch": str(torch.__version__),
                "numpy": np.__version__, "polars": pl.__version__}
    try:
        import sklearn
        versions["sklearn"] = sklearn.__version__
    except ImportError:
        versions["sklearn"] = None
    cuda = device is not None and torch.device(device).type == "cuda"
    return {
        **versions,
        "cuda": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
        "gpu": torch.cuda.get_device_name(device) if cuda else None,
        "upstream": f"{UPSTREAM_REPO}@{UPSTREAM_COMMIT}",
    }


def write_artifacts(
    model: Forecaster,
    tensors: OriginTensors,
    spec: RunSpec,
    cfg: Architecture,
    outcome: TrainOutcome,
    device: torch.device,
    root: Path = ARTIFACTS,
    requested_config: Architecture | None = None,
) -> tuple[Path, Path]:
    """Publish predictions and weights atomically, then the metadata that marks the run complete.

    ``meta`` is written last and carries the sha256 of both files, so a run is
    complete only when all three agree. The run's checkpoint is removed after.
    """
    (root / "preds").mkdir(parents=True, exist_ok=True)
    (root / "meta").mkdir(parents=True, exist_ok=True)

    frames = []
    for b, split in zip(tensors.block_labels, tensors.test_blocks):
        if len(split) == 0:
            continue
        x, _ = _to_device(split, device)
        pred = predict(model, x)
        n, h = pred.shape
        frames.append(
            pl.DataFrame(
                {
                    "block": np.full(n * h, b, dtype=np.int8),
                    "step": np.tile(np.arange(1, h + 1, dtype=np.int16), n),
                    "timestamp": np.repeat(split.ts, h),
                    "forecast_origin": np.repeat(split.ts, h),
                    "input_start": np.repeat(split.ts - cfg.seq_len * 3_600_000, h),
                    "target_timestamp": (split.ts[:, None] + np.arange(h) * 3_600_000).reshape(-1),
                    "y_true": split.y.reshape(-1),
                    "y_pred": pred.reshape(-1),
                }
            )
        )
    preds = (
        pl.concat(frames)
        if frames
        else pl.DataFrame(
            schema={
                "block": pl.Int8,
                "step": pl.Int16,
                "timestamp": pl.Int64,
                "forecast_origin": pl.Int64,
                "input_start": pl.Int64,
                "target_timestamp": pl.Int64,
                "y_true": pl.Float32,
                "y_pred": pl.Float32,
            }
        )
    )

    preds_path = root / "preds" / f"{spec.run_id}.parquet"
    meta_path = root / "meta" / f"{spec.run_id}.json"
    staging_preds = preds_path.with_suffix(".parquet.tmp")
    preds.write_parquet(staging_preds)
    staging_preds.replace(preds_path)

    weights_path = root / "weights" / f"{spec.run_id}.pt"
    weights_path.parent.mkdir(parents=True, exist_ok=True)
    staging_weights = weights_path.with_suffix(".pt.tmp")
    torch.save(model.state_dict(), staging_weights)
    staging_weights.replace(weights_path)
    input_parquet = resolve_input_parquet()
    input_digest, input_provenance = _input_sha256(input_parquet)
    schedule = cfg.schedule() if hasattr(cfg, "schedule") else None
    meta = {
        "run_id": spec.run_id,
        "prediction_schema_version": 2,
        "weights_sha256": hashlib.sha256(weights_path.read_bytes()).hexdigest(),
        "torch_version": str(torch.__version__),
        "predictions_sha256": hashlib.sha256(preds_path.read_bytes()).hexdigest(),
        "timestamp_semantics": "forecast_origin",
        "forecast_origin_definition": "first target bar open, UTC",
        "evaluation_population": "surviving contiguous windows",
        "selection_time_ms": int(tensors.origin.test_start.timestamp() * 1000),
        "training_cutoff_ms": int(tensors.origin.train_sub_end.timestamp() * 1000),
        "latest_training_target_ms": int(tensors.train.ts.max() + (cfg.pred_len-1)*3600000),
        "spec": asdict(spec),
        "config": asdict(cfg),
        "requested_config": asdict(requested_config or cfg),
        "schedule": asdict(schedule) if schedule is not None else None,
        "origin": tensors.origin.label,
        "origin_index": tensors.origin.index,
        "block_labels": list(tensors.block_labels),
        "k": tensors.k,
        "variates": list(tensors.scaler.columns),
        "git_sha": _git_sha(),
        "code_sha256": code_sha256(),
        "input_parquet": str(input_parquet),
        "input_sha256": input_digest,
        "input_sha256_source": input_provenance,
        "environment": environment(device),
        "n_train": len(tensors.train),
        "training_selection": tensors.training_selection,
        "n_val": len(tensors.val),
        "n_test_per_block": [len(s) for s in tensors.test_blocks],
        "mu_g": float(tensors.scaler.mean[0]),
        "sigma_g": float(tensors.scaler.std[0]),
        "mu_over_sigma": tensors.scaler.target_mu_over_sigma,
        "naive_rw_z": tensors.naive_rw_z,
        "epochs_run": outcome.epochs_run,
        "best_val_mse": outcome.best_val_mse,
        "train_loss": outcome.train_loss,
        "wall_time_s": outcome.wall_time_s,
        "n_parameters": outcome.n_parameters,
        "n_allocated_parameters": sum(p.numel() for p in model.parameters()) if isinstance(model, nn.Module) else outcome.n_parameters,
        "loss_target": "target",
        "reached_epoch_cap": bool(schedule is not None and outcome.epochs_run >= schedule.max_epochs),
        "device": outcome.device,
        "status": "complete",
    }
    staging_meta = meta_path.with_suffix(".json.tmp")
    staging_meta.write_text(json.dumps(meta, indent=2), encoding="utf-8")
    staging_meta.replace(meta_path)
    (root / "checkpoints" / f"{spec.run_id}.pt").unlink(missing_ok=True)
    return preds_path, meta_path


def is_complete(
    run_id: str, root: Path = ARTIFACTS, *, strict: bool = False,
    cfg: Architecture | None = None, columns: tuple[str, ...] | None = None,
) -> bool:
    """Whether ``run_id`` is complete under ``root``.

    Loose: both files exist and ``meta`` says complete. Strict (resume): the
    input digest, code digest, requested config, schedule, variates, prediction
    schema and both file hashes must also match the current run request.
    """
    preds = root / "preds" / f"{run_id}.parquet"
    meta_path = root / "meta" / f"{run_id}.json"
    if not (preds.is_file() and meta_path.is_file()):
        return False
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("status") != "complete":
            return False
        if not strict:
            return True
        digest, _ = _input_sha256()
        if digest == "unknown" or meta.get("input_sha256") != digest:
            return False
        if meta.get("code_sha256") != code_sha256() or meta.get("run_id") != run_id:
            return False
        if cfg is None or meta.get("requested_config") != json.loads(json.dumps(asdict(cfg))):
            return False
        schedule = asdict(cfg.schedule()) if hasattr(cfg, "schedule") else None
        if meta.get("schedule") != json.loads(json.dumps(schedule)):
            return False
        if columns is not None and meta.get("variates") != list(columns):
            return False
        required = {"block", "step", "timestamp", "forecast_origin", "input_start",
                    "target_timestamp", "y_true", "y_pred"}
        if meta.get("prediction_schema_version") != 2 or not required <= set(pl.read_parquet_schema(preds)):
            return False
        weights = root / "weights" / f"{run_id}.pt"
        if not weights.is_file() or meta.get("weights_sha256") != hashlib.sha256(weights.read_bytes()).hexdigest():
            return False
        if meta.get("predictions_sha256") != hashlib.sha256(preds.read_bytes()).hexdigest():
            return False
        return True
    except (OSError, ValueError, TypeError, pl.exceptions.PolarsError):
        return False


###

<div style="background: linear-gradient(90deg, #0a1a12, #0f2a1c); border-left: 3px solid #52b788; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #95d5b2; font-size: 1em; margin: 0;">🧷 <code>baselines.py</code></h3> <p style="display: inline; color: #b7e4c7; font-size: 0.9em; margin: 0;">· Ridge dari scikit-learn; α dipilih pada validation.</p></div>

In [27]:
"""Ridge regression, the linear comparator on the same information.

Ridge maps the flattened ``K x 96`` lookback to the 24-step target with an L2
penalty, and C1 compares the iTransformer against it at every rung. Alpha is the
only hyperparameter selected anywhere in the study, on validation MSE. The fit
is closed-form and draws no random numbers, so the five seeds of a cell give
identical predictions.

Upstream:
    ``sklearn.linear_model.Ridge`` (https://github.com/scikit-learn/scikit-learn,
    BSD-3-Clause); F. Pedregosa et al., "Scikit-learn: Machine learning in
    Python," JMLR 12, 2011; A. E. Hoerl and R. W. Kennard, "Ridge regression:
    Biased estimation for nonorthogonal problems," Technometrics 12(1), 1970.
"""

#: Alpha candidates, chosen on validation. The penalty is unnormalised, so the
#: scale that matters is ``diag(X'X)``, of the order of the training sample size.
RIDGE_ALPHAS: tuple[float, ...] = (1e-1, 1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6)


@dataclass(frozen=True, slots=True)
class RidgeConfig:
    """L2-regularised linear map from the flattened window to the H-step target.

    ``k`` is a field because the weight matrix is ``(L*K, H)``. ``alpha`` is set by
    :meth:`fit`, and the fitted value is what enters ``meta['config']``.
    """

    seq_len: int = SEQ_LEN
    pred_len: int = PRED_LEN
    k: int = 8
    alphas: tuple[float, ...] = RIDGE_ALPHAS
    solver: str = "cholesky"
    alpha: float | None = None

    def build(self) -> "RidgeForecaster":
        return RidgeForecaster(self)

    def fit(
        self,
        tensors: OriginTensors,
        spec: RunSpec,
        *,
        device: torch.device | None = None,
    ) -> tuple["RidgeForecaster", "RidgeConfig", TrainOutcome]:
        """Fit one ``Ridge`` per alpha on the training windows, keep the best on validation.

        Inputs are cast to float64 before fitting. The intercept is fitted and not
        penalised, so the forecast is not shrunk toward the training drift.
        """
        from sklearn.linear_model import Ridge

        device = device or pick_device()
        started = time.perf_counter()
        with SEED_LOCK:
            set_seed(spec.seed, device)
            model = self.build().to(device)
        x_tr = tensors.train.x.reshape(len(tensors.train), -1).astype(np.float64)
        y_tr = tensors.train.y.astype(np.float64)
        x_va = tensors.val.x.reshape(len(tensors.val), -1).astype(np.float64)
        y_va = tensors.val.y.astype(np.float64)

        best = None
        for alpha in self.alphas:
            fitted = Ridge(alpha=alpha, fit_intercept=True, solver=self.solver).fit(x_tr, y_tr)
            val_mse = float(np.mean((fitted.predict(x_va) - y_va) ** 2))
            if best is None or val_mse < best[0]:
                best = (val_mse, float(alpha), fitted)
        if best is None:
            raise ValueError("no ridge alpha to select; `alphas` is empty")

        val_mse, alpha, fitted = best
        if len(self.alphas) > 1 and alpha in (self.alphas[0], self.alphas[-1]):
            warnings.warn(
                f"{spec.run_id}: ridge alpha {alpha:g} sits at the edge of "
                f"{self.alphas}; the grid may not bracket the optimum",
                stacklevel=2,
            )
        with torch.no_grad():
            model.weight.copy_(torch.from_numpy(fitted.coef_.T.astype(np.float32)))
            model.bias.copy_(torch.from_numpy(np.asarray(fitted.intercept_, dtype=np.float32)))
        train_mse = float(np.mean((fitted.predict(x_tr) - y_tr) ** 2))
        return (
            model,
            replace(self, alpha=alpha),
            TrainOutcome(
                run_id=spec.run_id,
                epochs_run=0,
                best_val_mse=val_mse,
                train_loss=train_mse,
                wall_time_s=time.perf_counter() - started,
                n_parameters=model.n_parameters(),
                device=str(device),
            ),
        )


class RidgeForecaster(nn.Module):
    """``y_hat = vec(x) @ W + b`` with the coefficients fitted by scikit-learn.

    ``W`` and ``b`` are buffers, not parameters, so no optimiser ever sees them.
    """

    def __init__(self, cfg: RidgeConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.register_buffer(
            "weight",
            torch.zeros(cfg.seq_len * cfg.k, cfg.pred_len, dtype=torch.float32),
        )
        self.register_buffer("bias", torch.zeros(cfg.pred_len, dtype=torch.float32))

    def forward(self, x: Tensor) -> Tensor:
        """``(B, L, K) -> (B, H)``."""
        return x.reshape(len(x), -1) @ self.weight + self.bias

    def forecast_target(self, x: Tensor) -> Tensor:
        return self(x)

    def n_parameters(self) -> int:
        return self.weight.numel() + self.bias.numel()


def assert_baseline_alignment(
    baseline_run_id: str, reference_run_id: str, roots: list[Path]
) -> None:
    """Refuse a comparator scored on windows other than its reference's.

    Test-window survival depends on future gaps, which cluster in stress, so a
    ratio over two different window sets would be biased, not noisy.

    Raises:
        ValueError: If the evaluated ``(block, timestamp)`` sets differ.
        FileNotFoundError: If either run is absent from ``roots``.
    """
    assert_same_windows(
        load_predictions(baseline_run_id, roots),
        load_predictions(reference_run_id, roots),
        f"{baseline_run_id} vs {reference_run_id}",
    )


##

<a id="section-06"></a>

<div style="background: linear-gradient(135deg, #012a4a, #013a63); border-left: 4px solid #48cae4; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #90e0ef; margin: 0 0 4px; font-size: 1.35em;">⚙️ 06 · Persiapan evaluasi dan eksekutor</h2>
  <p style="color: #caf0f8; margin: 0; font-size: 0.95em;">Metrik, perbandingan antarmodel, manifes 900 run, eksekutor dua GPU, dan laporan; digest kode dicatat sebelum training.</p>
</div>

###

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e94560; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #f5a623; font-size: 1em; margin: 0;">🎯 <code>metrics.py</code></h3> <p style="display: inline; color: #ffd6a5; font-size: 0.9em; margin: 0;">· Metrik, akurasi arah, DM/HLN/Clark–West, β₁ dan bootstrap klaster.</p></div>

In [28]:
"""Metrics, tests and the RQ estimators, computed from saved predictions only.

Everything here reads ``preds/{run_id}.parquet`` and ``meta/{run_id}.json``; no
model or tensor is touched, so every number can be regenerated from the files.

- Naive-RW is ``y_z = -mu_g / sigma_g`` in scaler space (a zero raw return).
- Ratios are formed from seed-averaged MSEs, never averaged across seeds.
- Blocks are assigned by forecast origin, the first target hour.
- Diebold-Mariano uses the Harvey-Leybourne-Newbold correction and a
  rectangular long-run variance at lag ``h - 1``; Clark-West is used only
  against Naive-RW.
- All inference is diagnostic: origins overlap in their training data.

Upstream:
    Written here on numpy after F. X. Diebold and R. S. Mariano, J. Bus. Econ.
    Statist. 13(3), 1995; D. Harvey, S. Leybourne and P. Newbold, Int. J.
    Forecast. 13(2), 1997; T. E. Clark and K. D. West, J. Econometrics 138(1),
    2007; M. H. Pesaran and A. Timmermann, J. Bus. Econ. Statist. 10(4), 1992;
    A. C. Cameron, J. B. Gelbach and D. L. Miller, Rev. Econ. Statist. 90(3),
    2008; J. G. MacKinnon, M. O. Nielsen and M. D. Webb, J. Econometrics 232(2),
    2023.
"""

RUN_ID_PATTERN = re.compile(
    r"^(?P<model>[a-z0-9]+)_o(?P<origin>\d{2})_K(?P<k>\d{2})"
    r"_H(?P<h>\d{3})_s(?P<seed>\d+)$"
)

HOUR_MS = 3_600_000


# -- artifact I/O ------------------------------------------------------------


def parse_run_id(run_id: str) -> dict[str, int | str]:
    """Split a ``run_id`` into model, origin index, K, horizon and seed."""
    match = RUN_ID_PATTERN.match(run_id)
    if match is None:
        raise ValueError(f"{run_id!r} is not a {{model}}_o{{origin}}_K{{K}}_H{{H}}_s{{seed}} run_id")
    g = match.groupdict()
    return {
        "model": g["model"],
        "origin_index": int(g["origin"]),
        "k": int(g["k"]),
        "pred_len": int(g["h"]),
        "seed": int(g["seed"]),
    }


def _locate(run_id: str, roots: list[Path], kind: str, suffix: str) -> Path:
    for root in roots:
        candidate = Path(root) / kind / f"{run_id}{suffix}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"{kind}/{run_id}{suffix} in none of {[str(r) for r in roots]}"
    )


def load_predictions(run_id: str, roots: list[Path]) -> pl.DataFrame:
    """Read one run's predictions, with UTC target times and blocks by forecast origin."""
    path = _locate(run_id, roots, "preds", ".parquet")
    meta_path = path.parent.parent / "meta" / f"{run_id}.json"
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    frame = pl.read_parquet(path)
    length = int(meta["config"]["seq_len"])
    horizon = int(meta["spec"]["pred_len"])
    if length < 1 or horizon < 1:
        raise ValueError(f"{run_id}: invalid lookback or horizon")
    if meta.get("timestamp_semantics") != "forecast_origin":
        raise ValueError(f"{run_id}: unknown timestamp semantics")
    if not {"input_start", "forecast_origin", "target_timestamp"} <= set(frame.columns):
        raise ValueError(f"{run_id}: incomplete timestamp schema")
    if frame.filter(
        (pl.col("timestamp") != pl.col("forecast_origin")) |
        (pl.col("forecast_origin") - pl.col("input_start") != length * HOUR_MS) |
        (pl.col("target_timestamp") != pl.col("timestamp") +
         (pl.col("step").cast(pl.Int64) - 1) * HOUR_MS)
    ).height:
        raise ValueError(f"{run_id}: inconsistent target timestamps")
    required = ["block", "timestamp", "step", "forecast_origin", "input_start", "target_timestamp", "y_true", "y_pred"]
    if frame.is_empty() or any(frame[n].null_count() for n in required):
        raise ValueError(f"{run_id}: empty predictions or null prediction keys")
    counts = frame.group_by("timestamp").agg(
        pl.len().alias("n"), pl.col("step").n_unique().alias("unique"),
        pl.col("step").min().alias("first"), pl.col("step").max().alias("last"),
    )
    if counts.filter((pl.col("n") != horizon) | (pl.col("unique") != horizon) |
                     (pl.col("first") != 1) | (pl.col("last") != horizon)).height:
        raise ValueError(f"{run_id}: incomplete or duplicated forecast horizon")
    if frame.select(pl.any_horizontal(pl.col("y_true", "y_pred").is_null() |
                                     ~pl.col("y_true", "y_pred").is_finite()).any()).item():
        raise ValueError(f"{run_id}: non-finite predictions")
    return frame.sort(["block", "timestamp", "step"])


def load_meta(run_id: str, roots: list[Path]) -> dict:
    """Read metadata paired with the selected prediction file, never another root."""
    path = _locate(run_id, roots, "preds", ".parquet").parent.parent / "meta" / f"{run_id}.json"
    return json.loads(path.read_text(encoding="utf-8"))


# -- point metrics -----------------------------------------------------------


def mse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.square(np.asarray(y_true) - np.asarray(y_pred))))


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def rel_mse(model: float, naive: float) -> float:
    """``MSE_model / MSE_naive`` on the same windows, which controls for period difficulty."""
    return model / naive


def r2_oos(model: float, naive: float) -> float:
    """``1 - RelMSE``: positive means the model beats Naive-RW."""
    return 1.0 - rel_mse(model, naive)


def raw_rmse(mse_z: float, sigma_g: float) -> float:
    """RMSE back in raw log-return units: ``sqrt(mse_z) * sigma_g``."""
    return math.sqrt(mse_z) * sigma_g


def non_overlapping_mask(timestamps: np.ndarray) -> np.ndarray:
    """Issuances whose forecast period opens at 00:00 UTC: one per day, none overlapping."""
    return (np.asarray(timestamps) // HOUR_MS) % 24 == 0


def pesaran_timmermann(actual: np.ndarray, predicted: np.ndarray) -> tuple[float, float]:
    """Pesaran-Timmermann (1992) test of directional predictability.

    Returns:
        ``(statistic, one-sided p)`` against ``N(0,1)``. Without a null
        hypothesis, directional accuracy is a descriptive number; this supplies
        the null.

    Zero targets are excluded rather than assigned a direction: a zero
    log-return has no sign to predict, and assigning one would inflate the hit
    rate by whatever the model happened to output there.
    """
    a = np.sign(np.asarray(actual, dtype=np.float64))
    f = np.sign(np.asarray(predicted, dtype=np.float64))
    keep = a != 0
    a, f = a[keep], f[keep]
    n = len(a)
    if n < 2:
        return float("nan"), float("nan")

    hit = float(np.mean(a == f))
    py = float(np.mean(a > 0))
    px = float(np.mean(f > 0))
    p_star = py * px + (1 - py) * (1 - px)

    var_hit = p_star * (1 - p_star) / n
    var_star = (
        (2 * py - 1) ** 2 * px * (1 - px)
        + (2 * px - 1) ** 2 * py * (1 - py)
        + 4 * py * px * (1 - py) * (1 - px) / n
    ) / n
    denom = var_hit - var_star
    if denom <= 0:
        return float("nan"), float("nan")

    stat = (hit - p_star) / math.sqrt(denom)
    return stat, 0.5 * math.erfc(stat / math.sqrt(2.0))


def _hit_rate(actual: np.ndarray, predicted: np.ndarray) -> float:
    keep = np.sign(actual) != 0
    if not keep.any():
        return float("nan")
    return float(np.mean(np.sign(actual[keep]) == np.sign(predicted[keep])))


# -- per-block tables --------------------------------------------------------


@dataclass(frozen=True, slots=True)
class DirectionalAccuracy:
    """Hit rates of the three directional variants on raw returns.

    ``up_*`` is the share of positive actual moves, from which the per-origin
    majority baseline ``max(up, 1 - up)`` is formed. ``p_*`` are Pesaran-Timmermann
    p-values, reported as a diagnostic only.
    """

    da_1h: float
    da_24h: float
    da_cum: float
    up_1h: float
    up_24h: float
    up_cum: float
    p_1h: float
    p_24h: float
    p_cum: float
    n_1h: int
    n_daily: int


def _up_share(actual: np.ndarray) -> float:
    moved = np.sign(actual) != 0
    return float(np.mean(actual[moved] > 0)) if moved.any() else float("nan")


def directional_accuracy(frame: pl.DataFrame, *, sigma_g: float, mu_g: float) -> DirectionalAccuracy:
    """DA-1h, DA-24h and DA-cum of one run, on raw returns ``r = z sigma_g + mu_g``.

    DA-1h scores the first step at every issuance hour. DA-24h scores the last
    step and DA-cum the sign of the summed 24-hour return, both at one issuance a
    day (00:00 UTC), so no two scored windows overlap. A zero actual return is
    excluded and a zero forecast counts as wrong.
    """
    if not np.isfinite(sigma_g) or sigma_g <= 0 or not np.isfinite(mu_g):
        raise ValueError("directional accuracy requires a finite mean and positive scale")
    frame = frame.with_columns(
        (pl.col("y_true") * sigma_g + mu_g).alias("y_true"),
        (pl.col("y_pred") * sigma_g + mu_g).alias("y_pred"),
    )
    last_step = int(frame.get_column("step").max())
    step1 = frame.filter(pl.col("step") == 1)
    a1, f1 = step1.get_column("y_true").to_numpy(), step1.get_column("y_pred").to_numpy()
    last = frame.filter(pl.col("step") == last_step)
    keep_h = non_overlapping_mask(last.get_column("timestamp").to_numpy())
    a_h = last.get_column("y_true").to_numpy()[keep_h]
    f_h = last.get_column("y_pred").to_numpy()[keep_h]
    cum = frame.group_by("timestamp").agg(pl.col("y_true").sum(), pl.col("y_pred").sum()).sort("timestamp")
    keep_c = non_overlapping_mask(cum.get_column("timestamp").to_numpy())
    a_c = cum.get_column("y_true").to_numpy()[keep_c]
    f_c = cum.get_column("y_pred").to_numpy()[keep_c]
    return DirectionalAccuracy(
        da_1h=_hit_rate(a1, f1), da_24h=_hit_rate(a_h, f_h), da_cum=_hit_rate(a_c, f_c),
        up_1h=_up_share(a1), up_24h=_up_share(a_h), up_cum=_up_share(a_c),
        p_1h=pesaran_timmermann(a1, f1)[1], p_24h=pesaran_timmermann(a_h, f_h)[1],
        p_cum=pesaran_timmermann(a_c, f_c)[1], n_1h=len(a1), n_daily=int(keep_c.sum()),
    )


DA_VARIANTS: tuple[str, ...] = ("1h", "24h", "cum")


def directional_accuracy_table(run_ids: list[str], roots: list[Path], *,
                               windows: pl.DataFrame | None = None) -> pl.DataFrame:
    """One row per run: the three hit rates, the majority baselines and Pesaran-Timmermann p.

    With ``windows`` (from :func:`evaluation_windows`) every run is scored on the
    common calendar, so the baseline is the same for every model and K at an origin.
    """
    rows = []
    for run_id in run_ids:
        parts = parse_run_id(run_id)
        meta = load_meta(run_id, roots)
        frame = load_predictions(run_id, roots)
        if windows is not None:
            keep = windows.filter((pl.col("origin_index") == parts["origin_index"]) &
                                  (pl.col("pred_len") == parts["pred_len"]))
            frame = frame.join(keep.select("block", "timestamp"), on=["block", "timestamp"], how="semi")
        da = directional_accuracy(frame, sigma_g=float(meta["sigma_g"]), mu_g=float(meta["mu_g"]))
        row = {"run_id": run_id, "model": str(parts["model"]), "origin": str(meta["origin"]),
               "origin_index": int(parts["origin_index"]), "k": int(parts["k"]),
               "seed": int(parts["seed"]), "n_1h": da.n_1h, "n_daily": da.n_daily}
        for v in DA_VARIANTS:
            up = getattr(da, f"up_{v}")
            row[f"da_{v}"] = getattr(da, f"da_{v}")
            row[f"base_{v}"] = max(up, 1.0 - up)
            row[f"p_{v}"] = getattr(da, f"p_{v}")
        rows.append(row)
    return pl.DataFrame(rows)


def directional_accuracy_summary(table: pl.DataFrame) -> pl.DataFrame:
    """Per (model, K): mean and SE across origins of seed-averaged ``DA - baseline``.

    Also the number of origins where DA beats the baseline and, as a diagnostic,
    the number of runs with Pesaran-Timmermann p < 0.05.
    """
    per_origin = table.group_by("model", "k", "origin_index").agg(
        *[(pl.col(f"da_{v}") - pl.col(f"base_{v}")).mean().alias(f"dda_{v}") for v in DA_VARIANTS],
        *[(pl.col(f"p_{v}") < 0.05).sum().alias(f"pt_{v}") for v in DA_VARIANTS],
        pl.len().alias("runs"),
    )
    return per_origin.group_by("model", "k").agg(
        pl.col("origin_index").n_unique().alias("n_origins"),
        pl.col("runs").sum(),
        *[pl.col(f"dda_{v}").mean().alias(f"dda_{v}") for v in DA_VARIANTS],
        *[(pl.col(f"dda_{v}").std(ddof=1) / pl.col(f"dda_{v}").count().sqrt()).alias(f"dda_{v}_se")
          for v in DA_VARIANTS],
        *[(pl.col(f"dda_{v}") > 0).sum().alias(f"wins_{v}") for v in DA_VARIANTS],
        *[pl.col(f"pt_{v}").sum().alias(f"pt_{v}") for v in DA_VARIANTS],
    ).sort("model", "k")


def assert_same_windows(left: pl.DataFrame, right: pl.DataFrame, what: str) -> None:
    """Raise unless two runs share every forecast origin, step and actual target time."""
    columns = ["block", "timestamp", "step"]
    if "target_timestamp" in left.columns or "target_timestamp" in right.columns:
        if not all("target_timestamp" in f.columns for f in (left, right)):
            raise ValueError(f"{what}: target timestamp contract missing on one side")
        columns.append("target_timestamp")
    a, b = [f.select(columns).sort(columns) for f in (left, right)]
    if a.is_duplicated().any() or b.is_duplicated().any() or not a.equals(b):
        raise ValueError(f"{what}: evaluated window sets differ ({a.height} vs {b.height} points)")


def block_metrics(frame: pl.DataFrame, naive_z: float) -> pl.DataFrame:
    """Per-block MSE, MAE, RelMSE and ``R2_oos`` for one run."""
    return (
        frame.with_columns(
            (pl.col("y_true") - pl.col("y_pred")).pow(2).alias("_se"),
            (pl.col("y_true") - pl.col("y_pred")).abs().alias("_ae"),
            (pl.col("y_true") - naive_z).pow(2).alias("_se_naive"),
        )
        .group_by("block")
        .agg(
            pl.col("timestamp").n_unique().alias("n_windows"),
            pl.col("_se").count().alias("n_points"),
            pl.col("_se").mean().alias("mse"),
            pl.col("_ae").mean().alias("mae"),
            pl.col("_se_naive").mean().alias("mse_naive"),
        )
        .with_columns(
            (pl.col("mse") / pl.col("mse_naive")).alias("rel_mse"),
            (1.0 - pl.col("mse") / pl.col("mse_naive")).alias("r2_oos"),
        )
        .sort("block")
    )


def evaluation_windows(run_ids: list[str], roots: list[Path]) -> pl.DataFrame:
    """Common forecast times per origin, horizon and block across supplied runs.

    The intersection does not recover forecasts absent from an arm or outcomes
    absent from the data.
    """
    common = {}
    vintages = set()
    for run_id in sorted(run_ids):
        parts = parse_run_id(run_id)
        meta = load_meta(run_id, roots)
        vintage = (meta.get("input_sha256"), meta.get("code_sha256"))
        if any(not v or v == "unknown" for v in vintage):
            raise ValueError(f"{run_id}: missing analysis provenance")
        vintages.add(vintage)
        frame = load_predictions(run_id, roots)
        labels = meta.get("block_labels", [1, 2, 3, 4, 5, 6])
        for b in labels:
            key = (parts["origin_index"], parts["pred_len"], int(b))
            stamps = set(frame.filter(pl.col("block") == b)["timestamp"].unique().to_list())
            common[key] = common[key] & stamps if key in common else stamps
    if len(vintages) != 1:
        raise ValueError("analysis mixes code or input vintages")
    if any(not stamps for stamps in common.values()):
        raise ValueError("a required origin/horizon/block has no common forecast times")
    return pl.DataFrame([
        {"origin_index": i, "pred_len": h, "block": b, "timestamp": t}
        for (i, h, b), stamps in sorted(common.items()) for t in sorted(stamps)
    ])


def gather_grid(run_ids: list[str], roots: list[Path], *,
                windows: pl.DataFrame | None = None) -> pl.DataFrame:
    """Mean step errors on identical actual targets; average seeds afterwards.

    Common times and per-block hashes make downstream sample equality checkable.
    Forecast files and their original metadata are never modified.
    """
    import hashlib
    if windows is None:
        windows = evaluation_windows(run_ids, roots)
    rows = []
    raw_targets = {}
    for run_id in sorted(run_ids):
        parts = parse_run_id(run_id)
        meta = load_meta(run_id, roots)
        keep = windows.filter((pl.col("origin_index") == parts["origin_index"]) &
                              (pl.col("pred_len") == parts["pred_len"]))
        frame = load_predictions(run_id, roots).join(
            keep.select("block", "timestamp"), on=["block", "timestamp"], how="semi"
        ).sort(["block", "timestamp", "step"])
        hashes = []
        for (b,), block_frame in frame.group_by("block", maintain_order=True):
            key = (parts["origin_index"], parts["pred_len"], b)
            values = block_frame["y_true"].to_numpy().astype(np.float64) * float(meta["sigma_g"]) + float(meta["mu_g"])
            if key in raw_targets and not np.allclose(values, raw_targets[key], rtol=1e-5, atol=1e-8):
                raise ValueError(f"{run_id}: actual raw targets disagree")
            raw_targets[key] = values
            keys = block_frame.select("timestamp", "step", "target_timestamp").to_numpy().astype("<i8")
            hashes.append({"block": int(b), "evaluation_keys_sha256": hashlib.sha256(keys.tobytes()).hexdigest()})
        rows.append(block_metrics(frame, float(meta["naive_rw_z"])).join(
            pl.DataFrame(hashes), on="block"
        ).with_columns(
            pl.lit(run_id).alias("run_id"), pl.lit(str(parts["model"])).alias("model"),
            pl.lit(int(parts["origin_index"])).cast(pl.Int32).alias("origin_index"),
            pl.lit(str(meta["origin"])).alias("origin"),
            pl.lit(int(parts["k"])).cast(pl.Int32).alias("k"),
            pl.lit(int(parts["pred_len"])).cast(pl.Int32).alias("pred_len"),
            pl.lit(int(parts["seed"])).cast(pl.Int32).alias("seed"),
            pl.lit(float(meta["sigma_g"])).alias("sigma_g"),
        ))
    return pl.concat(rows)


def seed_average(grid: pl.DataFrame) -> pl.DataFrame:
    """Average MSE across seeds before any ratio is formed."""
    identity = ["model", "origin_index", "origin", "k", "pred_len", "block"]
    extra = []
    if "evaluation_keys_sha256" in grid.columns:
        if grid.group_by(identity).agg(pl.col("evaluation_keys_sha256").n_unique().alias("n")).filter(pl.col("n") != 1).height:
            raise ValueError("seeds were evaluated on different target calendars")
        extra = [pl.col("evaluation_keys_sha256").first()]
    return (
        grid.group_by(identity)
        .agg(
            pl.col("mse").mean().alias("mse"),
            pl.col("mae").mean().alias("mae"),
            pl.col("mse_naive").mean().alias("mse_naive"),
            pl.col("mse").std().alias("mse_seed_std"),
            pl.col("n_windows").first().alias("n_windows"),
            pl.col("sigma_g").first().alias("sigma_g"),
            pl.col("mse").count().alias("n_seeds"),
            *extra,
        )
        .with_columns(
            (pl.col("mse") / pl.col("mse_naive")).alias("rel_mse"),
            (1.0 - pl.col("mse") / pl.col("mse_naive")).alias("r2_oos"),
        )
        .sort(["model", "origin_index", "k", "pred_len", "block"])
    )


# -- RQ2: the K=1 against K=8 gap -------------------------------------------


def amplification(
    seed_avg: pl.DataFrame,
    k_small: int = 1,
    k_large: int = 8,
    model: str = "itr",
    pred_len: int = 24,
) -> pl.DataFrame:
    """``A(i,b) = (MSE_K1 - MSE_K8) / MSE_K1`` per origin and block: RQ2's outcome."""
    base = seed_avg.filter(
        (pl.col("model") == model) & (pl.col("pred_len") == pred_len)
    )
    small = (
        base.filter(pl.col("k") == k_small)
        .select(["origin_index", "origin", "block", "mse", "n_windows"])
        .rename({"mse": "mse_small", "n_windows": "n_small"})
    )
    large = (
        base.filter(pl.col("k") == k_large)
        .select(["origin_index", "block", "mse", "n_windows"])
        .rename({"mse": "mse_large", "n_windows": "n_large"})
    )

    joined = small.join(large, on=["origin_index", "block"], how="inner")
    if joined.height != small.height:
        raise ValueError(
            f"K={k_small} has {small.height} cells but only {joined.height} "
            f"matched K={k_large}; the panel must be balanced before beta1"
        )
    mismatched = joined.filter(pl.col("n_small") != pl.col("n_large"))
    if mismatched.height:
        raise ValueError(
            f"{mismatched.height} cells evaluate K={k_small} and "
            f"K={k_large} on different window counts; A would be a ratio across "
            f"two samples"
        )
    return joined.with_columns(
        ((pl.col("mse_small") - pl.col("mse_large")) / pl.col("mse_small")).alias("A")
    ).sort(["origin_index", "block"])


# -- forecast-loss tests: Diebold-Mariano, HLN, Clark-West --------------------


def _normal_quantile(p: float) -> float:
    """Acklam's inverse normal CDF — avoids a scipy import for one number."""
    a = [-3.969683028665376e+01, 2.209460984245205e+02, -2.759285104469687e+02,
         1.383577518672690e+02, -3.066479806614716e+01, 2.506628277459239e+00]
    b = [-5.447609879822406e+01, 1.615858368580409e+02, -1.556989798598866e+02,
         6.680131188771972e+01, -1.328068155288572e+01]
    c = [-7.784894002430293e-03, -3.223964580411365e-01, -2.400758277161838e+00,
         -2.549732539343734e+00, 4.374664141464968e+00, 2.938163982698783e+00]
    d = [7.784695709041462e-03, 3.224671290700398e-01, 2.445134137142996e+00,
         3.754408661907416e+00]
    plow, phigh = 0.02425, 1 - 0.02425
    if p < plow:
        q = math.sqrt(-2 * math.log(p))
        return (((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) / \
               ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
    if p > phigh:
        q = math.sqrt(-2 * math.log(1 - p))
        return -(((((c[0]*q+c[1])*q+c[2])*q+c[3])*q+c[4])*q+c[5]) / \
                ((((d[0]*q+d[1])*q+d[2])*q+d[3])*q+1)
    q, r = p - 0.5, (p - 0.5) ** 2
    return (((((a[0]*r+a[1])*r+a[2])*r+a[3])*r+a[4])*r+a[5])*q / \
           (((((b[0]*r+b[1])*r+b[2])*r+b[3])*r+b[4])*r+1)


def _rectangular_lrv(d: np.ndarray, h: int) -> float:
    """``(gamma_0 + 2 sum_{k=1}^{h-1} gamma_k) / T``, the variance of the mean differential."""
    t = len(d)
    dm = d - d.mean()
    total = float(dm @ dm) / t
    for k in range(1, min(h, t)):
        total += 2.0 * float(dm[k:] @ dm[:-k]) / t
    return total / t


def _bartlett_lrv(d: np.ndarray, h: int) -> float:
    """Bartlett-weighted long-run variance, used only if the rectangular one is not positive."""
    t = len(d)
    dm = d - d.mean()
    total = float(dm @ dm) / t
    for k in range(1, min(h, t)):
        total += 2.0 * (1.0 - k / h) * float(dm[k:] @ dm[:-k]) / t
    return total / t


@dataclass(frozen=True, slots=True)
class TestResult:
    """One forecast-comparison test, carrying everything needed to redo it."""

    name: str
    statistic: float
    p_value: float
    T: int
    h: int
    one_sided: bool
    fallback_fired: bool

    def __str__(self) -> str:
        tail = "  [Bartlett fallback fired]" if self.fallback_fired else ""
        side = "one-sided" if self.one_sided else "two-sided"
        return (
            f"{self.name}: S*={self.statistic:+.4f}  p={self.p_value:.4g} "
            f"({side})  T={self.T}  h={self.h}{tail}"
        )


def _upper_tail(stat: float, df: int) -> float:
    """``P(T_df > stat)`` — Student-t, falling back to the normal without scipy."""
    try:
        from scipy import stats as _stats

        return float(_stats.t.sf(stat, df=df))
    except ImportError:  # pragma: no cover - scipy ships with the Kaggle image
        return 0.5 * math.erfc(stat / math.sqrt(2.0))


def _hln_and_p(d: np.ndarray, h: int, name: str, one_sided: bool) -> TestResult:
    """Harvey-Leybourne-Newbold correction, referred to ``t(T-1)``.

    ``S* = S sqrt[(T + 1 - 2h + h(h-1)/T) / T]``, compared against Student-t
    with ``T-1`` degrees of freedom — **not** the standard normal. The factor is
    asserted positive before use: at ``h = 24`` it is exactly 0 at ``T = 24`` and
    0.047 at ``T = 30``, precisely the T a non-overlapping 30-day block would
    produce, so a silent negative would yield a complex statistic reported as a
    real one.
    """
    d = np.asarray(d, dtype=np.float64)
    t = len(d)
    if t < 2:
        raise ValueError(f"{name}: T={t} is too small for a loss differential")
    factor = (t + 1 - 2 * h + h * (h - 1) / t) / t
    if factor <= 0:
        raise ValueError(
            f"{name}: the HLN factor is {factor:.4f} <= 0 at T={t}, h={h}. "
            f"The test is not reported where the factor fails; state T instead."
        )

    variance = _rectangular_lrv(d, h)
    fallback = False
    if variance <= 0:
        variance = _bartlett_lrv(d, h)
        fallback = True
        if variance <= 0:
            raise ValueError(f"{name}: no positive long-run variance at T={t}")

    stat = float(d.mean() / math.sqrt(variance) * math.sqrt(factor))
    upper = _upper_tail(abs(stat), t - 1)
    if one_sided:
        p = upper if stat >= 0 else 1.0 - upper
    else:
        p = 2.0 * upper
    return TestResult(name, stat, float(min(p, 1.0)), t, h, one_sided, fallback)


def hln_test(
    d: np.ndarray, h: int, name: str = "HLN", one_sided: bool = False
) -> TestResult:
    """Harvey-Leybourne-Newbold test on a loss differential ``d`` at horizon ``h``.

    The statistic uses the rectangular long-run variance, falling back to Bartlett
    (and saying so) if it is not positive, and is compared with ``t(T-1)``.
    """
    return _hln_and_p(np.asarray(d, dtype=np.float64), h, name, one_sided)


def dm_test(
    loss_a: np.ndarray, loss_b: np.ndarray, h: int, name: str = "DM"
) -> TestResult:
    """Diebold-Mariano on the loss differential ``loss_a - loss_b``, two-sided.

    Against Naive-RW, which every model nests, use :func:`clark_west_test`.
    """
    return _hln_and_p(
        np.asarray(loss_a) - np.asarray(loss_b), h, name, one_sided=False
    )


def clark_west_test(
    y: np.ndarray,
    pred_small: np.ndarray,
    pred_large: np.ndarray,
    h: int,
    name: str = "Clark-West",
) -> TestResult:
    """Clark-West (2007) for a nested pair; here only a model against Naive-RW.

    ``f_t = (y - y_small)^2 - (y - y_large)^2 + (y_small - y_large)^2``

    The third term removes the larger model's estimation noise, which under the
    null would make it look worse. One-sided: the alternative is that the larger
    model helps. A diagnostic, and it can be positive beside a negative R2_oos.
    """
    y = np.asarray(y, dtype=np.float64)
    s = np.asarray(pred_small, dtype=np.float64)
    lg = np.asarray(pred_large, dtype=np.float64)
    f = np.square(y - s) - np.square(y - lg) + np.square(s - lg)
    return _hln_and_p(f, h, name, one_sided=True)


def per_origin_loss(frame: pl.DataFrame) -> pl.DataFrame:
    """Mean squared error per forecast origin: the series the DM test consumes."""
    return (
        frame.with_columns((pl.col("y_true") - pl.col("y_pred")).pow(2).alias("_se"))
        .group_by(["block", "timestamp"])
        .agg(pl.col("_se").mean().alias("loss"))
        .sort(["block", "timestamp"])
    )


# -- RQ2's core regression: beta1 with origin FE and a wild cluster bootstrap -


@dataclass(frozen=True, slots=True)
class Beta1Result:
    """``A(i,b) = alpha_i + beta1 b + eps`` with origin-clustered inference."""

    beta1: float
    t_statistic: float
    cluster_se: float
    p_rademacher: float
    p_webb: float
    n_clusters: int
    n_observations: int
    within_slopes: np.ndarray
    B: int

    @property
    def headline_p(self) -> float:
        """The more conservative of the Rademacher and Webb bootstrap p-values."""
        return max(self.p_rademacher, self.p_webb)

    def __str__(self) -> str:
        return (
            f"beta1 = {self.beta1:+.6f}   t = {self.t_statistic:+.3f}   "
            f"G = {self.n_clusters}   N = {self.n_observations}\n"
            f"WCR one-sided p (H1: beta1 < 0): Rademacher {self.p_rademacher:.4f}, "
            f"Webb {self.p_webb:.4f}  ->  headline {self.headline_p:.4f}\n"
            f"Effective independence is bounded near 4 by the training-window "
            f"overlap, well below G = {self.n_clusters}."
        )


def _weights(kind: str, shape: tuple[int, int], rng: np.random.Generator) -> np.ndarray:
    if kind == "rademacher":
        return rng.choice(np.array([-1.0, 1.0]), size=shape)
    if kind == "webb":
        # Webb's 6-point distribution. At G = 15 Rademacher already admits
        # 2^15 = 32,768 distinct draws, a minimum two-sided p of about 6e-5, so
        # the original small-G justification for preferring Webb no longer
        # binds — both are reported and the more conservative is the headline.
        atoms = np.array([
            -math.sqrt(1.5), -1.0, -math.sqrt(0.5),
            math.sqrt(0.5), 1.0, math.sqrt(1.5),
        ])
        return rng.choice(atoms, size=shape)
    raise ValueError(f"unknown weight scheme {kind!r}")


def _balanced_matrix(panel: pl.DataFrame, value: str) -> tuple[np.ndarray, np.ndarray]:
    """``(G x B)`` outcome matrix and the block axis, or a loud failure.

    Built by hand rather than with ``pivot`` so the code does not depend on which
    polars major version the Kaggle image happens to ship.
    """
    origins = sorted(set(panel.get_column("origin").to_list()))
    blocks = sorted(set(int(b) for b in panel.get_column("block").to_list()))
    if not origins:
        raise ValueError("empty panel: no origin left to estimate beta1 on")
    index ={(o, b): i for i, (o, b) in enumerate([(o, b) for o in origins for b in blocks])}

    out = np.full(len(index), np.nan)
    for origin, block, val in zip(
        panel.get_column("origin").to_list(),
        panel.get_column("block").to_list(),
        panel.get_column(value).to_list(),
    ):
        out[index[(str(origin), int(block))]] = float(val)

    matrix = out.reshape(len(origins), len(blocks))
    if np.isnan(matrix).any():
        raise ValueError(
            "unbalanced panel: beta1's reduction to the mean of within-slopes "
            "holds only when every origin carries every block"
        )
    return matrix, np.array(blocks, dtype=np.float64)


def panel_beta1(
    panel: pl.DataFrame,
    value: str = "A",
    B: int = 99_999,
    seed: int = 42,
) -> Beta1Result:
    """Fit ``A(i,b) = alpha_i + beta1 b + eps`` and test ``H1: beta1 < 0``.

    Origin fixed effects; a restricted wild cluster bootstrap of the cluster-robust t
    with Rademacher and Webb weights; ``p = (1 + count) / (1 + B)``; reference
    ``t(G - 1)``.
    """
    a, x = _balanced_matrix(panel, value)
    g, n_blocks = a.shape
    xd = x - x.mean()
    sxx = float(xd @ xd)

    within = a - a.mean(axis=1, keepdims=True)
    beta = float((within * xd).sum() / (g * sxx))
    resid = within - beta * xd
    score = resid @ xd
    variance = float((score @ score) / (g * sxx) ** 2)
    se = math.sqrt(variance) if variance > 0 else float("nan")
    t_obs = beta / se if se == se and se > 0 else float("nan")

    # Restricted residuals: with beta1 = 0 imposed the fitted value is the origin
    # mean, so u_tilde is exactly the within-origin demeaned outcome. Because
    # each row of u_tilde already sums to zero, the bootstrap origin means are
    # unchanged and the whole replication collapses to s = u_tilde @ xd.
    s = within @ xd

    def _p(kind: str) -> float:
        rng = np.random.default_rng(seed)
        weights = _weights(kind, (B, g), rng)
        beta_star = (weights @ s) / (g * sxx)
        score_star = weights * s[None, :] - beta_star[:, None] * sxx
        var_star = np.square(score_star).sum(axis=1) / (g * sxx) ** 2
        ok = var_star > 0
        t_star = beta_star[ok] / np.sqrt(var_star[ok])
        # (1 + count) / (1 + B), not count / B (Davison & Hinkley 1997): the
        # observed statistic is one of its own reference distribution, and the
        # naive form returns a literal p = 0, which is not a probability any
        # finite bootstrap can support. At B = 99,999 the floor it reports is
        # 1e-5, and at G = 15 Rademacher's own granularity bounds it at ~3e-5
        # anyway — so the floor is honest rather than conservative padding.
        below = int(np.sum(t_star <= t_obs))  # H1: beta1 < 0, left tail
        return (1.0 + below) / (1.0 + int(ok.sum()))

    return Beta1Result(
        beta1=beta,
        t_statistic=t_obs,
        cluster_se=se,
        p_rademacher=_p("rademacher"),
        p_webb=_p("webb"),
        n_clusters=g,
        n_observations=g * n_blocks,
        within_slopes=(within * xd).sum(axis=1) / sxx,
        B=B,
    )


@dataclass(frozen=True, slots=True)
class EquivalenceResult:
    """TOST verdict on a rung expected to be flat."""

    mean_delta: float
    margin: float
    p_lower: float
    p_upper: float
    n: int

    @property
    def equivalent(self) -> bool:
        return max(self.p_lower, self.p_upper) < 0.05

    def __str__(self) -> str:
        verdict = "EQUIVALENT (flat)" if self.equivalent else "NOT shown equivalent"
        return (
            f"TOST: mean delta = {self.mean_delta:+.6f}, margin = +/-{self.margin:.6f}, "
            f"p = ({self.p_lower:.4f}, {self.p_upper:.4f}), G = {self.n}  ->  {verdict}"
        )


def tost_equivalence(
    deltas: np.ndarray, margin: float, alpha: float = 0.05
) -> EquivalenceResult:
    """Two one-sided tests of ``|mean(deltas)| < margin``, RQ1's equivalence check."""
    d = np.asarray(deltas, dtype=np.float64)
    n = len(d)
    if n < 2:
        raise ValueError("TOST needs at least two clusters")
    se = float(np.std(d, ddof=1) / math.sqrt(n))
    if se <= 0:
        raise ValueError("zero dispersion across clusters; TOST is undefined")
    mean = float(d.mean())
    return EquivalenceResult(
        mean_delta=mean,
        margin=abs(margin),
        p_lower=_upper_tail((mean + abs(margin)) / se, n - 1),   # H0: mu <= -margin
        p_upper=_upper_tail(-(mean - abs(margin)) / se, n - 1),  # H0: mu >= +margin
        n=n,
    )


def j_test(
    y: np.ndarray, x_a: np.ndarray, x_b: np.ndarray, groups: np.ndarray,
    *, clusters: np.ndarray | None = None,
) -> tuple[float, float]:
    """Davidson-MacKinnon J test of K against K_eff, with CR1 covariance and ``t(G - 1)``."""
    y, x_a, x_b = [np.asarray(v, dtype=np.float64) for v in (y, x_a, x_b)]
    groups = np.asarray(groups)
    clusters = groups if clusters is None else np.asarray(clusters)
    if any(v.ndim != 1 or len(v) != len(y) for v in (y, x_a, x_b, groups, clusters)):
        raise ValueError("J-test arrays must be one-dimensional with equal lengths")
    if not all(np.all(np.isfinite(v)) for v in (y, x_a, x_b)):
        raise ValueError("J-test inputs must be finite")
    def _demean(v: np.ndarray) -> np.ndarray:
        out = v.copy()
        for group in np.unique(groups):
            mask = groups == group
            out[mask] -= v[mask].mean()
        return out
    yd, ad, bd = [_demean(v) for v in (y, x_a, x_b)]
    n, g = len(y), len(np.unique(clusters))
    dof = n - 2 - len(np.unique(groups))
    if g < 2 or dof <= 0 or not bd @ bd > 0:
        return float("nan"), float("nan")
    fitted_b = bd * float((bd @ yd) / (bd @ bd))
    design = np.column_stack([ad, fitted_b])
    if np.linalg.matrix_rank(design) < 2:
        return float("nan"), float("nan")
    coef = np.linalg.lstsq(design, yd, rcond=None)[0]
    resid = yd - design @ coef
    bread = np.linalg.pinv(design.T @ design)
    scores = np.array([design[clusters == c].T @ resid[clusters == c]
                       for c in np.unique(clusters)])
    cov = (g / (g - 1)) * ((n - 1) / dof) * bread @ (scores.T @ scores) @ bread
    se = math.sqrt(max(float(cov[1, 1]), 0.0))
    if se <= 0:
        return float("nan"), float("nan")
    statistic = float(coef[1] / se)
    return statistic, 2.0 * _upper_tail(abs(statistic), g - 1)


def minimum_detectable_beta1(
    within_slopes: np.ndarray, alpha: float = 0.05, power: float = 0.80
) -> float:
    """Plug-in sensitivity of beta1 under independent-origin assumptions.

    The caller supplies slopes. In this study these are observed TEST slopes,
    so the computed value is post-analysis sensitivity, not prospective power."""
    g = len(within_slopes)
    if g < 2:
        return float("nan")
    se = float(np.std(within_slopes, ddof=1) / math.sqrt(g))
    return -(_normal_quantile(1 - alpha) + _normal_quantile(power)) * se


def raw_scale_table(seed_avg: pl.DataFrame) -> pl.DataFrame:
    """Add RMSE in raw log-return units to a seed-averaged table."""
    return seed_avg.with_columns(
        (pl.col("mse").sqrt() * pl.col("sigma_g")).alias("rmse_raw")
    )


def per_origin_relmse(seed_avg: pl.DataFrame, model: str, k: int | None = None) -> pl.DataFrame:
    """Equal-weight block RelMSE, matching the headline and comparison panel."""
    part = seed_avg.filter((pl.col("model") == model) & (pl.col("pred_len") == PRED_LEN))
    if k is not None:
        part = part.filter(pl.col("k") == k)
    return part.group_by("origin").agg(
        pl.col("rel_mse").mean(), pl.col("n_windows").sum()
    ).sort("origin")


def paired_contrast(
    seed_avg: pl.DataFrame,
    left: tuple[str, int | None],
    right: tuple[str, int | None],
) -> dict:
    """Exploratory paired contrast of mean seed loss, equal blocks and origins.

    Positive means left is worse. Calendar hashes must agree when supplied.
    The t(G-1) interval/p-value assumes independent origins, which the overlapping
    study does not establish. Feature content and PR both change in matched-K."""
    if "evaluation_keys_sha256" in seed_avg.columns:
        parts = []
        for tag, k in (left, right):
            part = seed_avg.filter((pl.col("model") == tag) & (pl.col("pred_len") == PRED_LEN))
            if k is not None:
                part = part.filter(pl.col("k") == k)
            parts.append(part.select("origin_index", "block", "evaluation_keys_sha256"))
        check = parts[0].join(parts[1], on=["origin_index", "block"], suffix="_right")
        if check.filter(pl.col("evaluation_keys_sha256") != pl.col("evaluation_keys_sha256_right")).height:
            raise ValueError("paired contrast uses different actual targets")
    a = per_origin_relmse(seed_avg, left[0], left[1])
    b = per_origin_relmse(seed_avg, right[0], right[1])
    joined = a.join(b, on="origin", how="inner", suffix="_right").sort("origin")
    diff = (
        joined.get_column("rel_mse").to_numpy()
        - joined.get_column("rel_mse_right").to_numpy()
    )
    g = int(diff.size)
    if g == 0:
        raise ValueError(f"paired contrast {left} vs {right}: the two arms share no origin")
    label = lambda arm: arm[0] if arm[1] is None else f"{arm[0]}-K{arm[1]}"
    if g < 2:
        return {
            "left": label(left), "right": label(right), "n_origins": g,
            "mean_diff": None, "se": None, "t": None, "p_two_sided": None,
            "ci_low": None, "ci_high": None, "left_better": None,
        }

    mean = float(diff.mean())
    se = float(diff.std(ddof=1) / math.sqrt(g))
    if se > 0:
        t_stat = mean / se
        p = 2.0 * _upper_tail(abs(t_stat), g - 1)
        half = _t_critical(g - 1) * se
    else:
        t_stat, p, half = (0.0, 1.0, 0.0) if mean == 0.0 else (math.inf, 0.0, 0.0)

    return {
        "left": label(left),
        "right": label(right),
        "mean_diff": mean,
        "se": se,
        "t": t_stat,
        "p_two_sided": float(p),
        "ci_low": mean - half,
        "ci_high": mean + half,
        "n_origins": g,
        "left_better": int((diff < 0).sum()),
        "inference_status": "exploratory; independent-origin t approximation only",
    }


def _t_critical(df: int) -> float:
    """Two-sided 5% Student-t critical value, normal fallback without scipy."""
    try:
        from scipy import stats as _stats

        return float(_stats.t.ppf(0.975, df=df))
    except ImportError:  # pragma: no cover - scipy ships with the Kaggle image
        return 1.959963984540054


def beta1_with_coverage(
    panel: pl.DataFrame,
    min_coverage: float = 0.9,
    B: int = 99_999,
    seed: int = 42,
) -> tuple[Beta1Result, Beta1Result | None]:
    """``beta1`` on the full panel, and on blocks with at least ``min_coverage`` windows kept."""
    full = panel_beta1(panel, B=B, seed=seed)
    restricted = panel.filter((pl.col("n_large") / float(BLOCK_HOURS)) >= min_coverage)
    try:
        return full, panel_beta1(restricted, B=B, seed=seed)
    except ValueError:
        # Unbalanced after the restriction. Loosening the estimator to produce a
        # number here would answer a different question than the one asked.
        return full, None


def panel_beta1_covariate(
    panel: pl.DataFrame,
    value: str = "A",
    covariate: str = "coverage",
    B: int = 99_999,
    seed: int = 42,
) -> Beta1Result:
    """``A(i,b) = alpha_i + beta1 b + beta2 c(i,b) + eps``, clustered on origin."""
    a, blocks = _balanced_matrix(panel, value)
    c, _ = _balanced_matrix(panel, covariate)
    g, n_blocks = a.shape

    within = a - a.mean(axis=1, keepdims=True)
    cw = c - c.mean(axis=1, keepdims=True)
    xw = np.tile(blocks - blocks.mean(), (g, 1))

    scc = float((cw * cw).sum())
    if scc <= 0.0:
        raise ValueError(
            "coverage has no within-origin variation, so it cannot be a "
            "covariate here; panel_beta1 is the estimator that applies"
        )

    # Frisch-Waugh: residualise the regressor of interest and the outcome on the
    # control, both already swept of origin means. beta1 and the residuals of the
    # two-regressor fit are then exactly those of the simple fit on the residuals.
    xr = xw - (float((xw * cw).sum()) / scc) * cw
    ar = within - (float((within * cw).sum()) / scc) * cw
    sxx = float((xr * xr).sum())
    if sxx <= 0.0:
        raise ValueError("the block index is collinear with coverage within origin")

    beta = float((ar * xr).sum() / sxx)
    resid = ar - beta * xr
    score = (resid * xr).sum(axis=1)
    variance = float((score * score).sum()) / sxx**2
    se = math.sqrt(variance) if variance > 0 else float("nan")
    t_obs = beta / se if se == se and se > 0 else float("nan")

    # Restricted residuals: imposing beta1 = 0 leaves alpha_i and beta2, and ar is
    # already swept of both, so u_tilde is ar itself. Per-cluster inner products
    # against the fixed regressors are all a bootstrap draw needs.
    ux = (ar * xr).sum(axis=1)
    uc = (ar * cw).sum(axis=1)
    xx = (xr * xr).sum(axis=1)
    cx = (cw * xr).sum(axis=1)

    def _p(kind: str) -> float:
        rng = np.random.default_rng(seed)
        weights = _weights(kind, (B, g), rng)
        beta_star = (weights @ ux) / sxx
        delta_star = (weights @ uc) / scc
        score_star = (
            weights * ux[None, :]
            - delta_star[:, None] * cx[None, :]
            - beta_star[:, None] * xx[None, :]
        )
        var_star = np.square(score_star).sum(axis=1) / sxx**2
        ok = var_star > 0
        t_star = beta_star[ok] / np.sqrt(var_star[ok])
        below = int(np.sum(t_star <= t_obs))  # H1: beta1 < 0, left tail
        return (1.0 + below) / (1.0 + int(ok.sum()))

    return Beta1Result(
        beta1=beta,
        t_statistic=t_obs,
        cluster_se=se,
        p_rademacher=_p("rademacher"),
        p_webb=_p("webb"),
        n_clusters=g,
        n_observations=g * n_blocks,
        within_slopes=(ar * xr).sum(axis=1) / (xr * xr).sum(axis=1),
        B=B,
    )


###

<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b54; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #ffb4a2; font-size: 1em; margin: 0;">⚖️ <code>comparisons.py</code></h3> <p style="display: inline; color: #ffd8c2; font-size: 0.9em; margin: 0;">· Matriks pasangan, Romano–Wolf, dan Model Confidence Set.</p></div>

In [29]:
"""Table 6: every pair of models, the statistic for each, and multiplicity control.

Pairs are compared on identical actual targets. Losses are averaged over seeds
first, then over blocks with equal weight, then compared per origin; the 15
origins are the clusters. Romano-Wolf controls the family-wise error over all
pairs and within each declared family, and the Model Confidence Set is reported
at 90% and 75%. All of it is diagnostic, because origins share training data.

Upstream:
    Written here after J. P. Romano and M. Wolf, Econometrica 73(4), 2005, and
    P. R. Hansen, A. Lunde and J. M. Nason, Econometrica 79(2), 2011.
"""

ModelKey = tuple[str, int]

#: Naive-RW's sentinel key.
NAIVE: Final[ModelKey] = ("naive", 0)


DEFAULT_B: Final = 9_999

MCS_LEVELS: Final[tuple[float, ...]] = (0.10, 0.25)

FAMILY_ORDER: Final[tuple[str, ...]] = (
    "vs-naive", "ladder", "cross-model", "other",
)


def pair_family(left: ModelKey, right: ModelKey) -> str:
    """The claim a pair speaks to, decided from the keys alone.

    ``vs-naive``: any pair with Naive-RW. ``ladder``: the iTransformer at two
    rungs (RQ1). ``cross-model``: the iTransformer against Ridge or the vanilla
    Transformer at the same K (C1, C2). ``other``: every remaining pair.
    """
    if NAIVE in (left, right):
        return "vs-naive"
    if left[0] == right[0] == "itr":
        return "ladder"
    if left[1] == right[1] and "itr" in (left[0], right[0]):
        return "cross-model"
    return "other"


def label(key: ModelKey) -> str:
    """``itr-K8``, or ``Naive-RW`` for the sentinel."""
    return "Naive-RW" if key == NAIVE else f"{key[0]}-K{key[1]}"


# -- the aligned prediction panel --------------------------------------------


@dataclass(frozen=True, slots=True)
class PredictionPanel:
    """Aligned forecast points with mean-seed losses as the estimand.

    y_pred stores the ensemble for inspection only. Production loss comparisons
    read seed_losses, then equally average block RelMSE and origin values."""

    keys: tuple[ModelKey, ...]
    origin_indices: tuple[int, ...]
    origins: tuple[str, ...]
    #: origin index -> ``(n_rows,)`` block labels, sorted with the arrays below.
    block: dict[int, np.ndarray]
    #: origin index -> ``(n_rows,)`` realised target, shared by every model.
    y_true: dict[int, np.ndarray]
    #: ``(key, origin index)`` -> ``(n_rows,)`` seed-averaged forecast.
    y_pred: dict[tuple[ModelKey, int], np.ndarray]
    #: Forecast steps per window, so a per-origin reduction can recover ``T``.
    pred_len: int
    seed_losses: dict[tuple[ModelKey, int], np.ndarray] | None = None


def _run_ids(
    key: ModelKey, origin_index: int, roots: list[Path], pred_len: int
) -> list[str]:
    """Every seed of one cell that is actually on disk, in seed order."""
    model, k = key
    stem = f"{model}_o{origin_index:02d}_K{k:02d}_H{pred_len:03d}_s"
    found: set[str] = set()
    for root in roots:
        for path in (root / "preds").glob(f"{stem}*.parquet"):
            found.add(path.stem)
    return sorted(found, key=lambda run_id: int(parse_run_id(run_id)["seed"]))


def available_keys(
    keys: list[ModelKey],
    roots: list[Path],
    pred_len: int = PRED_LEN,
    origin_indices: tuple[int, ...] | None = None,
) -> tuple[list[ModelKey], list[ModelKey]]:
    """Split ``keys`` into those with a run at every origin, and the rest."""
    indices = origin_indices or tuple(o.index for o in ORIGINS)
    present: list[ModelKey] = []
    absent: list[ModelKey] = []
    for key in keys:
        if key == NAIVE or any(
            _run_ids(key, index, roots, pred_len) for index in indices
        ):
            present.append(key)
        else:
            absent.append(key)
    return present, absent


def build_panel(
    keys: list[ModelKey], roots: list[Path], pred_len: int = PRED_LEN,
    origin_indices: tuple[int, ...] | None = None,
    *, windows: pl.DataFrame | None = None,
) -> PredictionPanel:
    """Stack every model's predictions on identical actual targets, with seed-averaged losses.

    Raises:
        FileNotFoundError: If a model has no run at an origin.
        ValueError: If runs disagree on targets or lack provenance.
    """
    if not any(key != NAIVE for key in keys):
        raise ValueError("a comparison panel needs a persisted forecast")
    indices = origin_indices or tuple(o.index for o in ORIGINS)
    block, y_true, y_pred, seed_losses = {}, {}, {}, {}
    vintages = set()
    for index in indices:
        signature = None
        naive_z = None
        for key in keys:
            if key == NAIVE:
                continue
            runs = _run_ids(key, index, roots, pred_len)
            if not runs:
                raise FileNotFoundError(f"{key} has no run at origin {index} (H={pred_len})")
            stacked = []
            for run_id in runs:
                meta = load_meta(run_id, roots)
                vintage = (meta.get("input_sha256"), meta.get("code_sha256"))
                if any(not v or v == "unknown" for v in vintage):
                    raise ValueError(f"{run_id}: missing analysis provenance")
                vintages.add(vintage)
                frame = load_predictions(run_id, roots)
                if windows is not None:
                    keep = windows.filter((pl.col("origin_index") == index) & (pl.col("pred_len") == pred_len))
                    frame = frame.join(keep.select("block", "timestamp"), on=["block", "timestamp"], how="semi").sort(["block", "timestamp", "step"])
                sig = frame.select("block", "timestamp", "step", "target_timestamp")
                actual = frame["y_true"].to_numpy().astype(np.float64)
                if signature is None:
                    signature = sig
                    block[index] = frame["block"].to_numpy()
                    y_true[index] = actual
                    naive_z = float(meta["naive_rw_z"])
                elif not sig.equals(signature):
                    raise ValueError(f"{run_id}: evaluated window sets differ at origin {index}")
                elif not np.allclose(actual, y_true[index], rtol=1e-6, atol=1e-8):
                    raise ValueError(f"{run_id}: target values or scaler differ")
                stacked.append(frame["y_pred"].to_numpy().astype(np.float64))
            y_pred[key, index] = np.mean(stacked, axis=0)
            seed_losses[key, index] = np.mean(
                np.square(y_true[index][None, :] - np.stack(stacked)), axis=0
            )
        y_pred[NAIVE, index] = np.full(len(y_true[index]), naive_z)
        seed_losses[NAIVE, index] = np.square(y_true[index] - naive_z)
    if len(vintages) != 1:
        raise ValueError("comparison panel mixes code or input vintages")
    return PredictionPanel(
        keys=tuple(keys), origin_indices=tuple(indices),
        origins=tuple(ORIGINS[i-1].label for i in indices), block=block,
        y_true=y_true, y_pred=y_pred, pred_len=pred_len, seed_losses=seed_losses,
    )


def _per_window(values: np.ndarray, pred_len: int) -> np.ndarray:
    """Mean over the forecast steps of each window."""
    return values.reshape(-1, pred_len).mean(axis=1)


def differential(panel: PredictionPanel, left: ModelKey, right: ModelKey,
                 origin_index: int) -> np.ndarray:
    """Per-forecast loss differential, after averaging each model's loss across seeds."""
    def loss(key: ModelKey) -> np.ndarray:
        if panel.seed_losses is not None:
            return panel.seed_losses[key, origin_index]
        return np.square(panel.y_true[origin_index] - panel.y_pred[key, origin_index])
    return _per_window(loss(left) - loss(right), panel.pred_len)


def per_origin_differential(panel: PredictionPanel, left: ModelKey,
                            right: ModelKey) -> np.ndarray:
    """Per-origin loss differential with equal block weights."""
    return per_origin_loss(panel, left) - per_origin_loss(panel, right)


def per_origin_loss(panel: PredictionPanel, key: ModelKey) -> np.ndarray:
    """Per-origin RelMSE with equal block weights, from seed-averaged step losses."""
    means = []
    for index in panel.origin_indices:
        loss = (panel.seed_losses[key, index] if panel.seed_losses is not None else
                np.square(panel.y_true[index] - panel.y_pred[key, index]))
        values = []
        for b in np.unique(panel.block[index]):
            mask = panel.block[index] == b
            denominator = (panel.seed_losses[NAIVE, index][mask].mean()
                           if panel.seed_losses is not None else 1.0)
            if denominator <= 0:
                raise ValueError("Naive-RW block MSE must be positive")
            values.append(loss[mask].mean() / denominator)
        means.append(np.mean(values))
    return np.asarray(means)


# -- clustered inference over origins ----------------------------------------


def _studentised(matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Column means and their cluster standard errors, ``G = matrix.shape[0]``."""
    g = matrix.shape[0]
    return matrix.mean(axis=0), matrix.std(axis=0, ddof=1) / math.sqrt(g)


def cluster_bootstrap_t(
    per_origin: np.ndarray, B: int = DEFAULT_B, seed: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    """Observed and bootstrap studentised statistics, resampling **origins**.

    Args:
        per_origin: ``(G, P)`` -- one mean differential per origin, per pair.
        B: Bootstrap draws.
        seed: Generator seed.

    Returns:
        ``(t_obs, t_boot)`` of shapes ``(P,)`` and ``(B, P)``. The bootstrap
        statistics are centred on the observed mean, so they are draws from the
        null. A resample that happens to pick one origin ``G`` times has no
        dispersion; it contributes 0 rather than an infinity.
    """
    g = per_origin.shape[0]
    theta, se = _studentised(per_origin)
    with np.errstate(divide="ignore", invalid="ignore"):
        t_obs = np.where(se > 0, theta / se, 0.0)

    rng = np.random.default_rng(seed)
    draws = per_origin[rng.integers(0, g, size=(B, g))]
    theta_b = draws.mean(axis=1)
    se_b = draws.std(axis=1, ddof=1) / math.sqrt(g)
    with np.errstate(divide="ignore", invalid="ignore"):
        t_boot = np.where(se_b > 0, (theta_b - theta) / se_b, 0.0)
    return t_obs, t_boot


def romano_wolf(
    per_origin: np.ndarray, B: int = DEFAULT_B, seed: int = 42
) -> np.ndarray:
    """Stepdown FWER-controlled p-values across every pair (Romano & Wolf 2005).

    Two-sided: the family asks whether two models differ in predictive ability
    at all, in either direction.

    Args:
        per_origin: ``(G, P)`` mean differential per origin, per pair.
        B: Bootstrap draws.
        seed: Generator seed.

    Returns:
        ``(P,)`` adjusted p-values, monotone in ``|t|``.
    """
    t_obs, t_boot = cluster_bootstrap_t(per_origin, B=B, seed=seed)
    order = list(np.argsort(-np.abs(t_obs)))
    adjusted = np.empty(per_origin.shape[1])
    remaining = list(order)
    running = 0.0
    for position in order:
        block_max = np.abs(t_boot[:, remaining]).max(axis=1)
        raw = (1 + int((block_max >= abs(t_obs[position])).sum())) / (1 + B)
        running = max(running, raw)  # stepdown monotonicity
        adjusted[position] = min(running, 1.0)
        remaining.remove(position)
    return adjusted


def model_confidence_set(
    losses: np.ndarray, alpha: float, B: int = DEFAULT_B, seed: int = 42
) -> list[int]:
    """Model Confidence Set at level ``alpha`` by the ``T_max`` statistic (Hansen, Lunde and Nason, 2011)."""
    g, m = losses.shape
    alive = list(range(m))
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, g, size=(B, g))

    while len(alive) > 1:
        sub = losses[:, alive]
        deviation = sub - sub.mean(axis=1, keepdims=True)
        theta, se = _studentised(deviation)
        with np.errstate(divide="ignore", invalid="ignore"):
            t = np.where(se > 0, theta / se, 0.0)
        t_max = float(t.max())

        draws = deviation[idx]
        theta_b = draws.mean(axis=1)
        se_b = draws.std(axis=1, ddof=1) / math.sqrt(g)
        with np.errstate(divide="ignore", invalid="ignore"):
            t_b = np.where(se_b > 0, (theta_b - theta) / se_b, 0.0)
        p = (1 + int((t_b.max(axis=1) >= t_max).sum())) / (1 + B)

        if p >= alpha:
            break
        alive.pop(int(np.argmax(t)))  # eliminate the worst, then re-test
    return alive


# -- Table 6 -----------------------------------------------------------------


def _cell_diagnostics(
    panel: PredictionPanel, left: ModelKey, right: ModelKey
) -> dict[str, float | int | bool]:
    """Median HLN statistic over the (origin, block) cells, and how many reject."""
    stats: list[float] = []
    rejects = 0
    t_min = -1
    fallback = False
    name = f"{label(left)} vs {label(right)}"
    for index in panel.origin_indices:
        d = differential(panel, left, right, index)
        blocks = _per_window(
            panel.block[index].astype(np.float64), panel.pred_len
        ).round()
        for b in range(1, TEST_BLOCKS + 1):
            cell = d[blocks == float(b)]
            if len(cell) < 2:
                continue
            result = hln_test(cell, panel.pred_len, name=name)
            stats.append(result.statistic)
            rejects += int(result.p_value < 0.05)
            t_min = result.T if t_min < 0 else min(t_min, result.T)
            fallback = fallback or result.fallback_fired
    return {
        "s_star_median": float(np.median(stats)) if stats else float("nan"),
        "n_cells": len(stats),
        "n_cells_reject": rejects,
        "T_min": t_min,
        "fallback_fired": fallback,
    }


def pair_matrix(
    panel: PredictionPanel, B: int = DEFAULT_B, seed: int = 42
) -> pl.DataFrame:
    """Table 6: every unordered pair with its statistic, raw and Romano-Wolf p-values, and MCS flags."""
    keys = list(panel.keys)
    # No fitted pair is treated as nested, so every pair gets the same two-sided
    # unadjusted statistic; Clark-West is kept for comparisons with Naive-RW.
    pairs = [(a, b) for i, a in enumerate(keys) for b in keys[i + 1 :]]

    per_origin = np.column_stack(
        [per_origin_differential(panel, a, b) for a, b in pairs]
    )
    t_obs, t_boot = cluster_bootstrap_t(per_origin, B=B, seed=seed)
    p_adjusted = romano_wolf(per_origin, B=B, seed=seed)

    families = [pair_family(a, b) for a, b in pairs]
    p_family = np.ones(len(pairs))
    for name in FAMILY_ORDER:
        members = [i for i, f in enumerate(families) if f == name]
        if not members:
            continue
        p_family[members] = romano_wolf(per_origin[:, members], B=B, seed=seed)

    losses = np.column_stack([per_origin_loss(panel, k) for k in keys])
    members = {
        alpha: {keys[i] for i in model_confidence_set(losses, alpha, B=B, seed=seed)}
        for alpha in MCS_LEVELS
    }

    rows = []
    for position, (left, right) in enumerate(pairs):
        t = float(t_obs[position])
        count = int((np.abs(t_boot[:, position]) >= abs(t)).sum())
        rows.append(
            {
                "left": label(left),
                "right": label(right),
                "statistic_name": "unadjusted forecast-loss diagnostic",
                "inference_status": "exploratory; cross-origin dependence unresolved",
                "estimand": "mean seed loss, then equal block means",
                "t_cluster": t,
                "p_raw": (1 + count) / (1 + B),
                "p_romano_wolf": float(p_adjusted[position]),
                "family": families[position],
                "p_romano_wolf_family": float(p_family[position]),
                **_cell_diagnostics(panel, left, right),
                "h": panel.pred_len,
                "G": int(per_origin.shape[0]),
                "left_in_mcs_90": left in members[0.10],
                "right_in_mcs_90": right in members[0.10],
                "left_in_mcs_75": left in members[0.25],
                "right_in_mcs_75": right in members[0.25],
            }
        )
    return pl.DataFrame(rows)


def mcs_table(
    panel: PredictionPanel, B: int = DEFAULT_B, seed: int = 42
) -> pl.DataFrame:
    """MCS membership per model, with its mean loss and rank."""
    keys = list(panel.keys)
    losses = np.column_stack([per_origin_loss(panel, k) for k in keys])
    members = {
        alpha: {keys[i] for i in model_confidence_set(losses, alpha, B=B, seed=seed)}
        for alpha in MCS_LEVELS
    }
    mean_loss = losses.mean(axis=0)
    rank = {int(position): r + 1 for r, position in enumerate(np.argsort(mean_loss))}
    return pl.DataFrame(
        [
            {
                "model": label(key),
                "mean_loss": float(mean_loss[i]),
                "se_across_origins": float(
                    losses[:, i].std(ddof=1) / math.sqrt(losses.shape[0])
                ),
                "rank": rank[i],
                "in_mcs_90": key in members[0.10],
                "in_mcs_75": key in members[0.25],
            }
            for i, key in enumerate(keys)
        ]
    ).sort("rank")


###

<div style="background: linear-gradient(90deg, #012a4a, #013a63); border-left: 3px solid #48cae4; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #90e0ef; font-size: 1em; margin: 0;">🚀 <code>runner.py</code></h3> <p style="display: inline; color: #caf0f8; font-size: 0.9em; margin: 0;">· Manifes 900 run, resume, gerbang desain, eksekutor dua GPU, pilot.</p></div>

In [30]:
"""The 900-run manifest and the executor that walks it on one or two GPUs.

Three models, each at 15 origins x K in {1, 4, 8, 12} x 5 seeds at H = 24, run in
the order iTransformer, Ridge, vanilla Transformer. Runs are parallel across
devices, one worker thread per GPU pulling from a shared queue; batches are not
split. A run is complete only when its predictions, weights and metadata agree,
and resume accepts only outputs of the current code and input. The grid refuses
to start until the design digest has been frozen after the pilot.
"""

#: Arm name to the model tag in ``run_id``, in grid order.
ARM_MODEL_TAG: dict[str, str] = {"main": "itr", "ridge": "rdg", "vanilla": "vtr"}
ALL_ARMS: tuple[str, ...] = tuple(ARM_MODEL_TAG)

#: Defaults for the command-line entry point; the notebook sets its own deadline.
SESSION_BUDGET_H: float = 11.0
RESERVE_H: float = 0.5

#: Output folders that belong to one run.
RUN_FOLDERS: tuple[tuple[str, str], ...] = (("preds", ".parquet"), ("weights", ".pt"), ("meta", ".json"))


@dataclass(frozen=True, slots=True)
class RunCell:
    """One (model, origin, K, H, seed) cell of the grid."""

    arm: str
    origin_index: int
    k: int
    pred_len: int
    seed: int

    @property
    def model_tag(self) -> str:
        return ARM_MODEL_TAG[self.arm]

    @property
    def spec(self) -> RunSpec:
        return RunSpec(self.model_tag, self.origin_index, self.k, self.pred_len, self.seed)

    @property
    def run_id(self) -> str:
        return self.spec.run_id

    @property
    def tensor_key(self) -> tuple[int, int, int]:
        """All three models of a cell share windows, scaler and training sample."""
        return (self.origin_index, self.k, self.pred_len)

    def origin(self) -> Origin:
        return ORIGINS[self.origin_index - 1]

    def columns(self) -> tuple[str, ...]:
        return tuple(ladder_columns(self.k))

    def model_config(self) -> Architecture:
        if self.arm == "main":
            return ITransformerConfig(pred_len=self.pred_len)
        if self.arm == "ridge":
            return RidgeConfig(pred_len=self.pred_len, k=self.k)
        if self.arm == "vanilla":
            return VanillaConfig(pred_len=self.pred_len, k=self.k)
        raise ValueError(f"unknown arm {self.arm!r}")

    def reference_run_id(self) -> str:
        """The iTransformer run whose windows this cell must share."""
        return RunSpec(ARM_MODEL_TAG["main"], self.origin_index, self.k, self.pred_len, SEEDS[0]).run_id


def manifest(arms: tuple[str, ...] = ALL_ARMS) -> list[RunCell]:
    """Every run, in grid order: 300 per model, 900 for all three."""
    cells = [RunCell(arm, o.index, k, PRED_LEN, s)
             for arm in arms for o in ORIGINS for k in K_LADDER for s in SEEDS]
    if len({c.run_id for c in cells}) != len(cells):
        raise ValueError("manifest run ids are not unique")
    return cells


def discover_roots(working: Path = ARTIFACTS, inputs: Path = Path("/kaggle/input")) -> list[Path]:
    """The working directory, then every attached folder holding run outputs, at any depth."""
    roots = [Path(working)]
    marks = {"meta", "preds", "validation", "checkpoints"}
    if Path(inputs).exists():
        for dirpath, dirnames, _ in os.walk(inputs):
            if marks & set(dirnames):
                roots.append(Path(dirpath))
            dirnames[:] = [d for d in dirnames if d not in {*marks, "weights"}]
    seen: set[str] = set()
    return [r for r in roots if not (str(r) in seen or seen.add(str(r)))]


def completed_run_ids(roots: list[Path], code_digest: str = "") -> set[str]:
    """Run ids with predictions and a ``complete`` meta, optionally of one code vintage."""
    done: set[str] = set()
    for root in roots:
        meta_dir = Path(root) / "meta"
        if not meta_dir.is_dir():
            continue
        for meta_path in meta_dir.glob("*.json"):
            run_id = meta_path.stem
            if not (Path(root) / "preds" / f"{run_id}.parquet").exists():
                continue
            try:
                meta = json.loads(meta_path.read_text(encoding="utf-8"))
            except (json.JSONDecodeError, OSError):
                continue
            if meta.get("status") != "complete":
                continue
            if code_digest and meta.get("code_sha256") != code_digest:
                continue
            done.add(run_id)
    return done


def pending(cells: list[RunCell], roots: list[Path]) -> list[RunCell]:
    """Cells without a strictly complete run in the first root that holds one."""
    todo = []
    for cell in cells:
        candidates = [root for root in roots
                      if (root / "preds" / f"{cell.run_id}.parquet").exists()
                      or (root / "meta" / f"{cell.run_id}.json").exists()]
        if not candidates or not is_complete(cell.run_id, candidates[0], strict=True,
                                             cfg=cell.model_config(), columns=cell.columns()):
            todo.append(cell)
    return todo


def consolidate_resume_outputs(cells: list[RunCell], roots: list[Path], out_root: Path) -> int:
    """Copy strictly complete runs from attached roots into ``out_root``; return how many.

    Copies, not links, so the next saved output is self-contained. Checkpoints and
    cached validation fits are copied too, so an interrupted run resumes mid-way.
    """
    out_root = Path(out_root)
    copied = 0
    for cell in cells:
        cfg = cell.model_config()
        if is_complete(cell.run_id, out_root, strict=True, cfg=cfg, columns=cell.columns()):
            continue
        for root in roots:
            root = Path(root)
            if root == out_root or not is_complete(cell.run_id, root, strict=True, cfg=cfg,
                                                    columns=cell.columns()):
                continue
            for folder, suffix in RUN_FOLDERS:
                source = root / folder / f"{cell.run_id}{suffix}"
                destination = out_root / folder / source.name
                destination.parent.mkdir(parents=True, exist_ok=True)
                staging = destination.with_suffix(destination.suffix + ".tmp")
                shutil.copyfile(source, staging)
                staging.replace(destination)
            copied += 1
            break
    for root in roots:
        if Path(root) == out_root:
            continue
        for folder, pattern in (("checkpoints", "*.pt"), ("validation", "*.json")):
            for source in (Path(root) / folder).glob(pattern):
                destination = out_root / folder / source.name
                if not destination.exists():
                    destination.parent.mkdir(parents=True, exist_ok=True)
                    staging = destination.with_suffix(destination.suffix + ".tmp")
                    shutil.copyfile(source, staging)
                    staging.replace(destination)
    return copied


def resume_check(cells: list[RunCell], roots: list[Path], out_root: Path) -> tuple[int, int]:
    """Carry attached runs forward and refuse to continue if any were left behind.

    Returns ``(attached, available)``: complete runs of this code vintage found in
    attached inputs, and strictly complete runs now in ``out_root``.

    Raises:
        RuntimeError: If an attached run of this vintage could not be carried
            forward, so the session would silently retrain it.
    """
    wanted = {c.run_id for c in cells}
    attached = completed_run_ids([r for r in roots if Path(r) != Path(out_root)], code_sha256()) & wanted
    consolidate_resume_outputs(cells, roots, out_root)
    available = {c.run_id for c in cells} - {c.run_id for c in pending(cells, [Path(out_root)])}
    missing = sorted(attached - available)
    if missing:
        raise RuntimeError(
            f"{len(missing)} attached runs of this code vintage failed the strict check "
            f"(first: {missing[0]}). Attach the complete output (preds, weights, meta) "
            f"of the previous session instead of retraining."
        )
    return len(attached), len(available)


def design_digest(cells: list[RunCell] | None = None) -> str:
    """sha256 of the code and the manifest: what the frozen design commits to."""
    run_ids = [c.run_id for c in (cells or manifest())]
    payload = json.dumps({"code": code_sha256(), "runs": run_ids}, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def require_frozen_design(frozen: str | None, cells: list[RunCell] | None = None) -> str:
    """Return the design digest, or refuse to run the grid before and after a design change.

    Raises:
        RuntimeError: If no digest has been frozen yet, or the current one differs.
    """
    digest = design_digest(cells)
    if frozen is None:
        raise RuntimeError(
            f"the design is not frozen. Run the pilot, record {digest} as the frozen "
            f"design digest, then start the grid."
        )
    if digest != frozen:
        raise RuntimeError(
            f"design digest {digest[:12]} differs from the frozen {frozen[:12]}: the code or "
            f"the manifest changed after the freeze."
        )
    return digest


class BudgetGuard:
    """A monotonic deadline, and the decision whether another run fits before it."""

    def __init__(self, budget_h: float = SESSION_BUDGET_H, reserve_h: float = RESERVE_H,
                 *, started_at: float | None = None) -> None:
        if not np.isfinite(budget_h + reserve_h) or min(budget_h, reserve_h) < 0:
            raise ValueError("budget and reserve must be finite, nonnegative hours")
        start = time.perf_counter() if started_at is None else started_at
        self.deadline = start + (budget_h - reserve_h) * 3600.0
        self.durations: list[float] = []

    def record(self, seconds: float) -> None:
        self.durations.append(seconds)

    @property
    def mean_run_s(self) -> float:
        return sum(self.durations) / len(self.durations) if self.durations else 120.0

    @property
    def remaining_s(self) -> float:
        return self.deadline - time.perf_counter()

    def may_start(self) -> bool:
        return self.remaining_s > max(120.0, 1.5 * max(self.durations[-20:], default=120.0))


class _TensorCache:
    """A small LRU of per-(origin, K) tensors, shared by the three models of a cell."""

    def __init__(self, features: pl.DataFrame, size: int = 2) -> None:
        self.features = features
        self.size = size
        self._store: OrderedDict[tuple, OriginTensors] = OrderedDict()

    def get(self, cell: RunCell) -> OriginTensors:
        key = cell.tensor_key
        if key in self._store:
            self._store.move_to_end(key)
            return self._store[key]
        tensors = build_origin_tensors(
            self.features, cell.origin(), cell.k, seq_len=SEQ_LEN, pred_len=cell.pred_len,
            train_window_limit=TRAIN_WINDOW_LIMIT, selection_seed=SELECTION_SEED,
        )
        self._store[key] = tensors
        while len(self._store) > self.size:
            self._store.popitem(last=False)
        return tensors


@dataclass(frozen=True, slots=True)
class ExecutionSummary:
    completed: int
    skipped: int
    failed: int
    remaining: int
    wall_time_s: float
    mean_run_s: float

    def __str__(self) -> str:
        return (f"completed {self.completed}  skipped {self.skipped}  failed {self.failed}  "
                f"remaining {self.remaining}\nwall {self.wall_time_s / 3600:.2f} h  "
                f"mean run {self.mean_run_s:.1f} s")


def _run_cell(cell: RunCell, cache: _TensorCache, device: torch.device, out_root: Path,
              roots: list[Path], guard: BudgetGuard):
    """Fit one cell and write its files."""
    tensors = cache.get(cell)
    requested = cell.model_config()
    with TrainingSession(out_root, roots, guard.deadline):
        model, cfg, outcome = requested.fit(tensors, cell.spec, device=device)
    write_artifacts(model, tensors, cell.spec, cfg, outcome, device, root=out_root,
                    requested_config=requested)
    return tensors, outcome


def _check_alignment(cell: RunCell, roots: list[Path], log) -> None:
    """A comparator must be scored on the iTransformer's exact windows."""
    if cell.arm == "main":
        return
    try:
        assert_baseline_alignment(cell.run_id, cell.reference_run_id(), roots)
    except FileNotFoundError:
        log(f"  {cell.run_id}: window alignment unchecked, {cell.reference_run_id()} not on disk")


def visible_devices() -> list[torch.device]:
    if not torch.cuda.is_available():
        return [torch.device("cpu")]
    return [torch.device("cuda", i) for i in range(torch.cuda.device_count())]


def execute_parallel(cells: list[RunCell], features: pl.DataFrame, *,
                     devices: list[torch.device] | None = None, out_root: Path = ARTIFACTS,
                     roots: list[Path] | None = None, guard: BudgetGuard | None = None,
                     log=print) -> ExecutionSummary:
    """Run every pending cell, one worker per device, stopping cleanly at the deadline.

    A window misalignment between a comparator and the iTransformer stops the grid.
    """
    devices = devices or visible_devices()
    guard = guard or BudgetGuard()
    roots = list(dict.fromkeys([Path(out_root), *(roots or discover_roots(out_root))]))
    pending_ids = {c.run_id for c in pending(cells, roots)}
    queue_ = list(cells)
    cursor = 0
    completed = skipped = failed = 0
    state = threading.Lock()
    fatal: list[BaseException] = []
    started = time.perf_counter()
    log(f"workers on {[str(d) for d in devices]}")

    def take():
        nonlocal cursor
        with state:
            if fatal or cursor >= len(queue_) or not guard.may_start():
                return None
            cursor += 1
            return cursor, queue_[cursor - 1]

    def worker(device: torch.device) -> None:
        nonlocal completed, skipped, failed
        cache = _TensorCache(features)
        while (item := take()) is not None:
            position, cell = item
            if cell.run_id not in pending_ids:
                with state:
                    skipped += 1
                continue
            began = time.perf_counter()
            try:
                tensors, outcome = _run_cell(cell, cache, device, out_root, roots, guard)
            except SessionBudgetExhausted as exc:
                log(f"PAUSED: {exc}; save this output and attach it next session")
                break
            except Exception as exc:
                with state:
                    failed += 1
                log(f"[{position}/{len(queue_)}] {device} {cell.run_id} FAILED: {exc!r}")
                continue
            try:
                _check_alignment(cell, roots, log)
            except Exception as exc:
                with state:
                    fatal.append(exc)
                return
            elapsed = time.perf_counter() - began
            with state:
                guard.record(elapsed)
                completed += 1
            log(f"[{position}/{len(queue_)}] {device} {cell.run_id}  epochs={outcome.epochs_run}  "
                f"val={outcome.best_val_mse:.6f}  {elapsed:.1f}s  n_train={len(tensors.train)}")

    threads = [threading.Thread(target=worker, args=(d,), name=f"grid-{d}", daemon=True)
               for d in devices]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if fatal:
        raise RuntimeError("a comparator was scored on windows other than the iTransformer's") from fatal[0]
    if cursor < len(queue_):
        log(f"budget guard: {len(queue_) - cursor} cells unstarted; resume picks them up next session")
    return ExecutionSummary(completed=completed, skipped=skipped, failed=failed,
                            remaining=len(pending(queue_, roots)),
                            wall_time_s=time.perf_counter() - started, mean_run_s=guard.mean_run_s)


def validation_fit(tensors: OriginTensors, spec: RunSpec, cfg: Architecture, *,
                   device: torch.device, out_root: Path | None = None,
                   roots: list[Path] | None = None) -> dict:
    """Fit on the training sub-block and score on validation only, caching by identity.

    Nothing here reads a test block.
    """
    identity = json.loads(json.dumps({
        "spec": asdict(spec), "config": asdict(cfg), "code_sha256": code_sha256(),
        "input_sha256": _input_sha256()[0], "torch": str(torch.__version__),
        "device_type": device.type, "training_selection": tensors.training_selection,
    }))
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    destination = Path(out_root) / "validation" / f"{key}.json" if out_root else None
    if destination:
        for root in dict.fromkeys([Path(out_root), *(roots or [])]):
            path = root / "validation" / f"{key}.json"
            if path.exists():
                cached = json.loads(path.read_text(encoding="utf-8"))
                if cached.get("identity") == identity and np.isfinite(cached.get("val_mse", np.nan)):
                    return cached
    session = (TrainingSession(Path(out_root), list(roots or []), math.inf)
               if out_root else contextlib.nullcontext())
    with session:
        _, fitted, outcome = cfg.fit(tensors, spec, device=device)
    row = {"identity": identity, "val_mse": outcome.best_val_mse, "epochs_run": outcome.epochs_run,
           "wall_time_s": outcome.wall_time_s, "n_val": len(tensors.val),
           "config": asdict(fitted)}
    if destination:
        destination.parent.mkdir(parents=True, exist_ok=True)
        staging = destination.with_suffix(".json.tmp")
        staging.write_text(json.dumps(row, indent=2), encoding="utf-8")
        staging.replace(destination)
        (Path(out_root) / "checkpoints" / f"{spec.run_id}.pt").unlink(missing_ok=True)
    return row


@dataclass(frozen=True, slots=True)
class PilotResult:
    """Validation MSE and wall time per model and rung at the first origin."""

    rows: tuple[dict, ...]

    def mean_wall_s(self) -> dict[str, float]:
        """Mean seconds per run, per model tag."""
        out: dict[str, list[float]] = {}
        for row in self.rows:
            out.setdefault(row["model"], []).append(row["wall_time_s"])
        return {tag: float(np.mean(v)) for tag, v in out.items()}

    def __str__(self) -> str:
        lines = ["model  K   val MSE    Naive-RW   epochs  seconds"]
        lines += [f"{r['model']:5s} {r['k']:2d}  {r['val_mse']:.6f}  {r['naive_val_mse']:.6f}  "
                  f"{r['epochs_run']:6d}  {r['wall_time_s']:7.1f}" for r in self.rows]
        return "\n".join(lines)


def pilot(features: pl.DataFrame, *, origin_index: int = 1, rungs: tuple[int, ...] = K_LADDER,
          seed: int = SEEDS[0], devices: list[torch.device] | None = None,
          out_root: Path | None = None, roots: list[Path] | None = None, log=print) -> PilotResult:
    """Fit every model at every rung on one origin's validation split and time it.

    An engineering check: every loss must be finite, and the timings plan the
    grid. Nothing is chosen from these numbers.
    """
    devices = devices or visible_devices()
    tasks = [RunCell(arm, origin_index, k, PRED_LEN, seed) for arm in ALL_ARMS for k in rungs]
    free: queue.Queue = queue.Queue()
    for device in devices:
        free.put(device)
    cache_lock = threading.Lock()
    cache = _TensorCache(features, size=len(rungs))

    def run(cell: RunCell) -> dict:
        device = free.get()
        try:
            with cache_lock:
                tensors = cache.get(cell)
            spec = RunSpec(f"pilot{cell.model_tag}", origin_index, cell.k, PRED_LEN, seed)
            row = validation_fit(tensors, spec, cell.model_config(), device=device,
                                 out_root=out_root, roots=roots)
        finally:
            free.put(device)
        naive = float(np.mean((tensors.val.y - tensors.naive_rw_z) ** 2))
        if not np.isfinite(row["val_mse"]):
            raise ValueError(f"pilot {spec.run_id}: non-finite validation MSE")
        log(f"pilot {spec.run_id} on {device}: val {row['val_mse']:.6f} (Naive-RW {naive:.6f}), "
            f"{row['wall_time_s']:.1f}s")
        return {"model": cell.model_tag, "k": cell.k, "val_mse": row["val_mse"],
                "naive_val_mse": naive, "epochs_run": row["epochs_run"],
                "wall_time_s": row["wall_time_s"]}

    with ThreadPoolExecutor(max_workers=len(devices)) as pool:
        rows = tuple(pool.map(run, tasks))
    return PilotResult(rows=rows)


def session_plan(mean_wall_s: dict[str, float], cells: list[RunCell], *, devices: int,
                 session_left_h: float, usable_session_h: float,
                 weekly_left_h: float | None = None) -> str:
    """Hours and sessions the pending cells need, from the pilot's timings."""
    by_model: dict[str, int] = {}
    for cell in cells:
        by_model[cell.model_tag] = by_model.get(cell.model_tag, 0) + 1
    gpu_h = {tag: n * mean_wall_s.get(tag, float("nan")) / 3600 for tag, n in by_model.items()}
    wall_h = sum(gpu_h.values()) / max(1, devices)
    lines = [f"pending runs {by_model}",
             "GPU-hours per model " + ", ".join(f"{t} {h:.2f}" for t, h in gpu_h.items()),
             f"wall-clock on {devices} device(s): {wall_h:.2f} h; this session has {session_left_h:.2f} h left"]
    if wall_h > session_left_h:
        lines.append(f"sessions needed: {1 + math.ceil((wall_h - session_left_h) / usable_session_h)}")
    if weekly_left_h is not None:
        lines.append(f"weekly quota left {weekly_left_h:.1f} GPU-h against {wall_h * devices:.1f} needed")
    return "\n".join(lines)


def build_feature_frame(parquet: Path = DEFAULT_PARQUET) -> pl.DataFrame:
    """Load the input parquet and compute the twelve variates."""

    return build_features(usable_mask(load_bars(parquet)))






###

<div style="background: linear-gradient(90deg, #001a0d, #003317); border-left: 3px solid #52b788; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #95d5b2; font-size: 1em; margin: 0;">🖼️ <code>report.py</code></h3> <p style="display: inline; color: #b7e4c7; font-size: 0.9em; margin: 0;">· Tabel, figure, dan paper_numbers.json dari run tersimpan.</p></div>

In [31]:
"""Tables, figures and ``paper_numbers.json`` for the three-model study.

Everything is computed from saved runs (``preds/`` and ``meta/``), the input bars
and the feature frame; no table or figure is edited by hand. Dispersion is the
standard error across origins, with seed variation reported beside it. All
inference is diagnostic, because consecutive origins share training data.
"""

MODEL_TAGS: Final[tuple[str, ...]] = ("itr", "rdg", "vtr")
MODEL_NAMES: Final[dict[str, str]] = {"itr": "iTransformer", "rdg": "Ridge", "vtr": "Transformer"}
COMPARISON_KEYS: Final[tuple[ModelKey, ...]] = (
    *((tag, k) for tag in MODEL_TAGS for k in K_LADDER), NAIVE,
)
#: C1 (Ridge) and C2 (vanilla Transformer) against the iTransformer at every rung.
PAIRED_CONTRASTS: Final = tuple(
    (("itr", k), (other, k), claim)
    for other, claim in (("rdg", "C1"), ("vtr", "C2")) for k in K_LADDER
)
#: Share of training data consecutive origins have in common.
ORIGIN_OVERLAP: Final = (TRAIN_MONTHS - ORIGIN_SPACING_MONTHS) / TRAIN_MONTHS
INFERENCE_STATUS: Final = "diagnostic: consecutive origins share training data, so origins are not independent"

SPLIT_COLOUR: Final = {"train": "#1f4e79", "val": "#5b8db8", "purge": "#f4a259",
                       "test": "#c1121f", "test_alt": "#e07a1f", "oos": "#6c757d"}
MODEL_COLOUR: Final = {"itr": "#1f4e79", "rdg": "#2e7d32", "vtr": "#c1121f"}
RUNG_STYLE: Final = {1: ":", 4: "-.", 8: "-", 12: "--"}
DEFAULT_MAX_EPOCHS: Final = 30
_MISSING: Final = "---"


# -- formatting -------------------------------------------------------------------


def fmt(value: float | int | None, digits: int = 4) -> str:
    """A number for a table cell; ``---`` for missing or non-finite values."""
    if value is None:
        return _MISSING
    number = float(value)
    if not math.isfinite(number):
        return _MISSING
    if digits == 0:
        return f"{int(round(number)):,}"
    return f"{number:.{digits}f}"


def tex_escape(text: str) -> str:
    for old, new in (("_", "\\_"), ("%", "\\%"), ("&", "\\&"), ("#", "\\#")):
        text = text.replace(old, new)
    return text


def tabular(caption: str, tag: str, header: list[str], rows: list[list[str]], align: str,
            note: str = "") -> str:
    """A booktabs table with an optional note, marked as generated."""
    lines = ["% GENERATED --- do not hand-edit.", "% Regenerate: python tools/build_report.py",
             "\\begin{table}[!t]", "\\centering", "\\caption{" + caption + "}",
             "\\label{" + tag + "}", "\\begin{tabular}{" + align + "}", "\\toprule",
             " & ".join(header) + " \\\\", "\\midrule"]
    lines += [" & ".join(row) + " \\\\" for row in rows]
    lines += ["\\bottomrule", "\\end{tabular}"]
    if note:
        lines.append("\\vspace{2pt}\\par\\footnotesize " + note)
    lines.append("\\end{table}")
    return "\n".join(lines) + "\n"


def se_across(values: np.ndarray) -> float:
    """Standard error across the given values (origins)."""
    values = np.asarray(values, dtype=np.float64)
    if len(values) < 2:
        return float("nan")
    return float(values.std(ddof=1) / math.sqrt(len(values)))


def _star(p: float | None, threshold: float = 0.05) -> str:
    if p is None or not math.isfinite(float(p)):
        return ""
    return "$^{*}$" if float(p) < threshold else ""


# -- sections of paper_numbers.json -------------------------------------------------


@dataclass(frozen=True, slots=True)
class ReportInputs:
    numbers: dict
    seed_avg: pl.DataFrame
    amplification: pl.DataFrame
    rolling_pr: pl.DataFrame
    rolling_r2: pl.DataFrame
    da_summary: pl.DataFrame


def _dataset_section(bars: pl.DataFrame) -> dict:
    """Table 1 and Table 5: the data, its breaks, and the window budget per origin."""
    summary = break_summary(bars, DATA_START, DATA_END)
    return {
        "window": [DATA_START.isoformat(), DATA_END.isoformat()],
        "bars_expected": BARS_EXPECTED, "bars_actual": BARS_ACTUAL,
        "missing_bars": MISSING_BARS, "gap_blocks": GAP_BLOCKS,
        "measured": {
            "calendar_hours": summary.calendar_hours, "bars_present": summary.bars_present,
            "bars_usable": summary.bars_usable, "missing_bars": summary.missing_bars,
            "zero_volume_bars": summary.zero_volume_bars, "flat_bars": summary.flat_bars,
            "zero_trade_bars": summary.zero_trade_bars,
            "excluded_positions": summary.excluded_positions,
            "break_runs": summary.break_runs, "segments": summary.segments,
        },
        "per_origin": [
            {"origin": b.label, "train_windows": b.windows_measured,
             "closed_form": b.windows_closed_form, "closed_form_agrees": b.closed_form_agrees,
             "loss_pct": b.loss_pct, "test_block_starts": list(b.test_block_starts),
             "worst_block_starts": int(min(b.test_block_starts))}
            for b in budget_table(bars)
        ],
    }


def _keff_section(features: pl.DataFrame, table: pl.DataFrame) -> dict:
    """Table 2b: K_eff per rung across origins, ``corr(K, K_eff)`` and the gate."""
    per_rung = table.group_by("k").agg(
        pl.col("pr_raw").mean().alias("PR_raw"), pl.col("pr_raw").std().alias("PR_raw_sd"),
        pl.col("pr_window_norm").mean().alias("PR_windownorm"),
        pl.col("stable_rank_lookback").mean().alias("stable_rank"),
        pl.col("pr_lookback_ratio").mean().alias("crosslag_share"),
    ).sort("k")
    measured = gate_pr(features, k=8)
    return {"per_rung": per_rung.to_dicts(), "corr_k_keff": corr_k_keff(table),
            "gate_pr_k8": measured, "gate_floor": GATE_PR_FLOOR,
            "gate_passed": bool(measured >= GATE_PR_FLOOR), "table": table.to_dicts()}


def _architecture_section(run_ids: list[str], roots: list[Path]) -> dict:
    """Table 3: parameters, epochs and runs at the epoch cap, per model and rung."""
    rows: dict[tuple[str, int], dict] = {}
    for run_id in run_ids:
        parts = parse_run_id(run_id)
        meta = load_meta(run_id, roots)
        key = (str(parts["model"]), int(parts["k"]))
        row = rows.setdefault(key, {
            "model": key[0], "k": key[1], "n_parameters": meta.get("n_parameters"),
            "n_allocated_parameters": meta.get("n_allocated_parameters"),
            "epochs": [], "n_runs": 0, "config": meta.get("config", {}),
            "schedule": meta.get("schedule"),
        })
        row["epochs"].append(int(meta.get("epochs_run", 0)))
        row["n_runs"] += 1
    out = []
    for row in rows.values():
        epochs = np.asarray(row.pop("epochs"), dtype=np.float64)
        cap = int((row.get("schedule") or {}).get("max_epochs", DEFAULT_MAX_EPOCHS))
        row.update(epochs_mean=float(epochs.mean()), epochs_max=int(epochs.max()),
                   max_epochs=cap if row.get("schedule") else 0,
                   epochs_at_cap=int((epochs >= cap).sum()) if row.get("schedule") else 0)
        out.append(row)
    out.sort(key=lambda row: (MODEL_TAGS.index(row["model"]), row["k"]))
    return {"cells": out}


def _research_questions(seed_avg: pl.DataFrame, keff_tbl: pl.DataFrame, *, B: int, seed: int):
    """RQ1 on the iTransformer ladder, and RQ2 on the K=1 against K=8 gap by block."""
    main = seed_avg.filter((pl.col("model") == "itr") & (pl.col("pred_len") == PRED_LEN))
    origin = main.group_by("origin", "k").agg(pl.col("rel_mse").mean(), pl.col("r2_oos").mean())
    rung = origin.group_by("k").agg(
        pl.col("rel_mse").mean().alias("RelMSE"),
        (pl.col("rel_mse").std() / pl.len().sqrt()).alias("SE_across_origins"),
        pl.col("r2_oos").mean().alias("R2_oos"), pl.len().alias("n_origins"),
    ).sort("k")
    wide = {k: origin.filter(pl.col("k") == k).sort("origin")["rel_mse"].to_numpy() for k in K_LADDER}
    d48, d812 = wide[4] - wide[8], wide[8] - wide[12]
    margin = 0.25 * abs(float(d48.mean()))
    race = main.join(keff_tbl.select("origin", "k", "pr_raw"), on=["origin", "k"])
    groups = race["origin_index"].to_numpy() * 100 + race["block"].to_numpy()
    clusters = race["origin_index"].to_numpy()
    y, k, pr = (race[n].to_numpy().astype(float) for n in ("rel_mse", "k", "pr_raw"))
    t_ab, p_ab = j_test(y, k, pr, groups, clusters=clusters)
    t_ba, p_ba = j_test(y, pr, k, groups, clusters=clusters)

    amp = amplification(seed_avg)
    beta = panel_beta1(amp, B=B, seed=seed)
    stride = []
    for offset in range(5):  # every fifth origin shares no training data
        labels = [o.label for o in ORIGINS[offset::5]]
        part = amp.filter(pl.col("origin").is_in(labels))
        if part.height == len(labels) * 6 and len(labels) >= 2:
            sub = panel_beta1(part, B=B, seed=seed)
            stride.append({"origins": labels, "G": sub.n_clusters, "beta1": sub.beta1,
                           "p_diagnostic": sub.headline_p})
    try:
        fit = panel_beta1_covariate(
            amp.with_columns((pl.col("n_large") / float(BLOCK_HOURS)).alias("coverage")), B=B, seed=seed)
        covariate = {"beta1": fit.beta1, "t": fit.t_statistic, "headline_p": fit.headline_p}
    except ValueError as error:
        covariate = {"status": "not estimable", "reason": str(error)}
    _, covered = beta1_with_coverage(amp, B=B, seed=seed)
    return {
        "rq1": {"rung_effects": rung.to_dicts(), "delta_4_to_8": float(d48.mean()),
                "delta_8_to_12": float(d812.mean()), "tost_margin": margin,
                "tost": str(tost_equivalence(d812, margin)),
                "j_test_k_augmented_by_keff": {"t": t_ab, "p": p_ab},
                "j_test_keff_augmented_by_k": {"t": t_ba, "p": p_ba}},
        "rq2": {"beta1": beta.beta1, "t": beta.t_statistic, "cluster_se": beta.cluster_se,
                "p_rademacher": beta.p_rademacher, "p_webb": beta.p_webb,
                "headline_p": beta.headline_p, "G": beta.n_clusters, "N": beta.n_observations,
                "B": beta.B, "minimum_detectable_beta1": minimum_detectable_beta1(beta.within_slopes),
                "within_slopes": beta.within_slopes.tolist(), "stride5": stride,
                "origin_overlap": ORIGIN_OVERLAP,
                "coverage_covariate": covariate,
                "coverage_restricted": None if covered is None else {
                    "beta1": covered.beta1, "headline_p": covered.headline_p,
                    "G": covered.n_clusters, "N": covered.n_observations}},
    }, amp


def _main_results(seed_avg: pl.DataFrame, keys: list[ModelKey], mcs: pl.DataFrame,
                  da_summary: pl.DataFrame) -> list[dict]:
    """Table 4: RelMSE and R2_oos with the SE across origins, MCS membership and DA."""
    main = seed_avg.filter(pl.col("pred_len") == PRED_LEN)
    membership = {row["model"]: row for row in mcs.to_dicts()}
    da = {(row["model"], row["k"]): row for row in da_summary.to_dicts()}
    rows = []
    for tag, k in keys:
        if (tag, k) == NAIVE:
            continue
        cell = main.filter((pl.col("model") == tag) & (pl.col("k") == k))
        by_origin = cell.group_by("origin").agg(
            pl.col("rel_mse").mean(), pl.col("r2_oos").mean(),
            pl.col("mse_seed_std").mean().alias("seed_std"))
        r2 = by_origin["r2_oos"].to_numpy()
        name = label((tag, k))
        rows.append({
            "model": name, "model_tag": tag, "k": k,
            "rel_mse": float(by_origin["rel_mse"].mean()), "r2_oos": float(r2.mean()),
            "se_across_origins": se_across(r2), "seed_std": float(by_origin["seed_std"].mean()),
            "n_origins": by_origin.height, "n_seeds": int(cell["n_seeds"].max()),
            "in_mcs_90": bool(membership.get(name, {}).get("in_mcs_90", False)),
            "in_mcs_75": bool(membership.get(name, {}).get("in_mcs_75", False)),
            "directional": da.get((tag, k)),
        })
    return rows


def build_report(artifacts: Path, bars: pl.DataFrame, features: pl.DataFrame, *,
                 roots: list[Path] | None = None, bootstrap_b: int = 9999, seed: int = 42,
                 log=print) -> ReportInputs:
    """Every number of the study from the complete 900-run grid.

    Raises:
        ValueError: If any run of the manifest is missing, or a run trained on a
            sample other than the fixed 11,500 windows.
    """
    roots = list(roots) if roots else [Path(artifacts)]
    run_ids = [c.run_id for c in manifest()]
    missing = sorted(set(run_ids) - completed_run_ids(roots))
    if missing:
        raise ValueError(f"the report needs all {len(run_ids)} runs; {len(missing)} are missing, "
                         f"e.g. {missing[:3]}")
    metadata = [load_meta(run_id, roots) for run_id in run_ids]
    if {int(m["n_train"]) for m in metadata} != {TRAIN_WINDOW_LIMIT}:
        raise ValueError(f"every run must train on {TRAIN_WINDOW_LIMIT} windows")
    windows = evaluation_windows(run_ids, roots)
    seed_avg = seed_average(gather_grid(run_ids, roots, windows=windows))
    log(f"report: {len(run_ids)} runs, {windows.height} common forecast times, "
        f"{seed_avg.height} seed-averaged cells")

    keff_tbl = keff_table(features)
    roll_pr, roll_r2 = rolling_pr(features, k=8), rolling_ols_r2(features, k=8)
    keys, absent = available_keys(list(COMPARISON_KEYS), roots)
    panel = build_panel(keys, roots, windows=windows)
    pairs = pair_matrix(panel, B=bootstrap_b, seed=seed)
    mcs = mcs_table(panel, B=bootstrap_b, seed=seed)
    da_summary = directional_accuracy_summary(directional_accuracy_table(run_ids, roots, windows=windows))
    questions, amp = _research_questions(seed_avg, keff_tbl, B=bootstrap_b, seed=seed)
    raw_scale = (raw_scale_table(seed_avg).group_by("model", "k", "origin")
                 .agg(pl.col("rmse_raw").mean()).group_by("model", "k")
                 .agg(pl.col("rmse_raw").mean(), pl.len().alias("n_origins")).sort("model", "k"))
    architecture = _architecture_section(run_ids, roots)
    numbers = {
        "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "artifacts_root": str(artifacts),
        "input_sha256": sorted({m["input_sha256"] for m in metadata}),
        "prediction_code_sha256": sorted({m["code_sha256"] for m in metadata}),
        "analysis_code_sha256": code_sha256(),
        "runs_complete": len(run_ids), "manifest_run_ids": run_ids,
        "inference_status": INFERENCE_STATUS,
        "dataset": _dataset_section(bars),
        "efficiency": efficiency_table(features).to_dicts(),
        "keff": _keff_section(features, keff_tbl),
        "keff_rolling": {
            "window_days": 90, "descriptive_only": True,
            "pr": {"min": float(roll_pr["pr"].min()), "max": float(roll_pr["pr"].max()),
                   "mean": float(roll_pr["pr"].mean())},
            "ols_r2": {"min": float(roll_r2["r2"].min()), "max": float(roll_r2["r2"].max()),
                       "mean": float(roll_r2["r2"].mean())},
        },
        "architecture": architecture,
        "main_results": _main_results(seed_avg, keys, mcs, da_summary),
        "contrasts": [{**paired_contrast(seed_avg, left, right), "claim": claim}
                      for left, right, claim in PAIRED_CONTRASTS],
        "comparisons": {"models": [label(key) for key in keys],
                        "absent": [label(key) for key in absent], "B": bootstrap_b,
                        "p_floor": 1.0 / (1 + bootstrap_b), "pairs": pairs.to_dicts(),
                        "mcs": mcs.to_dicts()},
        "directional_accuracy": da_summary.to_dicts(),
        "raw_scale": raw_scale.to_dicts(),
        **questions,
        "training_sample": {"windows": TRAIN_WINDOW_LIMIT, "equal_across_models": True},
        "optimization": {"runs_at_epoch_cap": sum(row["epochs_at_cap"] for row in architecture["cells"])},
    }
    return ReportInputs(numbers=numbers, seed_avg=seed_avg, amplification=amp,
                        rolling_pr=roll_pr, rolling_r2=roll_r2, da_summary=da_summary)


def build_paper_numbers(artifacts: Path, bars: pl.DataFrame, features: pl.DataFrame, **kwargs) -> dict:
    return build_report(artifacts, bars, features, **kwargs).numbers


# -- tables ---------------------------------------------------------------------------


def _table1(numbers: dict) -> str:
    data = numbers["dataset"]
    measured = data["measured"]
    rows = [[row["origin"], fmt(row["train_windows"], 0), fmt(row["loss_pct"], 2),
             fmt(row["worst_block_starts"], 0), "yes" if row["closed_form_agrees"] else "no"]
            for row in data["per_origin"]]
    note = (f"BTCUSDT spot 1\\,h, Binance. Window {data['window'][0][:10]} to {data['window'][1][:10]}, "
            f"end exclusive: {data['bars_expected']:,} expected bars, {data['bars_actual']:,} present, "
            f"{data['missing_bars']} missing in {data['gap_blocks']} blocks. "
            f"{measured['zero_volume_bars']} zero-volume, {measured['flat_bars']} with $H=L$ and "
            f"{measured['zero_trade_bars']} zero-trade bars; {measured['excluded_positions']} "
            f"calendar hours are excluded in total. Windows are counted segment by segment; "
            f"the closed form is an upper bound.")
    return tabular("Data and the per-origin window budget.", "tab:dataset",
                   ["Origin", "Train windows", "Loss (\\%)", "Worst test block", "Closed form agrees"],
                   rows, "lrrrc", note)


def _table2(numbers: dict) -> str:
    rows = [[tex_escape(str(row["span"])), fmt(row["n"], 0), fmt(row["adf_stat"], 2) + _star(row["adf_p"]),
             fmt(row["hurst"], 3),
             *[fmt(row.get(f"vr_{lag}"), 3) + _star(row.get(f"vr_p_{lag}")) for lag in (2, 4, 8, 16)]]
            for row in numbers["efficiency"]]
    note = ("Log-returns. ADF tests for a unit root; Hurst by rescaled range ($H\\approx0.5$: no "
            "long memory); Lo--MacKinlay variance ratio ($VR\\approx1$: consistent with a random "
            "walk). $^{*}$ marks $p<0.05$. The evidence is reported, not read as market efficiency.")
    return tabular("Market-efficiency diagnostics, full sample and per training sub-block.",
                   "tab:efficiency", ["Span", "$n$", "ADF", "Hurst", "$VR_2$", "$VR_4$", "$VR_8$",
                                      "$VR_{16}$"], rows, "lrrrrrrr", note)


def _table2b(numbers: dict) -> str:
    keff = numbers["keff"]
    rows = [[fmt(row["k"], 0), fmt(row["PR_raw"], 3) + " $\\pm$ " + fmt(row["PR_raw_sd"], 3),
             fmt(row["PR_windownorm"], 3), fmt(row["stable_rank"], 3), fmt(row["crosslag_share"], 3)]
            for row in keff["per_rung"]]
    gate = ("above the floor" if keff["gate_passed"]
            else "below the floor, so the value is disclosed and the ladder is not re-cut")
    note = (f"Measured per origin on that origin's 21-month training sub-block; $\\pm$ is the "
            f"standard deviation across origins. $corr(K, K_{{eff}}) = {fmt(keff['corr_k_keff'], 3)}$. "
            f"Gate: PR at $K=8$ on 2018-01 to 2020-01 is {fmt(keff['gate_pr_k8'], 3)} against a floor "
            f"of {fmt(keff['gate_floor'], 1)} fixed in advance, {gate}.")
    return tabular("Effective dimensionality per rung.", "tab:keff",
                   ["$K$", "PR (raw)", "PR (window-norm.)", "Stable rank", "Cross-lag share"],
                   rows, "rrrrr", note)


def _table3(numbers: dict) -> str:
    rows = [[MODEL_NAMES.get(c["model"], c["model"]), fmt(c["k"], 0), fmt(c["n_parameters"], 0),
             fmt(c["epochs_mean"], 2), fmt(c["epochs_max"], 0), fmt(c["epochs_at_cap"], 0),
             fmt(c["n_runs"], 0)] for c in numbers["architecture"]["cells"]]
    note = ("Hyperparameters follow the official iTransformer code, with $d_{model}=128$ and "
            "$d_{ff}=256$ for the sample size, identical at every rung. The vanilla Transformer uses "
            "the same code's defaults: two encoder layers, one decoder layer, GELU and a 48-hour "
            "start token. Ridge's $\\alpha$ is the only hyperparameter selected, on validation; its "
            "fit has no epochs. At cap counts runs that reached the 30-epoch budget. A parameter "
            "count does not measure effective capacity.")
    return tabular("Models, parameters and epochs at $H=24$.", "tab:hyperparameters",
                   ["Model", "$K$", "Params", "Epochs (mean)", "Max", "At cap", "Runs"],
                   rows, "lrrrrrr", note)


def _dda(row: dict | None, variant: str) -> str:
    if not row:
        return _MISSING
    return f"{100 * row[f'dda_{variant}']:+.1f} ({row[f'wins_{variant}']}/{row['n_origins']})"


def _table4(numbers: dict) -> str:
    rows = []
    for row in numbers["main_results"]:
        mcs = "90\\%, 75\\%" if row["in_mcs_75"] else "90\\%" if row["in_mcs_90"] else _MISSING
        rows.append([tex_escape(row["model"]), fmt(row["rel_mse"], 4),
                     fmt(row["r2_oos"], 4) + " $\\pm$ " + fmt(row["se_across_origins"], 4),
                     fmt(row["seed_std"], 6), mcs,
                     *[_dda(row["directional"], v) for v in ("1h", "24h", "cum")]])
    note = ("Common forecast times. Block RelMSE from seed-averaged squared error, equal block and "
            "origin weights; $\\pm$ is the SE across 15 origins. $\\Delta DA$ is directional accuracy "
            "minus each origin's majority-sign rate, in percentage points, with the origins that "
            "beat it; the rate uses test-period frequencies, so the comparison is conservative. "
            "Ridge is deterministic (seed std 0). All inference is diagnostic.")
    return tabular("Main results at $H=24$ across fifteen origins.", "tab:main",
                   ["Model", "RelMSE", "$R^2_{oos}$", "Seed std", "In MCS", "$\\Delta DA_{1h}$",
                    "$\\Delta DA_{24h}$", "$\\Delta DA_{cum}$"], rows, "lrrrcrrr", note)


def _table5(numbers: dict) -> str:
    rows = [[row["origin"], *[fmt(n, 0) for n in row["test_block_starts"]],
             fmt(100 * min(row["test_block_starts"]) / BLOCK_HOURS, 1)]
            for row in numbers["dataset"]["per_origin"]]
    note = ("Forecast origins surviving in each 30-day block, out of 720. An origin is lost only when "
            "a gap falls inside its 120-hour window. Survival depends on future gaps, so block "
            "coverage enters RQ2 as a covariate.")
    return tabular("Surviving forecast origins per test block.", "tab:coverage",
                   ["Origin", "B1", "B2", "B3", "B4", "B5", "B6", "Min cover (\\%)"],
                   rows, "lrrrrrrr", note)


def _table6(numbers: dict) -> str:
    rows = [[tex_escape(row["left"]), tex_escape(row["right"]), fmt(row["t_cluster"], 3),
             fmt(row["p_raw"], 4), fmt(row["p_romano_wolf"], 4), row.get("family", _MISSING),
             fmt(row.get("p_romano_wolf_family"), 4), fmt(row["T_min"], 0)]
            for row in numbers["comparisons"]["pairs"]]
    note = ("Pairwise forecast-loss contrasts on common targets; positive $t$ means the left model is "
            "worse. Romano--Wolf adjusts over all pairs and within each family (ladder, cross-model, "
            "vs-naive). Origins share training data, so these are diagnostics, not confirmatory tests.")
    return tabular("Pairwise forecast-loss diagnostics.", "tab:dm",
                   ["Left", "Right", "$t$", "$p_{raw}$", "$p_{RW}$", "Family", "$p_{RW}^{fam}$",
                    "$T_{min}$"], rows, "llrrrlrr", note)


_DIGITS: Final = {"1": "One", "2": "Two", "4": "Four", "8": "Eight"}


def render_tables(numbers: dict, out_dir: Path) -> list[Path]:
    """Write Tables 1-6 and the manuscript macros."""
    out_dir.mkdir(parents=True, exist_ok=True)
    builders = {"table1_dataset.tex": _table1, "table2_efficiency.tex": _table2,
                "table2b_keff.tex": _table2b, "table3_architecture.tex": _table3,
                "table4_main.tex": _table4, "table5_coverage.tex": _table5, "table6_dm.tex": _table6}
    written = []
    for name, builder in builders.items():
        path = out_dir / name
        path.write_text(builder(numbers), encoding="utf-8")
        written.append(path)
    main = {(row["model_tag"], row["k"]): row for row in numbers["main_results"]}
    macros = {"StudyRuns": fmt(numbers["runs_complete"], 0), "StudyOrigins": fmt(numbers["rq2"]["G"], 0),
              **{f"{tag.upper()}K{k}Rtwo": fmt(main[tag, k]["r2_oos"], 6)
                 for tag in MODEL_TAGS for k in K_LADDER},
              "BetaSlope": fmt(numbers["rq2"]["beta1"], 6), "BetaSE": fmt(numbers["rq2"]["cluster_se"], 6)}
    lines = ["% Generated from paper_numbers.json."]
    for name, value in macros.items():
        clean = "".join(_DIGITS.get(ch, ch) for ch in name)
        lines.append("\\newcommand{\\" + clean + "}{" + value + "}")
    (out_dir / "manuscript_numbers.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
    return written


# -- figures --------------------------------------------------------------------------


def _pyplot():
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    return plt


def _save(fig, out_dir: Path, stem: str) -> list[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for suffix in ("pdf", "png"):
        path = out_dir / f"{stem}.{suffix}"
        fig.savefig(path, bbox_inches="tight", dpi=200)
        paths.append(path)
    return paths


def _as_datetime(ms: np.ndarray) -> np.ndarray:
    return np.asarray(ms, dtype="int64").astype("datetime64[ms]")


def _figure1(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """The walk-forward design: every origin, then one origin resolved."""
    plt = _pyplot()
    fig, (top, low) = plt.subplots(2, 1, figsize=(7.6, 6.8), gridspec_kw={"height_ratios": [2.0, 0.8]})

    def year(moment) -> float:
        return moment.year + (moment.timetuple().tm_yday - 1) / 365.25

    for row, origin in enumerate(ORIGINS):
        y = len(ORIGINS) - row
        top.barh(y, year(origin.train_sub_end) - year(origin.train_start), left=year(origin.train_start),
                 height=0.62, color=SPLIT_COLOUR["train"])
        top.barh(y, year(origin.val_end) - year(origin.val_start), left=year(origin.val_start),
                 height=0.62, color=SPLIT_COLOUR["val"])
        for b, start, end in origin.blocks():
            top.barh(y, year(end) - year(start), left=year(start), height=0.62,
                     color=SPLIT_COLOUR["test"] if b % 2 else SPLIT_COLOUR["test_alt"],
                     edgecolor="#ffffff", lw=0.4)
    top.set_yticks(range(1, len(ORIGINS) + 1))
    top.set_yticklabels([o.label for o in reversed(ORIGINS)], fontsize=7.2)
    top.set_xlabel("calendar time (UTC)", fontsize=8.5)
    top.set_title(f"Fifteen origins, rolling {TRAIN_MONTHS}-month window, {ORIGIN_SPACING_MONTHS}-month "
                  f"spacing: consecutive origins share {100 * ORIGIN_OVERLAP:.1f}% of their training data",
                  fontsize=9)
    top.grid(axis="x", alpha=0.25, lw=0.5)
    origin = ORIGINS[0]

    def day(moment) -> float:
        return (moment - origin.train_start).total_seconds() / 86400.0

    low.barh(1.0, day(origin.train_sub_end) - day(origin.train_start), left=0, height=0.5,
             color=SPLIT_COLOUR["train"])
    low.barh(1.0, day(origin.val_end) - day(origin.val_start), left=day(origin.val_start), height=0.5,
             color=SPLIT_COLOUR["val"])
    for b, start, end in origin.blocks():
        low.barh(1.0, day(end) - day(start), left=day(start), height=0.5,
                 color=SPLIT_COLOUR["test"] if b % 2 else SPLIT_COLOUR["test_alt"], edgecolor="#ffffff")
        low.text((day(start) + day(end)) / 2, 1.0, str(b), ha="center", va="center", fontsize=6.5,
                 color="#ffffff")
    for boundary in (origin.train_sub_end, origin.val_end):
        low.barh(1.0, 22, left=day(boundary) - 11, height=0.62, color=SPLIT_COLOUR["purge"],
                 hatch="////", edgecolor="#5a2d00", lw=0.6)
    low.set_yticks([])
    low.set_xlabel(f"days since the training start, origin {origin.label}", fontsize=8.5)
    low.set_title("Train (21 months), purge, validation (3 months), purge, six 30-day test blocks; "
                  "the purge is drawn 22x wider than its 24 hours", fontsize=8.5)
    handles = [plt.Line2D([], [], lw=6, color=SPLIT_COLOUR[key], label=text) for key, text in (
        ("train", "training sub-block (scaler fitted here)"), ("val", "validation"),
        ("purge", "24-hour purge at both boundaries"), ("test", "test blocks 1-6"))]
    low.legend(handles=handles, fontsize=7, frameon=False, ncol=2, loc="lower center",
               bbox_to_anchor=(0.5, -1.0))
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure1_walkforward")
    plt.close(fig)
    return paths


def _figure2(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """The three models side by side: what a token is and how the forecast leaves."""
    plt = _pyplot()
    columns = [
        ("iTransformer", "#1f4e79", ["window (B, 96, K)", "one token per variate:\nLinear(96 -> 128)",
                                     "encoder x2 over K tokens", "Linear(128 -> 24) per token",
                                     "read channel r"]),
        ("Transformer", "#c1121f", ["window (B, 96, K)", "one token per hour:\nConv1d(K -> 128) + position",
                                    "encoder x2 over 96 tokens;\ndecoder x1: last 48 h + 24 zeros",
                                    "Linear(128 -> K) per hour", "read channel r"]),
        ("Ridge", "#2e7d32", ["window (B, 96, K)", "flatten to 96K values", "one linear map with\nL2 penalty",
                              "24 outputs", "channel r only"]),
    ]
    fig, ax = plt.subplots(figsize=(7.6, 4.6))
    for col, (title, colour, steps) in enumerate(columns):
        x = 0.03 + col * 0.33
        ax.text(x + 0.14, 0.97, title, ha="center", va="top", fontsize=9.5, fontweight="bold", color=colour)
        for row, text in enumerate(steps):
            y = 0.80 - row * 0.17
            ax.add_patch(plt.Rectangle((x, y), 0.28, 0.12, facecolor=colour, alpha=0.12 + 0.12 * (row % 2),
                                       edgecolor=colour, lw=0.8))
            ax.text(x + 0.14, y + 0.06, text, ha="center", va="center", fontsize=7.0)
    ax.text(0.5, -0.02, "All three read the same windows and are scored on the target channel r only.",
            ha="center", va="top", fontsize=7.5)
    ax.set_xlim(0, 1.02)
    ax.set_ylim(-0.06, 1.0)
    ax.axis("off")
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure2_architecture")
    plt.close(fig)
    return paths


def _figure2b(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """Rolling participation ratio and in-window OLS R^2, descriptive only."""
    plt = _pyplot()
    fig, axes = plt.subplots(2, 1, figsize=(7.0, 4.6), sharex=True)
    for axis, frame, column, colour, ylabel in (
            (axes[0], inputs.rolling_pr, "pr", "#22577a", "PR at $K=8$"),
            (axes[1], inputs.rolling_r2, "r2", "#c1121f", "in-window $R^2$")):
        axis.plot(_as_datetime(frame["window_end_ms"].to_numpy()), frame[column].to_numpy(), lw=1.0,
                  color=colour)
        axis.set_ylabel(ylabel)
        axis.grid(alpha=0.25, lw=0.5)
    axes[1].set_xlabel("window end (UTC)")
    fig.suptitle("90-day rolling participation ratio and OLS fit (descriptive only)", fontsize=10)
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure2b_rolling")
    plt.close(fig)
    return paths


def _figure3(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """RQ2: the K=1 against K=8 gap by block, one line per origin, with the fit and MDE."""
    plt = _pyplot()
    amp = inputs.amplification
    rq2 = inputs.numbers["rq2"]
    fig, ax = plt.subplots(figsize=(7.0, 4.2))
    for _, part in amp.group_by(["origin"], maintain_order=True):
        part = part.sort("block")
        ax.plot(part["block"].to_numpy(), part["A"].to_numpy(), lw=0.8, alpha=0.55, marker="o", ms=2.5,
                color="#4a6fa5")
    blocks = np.arange(1, 7, dtype=float)
    intercept = float(amp["A"].mean()) - rq2["beta1"] * blocks.mean()
    ax.plot(blocks, intercept + rq2["beta1"] * blocks, lw=2.4, color="#c1121f",
            label=f"fitted $\\beta_1$ = {rq2['beta1']:+.6f}")
    mde = rq2["minimum_detectable_beta1"]
    ax.plot(blocks, intercept + mde * blocks, lw=1.6, ls="--", color="#333333",
            label=f"minimum detectable slope = {mde:+.6f}")
    ax.axhline(0.0, lw=0.8, color="#888888")
    ax.set_xlabel("test block $b$ (30 days each)")
    ax.set_ylabel("$A(i,b) = (MSE_{K1} - MSE_{K8}) / MSE_{K1}$")
    ax.set_title("The multivariate gap against model age, one line per origin", fontsize=10)
    ax.legend(fontsize=8, frameon=False)
    ax.grid(alpha=0.25, lw=0.5)
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure3_gap_by_age")
    plt.close(fig)
    return paths


def _figure4(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """RelMSE per block for every model and rung; Naive-RW is the dashed line at 1."""
    plt = _pyplot()
    main = inputs.seed_avg.filter(pl.col("pred_len") == PRED_LEN)
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    for tag in MODEL_TAGS:
        for k in K_LADDER:
            cell = main.filter((pl.col("model") == tag) & (pl.col("k") == k))
            if cell.height == 0:
                continue
            by_block = cell.group_by("block").agg(pl.col("rel_mse").mean()).sort("block")
            ax.plot(by_block["block"].to_numpy(), by_block["rel_mse"].to_numpy(), marker="o", ms=3,
                    lw=1.1, color=MODEL_COLOUR[tag], ls=RUNG_STYLE[k], label=f"{MODEL_NAMES[tag]} K={k}")
    ax.axhline(1.0, lw=1.2, color="#000000", ls="--", label="Naive-RW")
    ax.set_xlabel("test block $b$ (30 days each)")
    ax.set_ylabel("RelMSE (lower is better)")
    ax.set_title("RelMSE per block; above the dashed line loses to Naive-RW", fontsize=10)
    ax.grid(alpha=0.25, lw=0.5)
    ax.legend(fontsize=6.5, ncol=2, frameon=False, loc="center left", bbox_to_anchor=(1.01, 0.5))
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure4_relmse")
    plt.close(fig)
    return paths


def _figure5(inputs: ReportInputs, out_dir: Path) -> list[Path]:
    """Directional accuracy against the majority-sign baseline, three panels by variant."""
    plt = _pyplot()
    summary = inputs.da_summary
    fig, axes = plt.subplots(1, 3, figsize=(9.0, 3.4), sharey=True)
    for axis, variant, title in zip(axes, ("1h", "24h", "cum"), ("first hour", "24th hour", "24-hour sum")):
        for tag in MODEL_TAGS:
            part = summary.filter(pl.col("model") == tag).sort("k")
            if part.height == 0:
                continue
            axis.errorbar(part["k"].to_numpy(), 100 * part[f"dda_{variant}"].to_numpy(),
                          yerr=100 * part[f"dda_{variant}_se"].to_numpy(), marker="o", ms=3.5, lw=1.2,
                          capsize=2.5, color=MODEL_COLOUR[tag], label=MODEL_NAMES[tag])
        axis.axhline(0.0, lw=1.0, color="#000000", ls="--")
        axis.set_xticks(list(K_LADDER))
        axis.set_xlabel("$K$")
        axis.set_title(f"DA, {title}", fontsize=9)
        axis.grid(alpha=0.25, lw=0.5)
    axes[0].set_ylabel("DA minus majority rate (pp)")
    axes[-1].legend(fontsize=7.5, frameon=False)
    fig.suptitle("Directional accuracy above the per-origin majority-sign rate; bars are SE across origins",
                 fontsize=9.5)
    fig.tight_layout()
    paths = _save(fig, out_dir, "figure5_directional")
    plt.close(fig)
    return paths


def render_figures(inputs: ReportInputs, out_dir: Path, log=print) -> list[Path]:
    """Write Figures 1, 2, 2b, 3, 4 and 5 as PDF and PNG."""
    out_dir.mkdir(parents=True, exist_ok=True)
    written: list[Path] = []
    for builder in (_figure1, _figure2, _figure2b, _figure3, _figure4, _figure5):
        written.extend(builder(inputs, out_dir))
    log(f"report: {len(written)} figure files")
    return written


###

<div style="background: linear-gradient(90deg, #1b0033, #2d0052); border-left: 3px solid #bf5af2; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #e0aaff; font-size: 1em; margin: 0;">🔏 Provenance kode</h3> <p style="display: inline; color: #cbb2e8; font-size: 0.9em; margin: 0;">· Digest paket yang dipin, sumber tiap algoritma, dan cek bahwa semua sel definisi sudah berjalan.</p></div>

In [32]:
# There is no git repository on Kaggle, so the package digest is pinned at export;
# it equals code_sha256() of the same source in a checkout.
CODE_SHA256_OVERRIDE = "5a23e2725b00005c0870e7b3e4fc69f94125277e9492ff2d660db2208c207116"

_sentinels = ("ORIGINS", "__all__", "build_segments", "count_windows", "budget_table",
              "build_features", "efficiency_table", "build_origin_tensors", "keff_table",
              "load_upstream", "ITransformerConfig", "code_sha256", "RidgeConfig",
              "seed_average", "pair_matrix", "manifest", "build_report")
_missing = [name for name in _sentinels if name not in globals()]
assert not _missing, f"definition cells have not run: {_missing}; run the cells above in order"
assert "itransformer_btc" not in sys.modules, "an installed itransformer_btc was imported"

print(f"code_sha256 {code_sha256()}")
_status = {"copied": "copied from the official repository", "library": "imported and called",
           "own": "written here from the published method"}
print(f"\nsource of each component ({len(SOURCE_PROVENANCE)})")
for _prov in SOURCE_PROVENANCE:
    print(f"\n{_prov.component}\n  in         {_prov.module}\n"
          f"  status     {_prov.status} ({_status[_prov.status]})\n  reference  {_prov.reference}")
    if _prov.repo:
        print(f"  code       {_prov.repo} ({_prov.licence}, accessed {_prov.accessed})")
    if _prov.adapted:
        print(f"  adapted    {_prov.adapted}")


code_sha256 5a23e2725b00005c0870e7b3e4fc69f94125277e9492ff2d660db2208c207116

source of each component (15)

ITransformerForecaster (Model in model/iTransformer.py)
  in         model.py
  status     copied (copied from the official repository)
  reference  Y. Liu, T. Hu, H. Zhang, H. Wu, S. Wang, L. Ma, and M. Long, "iTransformer: Inverted transformers are effective for time series forecasting," in Proc. 12th Int. Conf. Learn. Represent. (ICLR), 2024. arXiv:2310.06625.
  code       https://github.com/thuml/iTransformer (MIT, accessed 2026-09-23)
  adapted    Copied unchanged at commit c2426e68ca13f74aaec08045c5c724d8ad328124. The adapter passes x_mark=None, reads the target channel, and uses d_model 128 and d_ff 256 for the sample size; use_norm stays on.

VanillaForecaster (Model in model/Transformer.py)
  in         model.py
  status     copied (copied from the official repository)
  reference  A. Vaswani, N. Shazeer, N. Parmar, J. Uszkoreit, L. Jones, A. N. Gomez, L. Kaiser, and I.

##

<a id="section-07"></a>

<div style="background: linear-gradient(135deg, #002200, #003300); border-left: 4px solid #7ae582; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #95d5b2; margin: 0 0 4px; font-size: 1.35em;">🛠️ 07 · Pemeriksaan sebelum training</h2>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.95em;">Invarian skala use_norm, overfit satu batch untuk kedua Transformer, jumlah parameter, dan Naive-RW per origin.</p>
</div>

<div style="background: #0e0e12; border-left: 3px solid #6c757d; border-radius: 0 6px 6px 0; padding: 4px 14px; font-size: 0.82em; color: #cfcfcf;"><span style="color: #ffd166; font-weight: 600;">MENULIS</span> <code>artifacts/naive_rw_by_origin.parquet</code></div>

In [33]:
device = torch.device("cpu") if ANALYSIS_ONLY else visible_devices()[0]
print(f"device {device}")

set_seed(42)
probe = ITransformerConfig().build().to(device).eval()
base, scaled = scale_invariance_check(probe, torch.randn(64, SEQ_LEN, 8, device=device),
                                      torch.randn(64, PRED_LEN, device=device), c=100.0)
print(f"use_norm invariance: MSE(x) {base:.8f}, MSE(100x)/100^2 {scaled:.8f}")
assert abs(base - scaled) / base < 1e-3, "use_norm inactive: MSE(c x)/c^2 differs from MSE(x)"

for tag, cfg in (("itr", ITransformerConfig(dropout=0.0)), ("vtr", VanillaConfig(k=8, dropout=0.0))):
    set_seed(42)
    plumb = cfg.build().to(device).train()
    # Drawn on the CPU so every device fits the same batch; a CUDA draw differs.
    probe = torch.Generator().manual_seed(42)
    xs = torch.randn(8, SEQ_LEN, 8, generator=probe).to(device)
    ys = torch.randn(8, PRED_LEN, generator=probe).to(device)
    opt = torch.optim.Adam(plumb.parameters(), lr=1e-3)
    for step in range(300):
        opt.zero_grad(set_to_none=True)
        loss = torch.nn.functional.mse_loss(plumb.forecast_target(xs), ys)
        start = loss.item() if step == 0 else start
        loss.backward()
        opt.step()
    print(f"{tag} one-batch overfit (dropout 0, 300 steps): {start:.2e} -> {loss.item():.2e}")
    # Relative, not absolute: how fast the last digits fall varies with device and
    # batch, but a broken path (no gradient, wrong target) cannot fall 100-fold.
    assert loss.item() < 1e-2 * start, f"{tag} cannot overfit one batch; the training path is broken"

print(pl.DataFrame([{"K": k, "itr": ITransformerConfig().build().n_parameters(),
                     "vtr": VanillaConfig(k=k).build().n_parameters(),
                     "rdg": RidgeConfig(k=k).build().n_parameters()} for k in K_LADDER]))

naive = pl.DataFrame([
    {"origin": o.label, "mu_g": float(t.scaler.mean[0]), "sigma_g": float(t.scaler.std[0]),
     "mu_over_sigma": t.scaler.target_mu_over_sigma, "naive_rw_z": t.naive_rw_z, "n_train": len(t.train)}
    for o in ORIGINS
    for t in [build_origin_tensors(features, o, 1, train_window_limit=TRAIN_WINDOW_LIMIT,
                                   selection_seed=SELECTION_SEED)]
])
print(naive)
naive.write_parquet(ARTIFACTS / "naive_rw_by_origin.parquet")


device cuda:0
use_norm invariance: MSE(x) 1.42240493, MSE(100x)/100^2 1.42240130
itr one-batch overfit (dropout 0, 300 steps): 1.23e+00 -> 1.40e-14
vtr one-batch overfit (dropout 0, 300 steps): 9.39e-01 -> 9.65e-05
shape: (4, 4)
┌─────┬────────┬────────┬───────┐
│ K   ┆ itr    ┆ vtr    ┆ rdg   │
│ --- ┆ ---    ┆ ---    ┆ ---   │
│ i64 ┆ i64    ┆ i64    ┆ i64   │
╞═════╪════════╪════════╪═══════╡
│ 1   ┆ 280728 ┆ 466177 ┆ 2328  │
│ 4   ┆ 280728 ┆ 468868 ┆ 9240  │
│ 8   ┆ 280728 ┆ 472456 ┆ 18456 │
│ 12  ┆ 280728 ┆ 476044 ┆ 27672 │
└─────┴────────┴────────┴───────┘
shape: (15, 6)
┌─────────┬───────────┬──────────┬───────────────┬────────────┬─────────┐
│ origin  ┆ mu_g      ┆ sigma_g  ┆ mu_over_sigma ┆ naive_rw_z ┆ n_train │
│ ---     ┆ ---       ┆ ---      ┆ ---           ┆ ---        ┆ ---     │
│ str     ┆ f64       ┆ f64      ┆ f64           ┆ f64        ┆ i64     │
╞═════════╪═══════════╪══════════╪═══════════════╪════════════╪═════════╡
│ 2020-01 ┆ -0.000043 ┆ 0.009151 ┆ -0.00475   

##

<a id="section-08"></a>

<div style="background: linear-gradient(135deg, #2b0a00, #3d1000); border-left: 4px solid #ff7b54; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #ffb4a2; margin: 0 0 4px; font-size: 1.35em;">🛡️ 08 · Validasi pilot dan rencana sesi</h2>
  <p style="color: #ffd8c2; margin: 0; font-size: 0.95em;">Tiga model × empat K pada validation origin pertama, satu run per GPU: cek teknis dan waktu, tanpa memilih apa pun; lalu rencana sesi dan digest desain untuk dibekukan.</p>
</div>

<div style="background: #0e0e12; border-left: 3px solid #6c757d; border-radius: 0 6px 6px 0; padding: 4px 14px; font-size: 0.82em; color: #cfcfcf;"><span style="color: #ffd166; font-weight: 600;">MENULIS</span> <code>artifacts/validation/*.json</code> <code>artifacts/pilot.json</code></div>

In [34]:
roots = discover_roots(ARTIFACTS)
SESSION_GUARD = BudgetGuard(
    0.0 if ANALYSIS_ONLY else min(float(WEEKLY_GPU_HOURS_REMAINING),
                                 max(0.0, SESSION_LIMIT_H - SESSION_ALREADY_USED_H)),
    SAVE_RESERVE_H, started_at=SESSION_T0)
DEVICES = [torch.device("cpu")] if ANALYSIS_ONLY else visible_devices()
print(f"devices {[str(d) for d in DEVICES]}; "
      f"{max(0.0, SESSION_GUARD.remaining_s) / 3600:.2f} h before the save reserve")

if ANALYSIS_ONLY:
    print("pilot skipped: rendering saved results only")
else:
    PILOT = pilot(features, devices=DEVICES, out_root=ARTIFACTS, roots=roots,
                  log=lambda msg: print(msg, flush=True))
    print(PILOT)
    (ARTIFACTS / "pilot.json").write_text(json.dumps(
        {"rows": list(PILOT.rows), "mean_wall_s": PILOT.mean_wall_s(),
         "design_digest": design_digest()}, indent=2), encoding="utf-8")
    print()
    print(session_plan(PILOT.mean_wall_s(), pending(manifest(), roots), devices=len(DEVICES),
                       session_left_h=max(0.0, SESSION_GUARD.remaining_s) / 3600,
                       usable_session_h=SESSION_LIMIT_H - SAVE_RESERVE_H,
                       weekly_left_h=float(WEEKLY_GPU_HOURS_REMAINING)))
    print(f"\ndesign digest {design_digest()}")


devices ['cuda:0', 'cuda:1']; 10.73 h before the save reserve
pilot pilotitr_o01_K04_H024_s42 on cuda:1: val 0.469845 (Naive-RW 0.456763), 36.2s
pilot pilotitr_o01_K01_H024_s42 on cuda:0: val 0.470210 (Naive-RW 0.456763), 45.5s
pilot pilotitr_o01_K08_H024_s42 on cuda:1: val 0.469007 (Naive-RW 0.456763), 36.7s
pilot pilotrdg_o01_K01_H024_s42 on cuda:1: val 0.456369 (Naive-RW 0.456763), 0.4s
pilot pilotrdg_o01_K04_H024_s42 on cuda:1: val 0.456075 (Naive-RW 0.456763), 2.0s
pilot pilotrdg_o01_K08_H024_s42 on cuda:1: val 0.455690 (Naive-RW 0.456763), 3.0s
pilot pilotrdg_o01_K12_H024_s42 on cuda:1: val 0.456121 (Naive-RW 0.456763), 4.6s
pilot pilotitr_o01_K12_H024_s42 on cuda:0: val 0.469603 (Naive-RW 0.456763), 42.3s
pilot pilotvtr_o01_K01_H024_s42 on cuda:1: val 0.457313 (Naive-RW 0.456763), 53.2s
pilot pilotvtr_o01_K04_H024_s42 on cuda:0: val 0.457913 (Naive-RW 0.456763), 52.7s
pilot pilotvtr_o01_K08_H024_s42 on cuda:1: val 0.458553 (Naive-RW 0.456763), 40.5s
pilot pilotvtr_o01_K12_H024_s

##

<a id="section-09"></a>

<div style="background: linear-gradient(135deg, #001233, #001845); border-left: 4px solid #4cc9f0; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #8ecae6; margin: 0 0 4px; font-size: 1.35em;">🚀 09 · Training grid walk-forward</h2>
  <p style="color: #a9d6e5; margin: 0; font-size: 0.95em;">900 run, satu worker per GPU, resume otomatis dari output sesi sebelumnya; berjalan hanya setelah digest desain dibekukan di sel ini.</p>
</div>

<div style="background: #0e0e12; border-left: 3px solid #6c757d; border-radius: 0 6px 6px 0; padding: 4px 14px; font-size: 0.82em; color: #cfcfcf;"><span style="color: #ffd166; font-weight: 600;">MENULIS</span> <code>artifacts/preds/*.parquet</code> <code>artifacts/meta/*.json</code> <code>artifacts/weights/*.pt</code> <code>artifacts/checkpoints/*.pt</code> <code>artifacts/session_status.json</code> &nbsp;·&nbsp; <span style="color: #9ec5fe; font-weight: 600;">MEMBACA</span> <code>data/raw/BTCUSDT_1h.parquet</code></div>

In [35]:
DESIGN_FREEZE_SHA256 = None  # the design digest printed by the pilot, once the design is frozen

for _name in ("probe", "plumb", "xs", "ys", "opt", "loss"):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

ALL = manifest()
roots = discover_roots(ARTIFACTS)
attached, available = resume_check(ALL, roots, ARTIFACTS)
print(f"manifest {len(ALL)} runs; attached from earlier sessions {attached}; complete here {available}")

try:
    FROZEN_DESIGN = require_frozen_design(DESIGN_FREEZE_SHA256, ALL)
except RuntimeError as exc:
    FROZEN_DESIGN = None
    print(f"GRID NOT STARTED: {exc}")

summary = None
if FROZEN_DESIGN and not ANALYSIS_ONLY:
    summary = execute_parallel(pending(ALL, roots), features, devices=DEVICES, out_root=ARTIFACTS,
                               roots=roots, guard=SESSION_GUARD,
                               log=lambda msg: print(msg, flush=True))
    print(summary)

left = pending(ALL, [ARTIFACTS])
GRID_COMPLETE = FROZEN_DESIGN is not None and not left
ANALYSIS_READY = GRID_COMPLETE and (ANALYSIS_ONLY or SESSION_GUARD.remaining_s > 3600)
status = {
    "code_sha256": code_sha256(), "input_sha256": _input_sha256()[0],
    "design_digest": design_digest(ALL), "design_frozen": FROZEN_DESIGN is not None,
    "manifest_runs": len(ALL), "grid_complete": GRID_COMPLETE, "analysis_ready": ANALYSIS_READY,
    "pending_run_ids": [c.run_id for c in left],
    "partial_checkpoints": sorted(p.name for p in (ARTIFACTS / "checkpoints").glob("*.pt")),
    "elapsed_session_h": (time.perf_counter() - SESSION_T0) / 3600 + SESSION_ALREADY_USED_H,
    "weekly_quota_entered_h": WEEKLY_GPU_HOURS_REMAINING,
    "summary": asdict(summary) if summary else None,
}
(ARTIFACTS / "session_status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
print(f"grid complete: {GRID_COMPLETE} ({len(left)} pending); analysis ready: {ANALYSIS_READY}")
if not GRID_COMPLETE and FROZEN_DESIGN:
    print("Save Version, attach this output to the next session, update the quota, then Run All.")
if GRID_COMPLETE and not ANALYSIS_READY:
    print("Less than an hour left: set ANALYSIS_ONLY = True and render in a CPU session.")


manifest 900 runs; attached from earlier sessions 0; complete here 0
GRID NOT STARTED: the design is not frozen. Run the pilot, record 9f2435aefac51199e6906f68ac7c3952d9bcc1638fe39873cb04f01c0b1ad7db as the frozen design digest, then start the grid.
grid complete: False (900 pending); analysis ready: False


##

<a id="section-10"></a>

<div style="background: linear-gradient(135deg, #03071e, #370617); border-left: 4px solid #e94560; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #f5a623; margin: 0 0 4px; font-size: 1.35em;">📈 10 · Evaluasi model dan research questions</h2>
  <p style="color: #ffd6a5; margin: 0; font-size: 0.95em;">C1 (iTransformer vs Ridge), C2 (iTransformer vs Transformer), RQ1, RQ2, dan akurasi arah dari prediksi tersimpan. Semua inferensi bersifat diagnostik: origin berbagi data latih.</p>
</div>

###

<div style="background: linear-gradient(90deg, #03071e, #370617); border-left: 3px solid #e94560; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #f5a623; font-size: 1em; margin: 0;">📋 Hasil utama</h3> <p style="display: inline; color: #ffd6a5; font-size: 0.9em; margin: 0;">· RelMSE dan R²_oos terhadap Naive-RW per model dan K, SE antar-origin, keanggotaan MCS.</p></div>

<div style="background: #0e0e12; border-left: 3px solid #6c757d; border-radius: 0 6px 6px 0; padding: 4px 14px; font-size: 0.82em; color: #cfcfcf;"><span style="color: #9ec5fe; font-weight: 600;">MEMBACA</span> <code>artifacts/preds/*.parquet</code> <code>artifacts/meta/*.json</code></div>

In [36]:
if not ANALYSIS_READY:
    print("main results: skipped until the 900-run grid is complete and the session has time to analyse it.")
else:
    REPORT = build_report(ARTIFACTS, bars, features, roots=discover_roots(ARTIFACTS),
                          bootstrap_b=9_999, seed=42, log=lambda msg: print(msg, flush=True))
    NUMBERS = REPORT.numbers
    print(NUMBERS["inference_status"])
    print(pl.DataFrame([
        {"model": r["model"], "RelMSE": r["rel_mse"], "R2_oos": r["r2_oos"], "SE": r["se_across_origins"],
         "seed_std": r["seed_std"], "origins": r["n_origins"], "MCS90": r["in_mcs_90"], "MCS75": r["in_mcs_75"]}
        for r in NUMBERS["main_results"]
    ]))


main results: skipped until the 900-run grid is complete and the session has time to analyse it.


###

<div style="background: linear-gradient(90deg, #2b0a00, #3d1000); border-left: 3px solid #ff7b54; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #ffb4a2; font-size: 1em; margin: 0;">⚖️ C1 dan C2</h3> <p style="display: inline; color: #ffd8c2; font-size: 0.9em; margin: 0;">· Selisih RelMSE berpasangan per K dan uji keluarga cross-model.</p></div>

In [37]:
if not ANALYSIS_READY:
    print("C1 and C2: skipped until the 900-run grid is complete and the session has time to analyse it.")
else:
    print(pl.DataFrame(NUMBERS["contrasts"]).select(
        "claim", "left", "right", "mean_diff", "se", "ci_low", "ci_high", "n_origins", "left_better"))
    print("mean_diff = RelMSE(iTransformer) - RelMSE(other); negative means the iTransformer is better.")
    print(pl.DataFrame(NUMBERS["comparisons"]["pairs"]).filter(pl.col("family") == "cross-model").select(
        "left", "right", "t_cluster", "p_raw", "p_romano_wolf_family", "T_min"))


C1 and C2: skipped until the 900-run grid is complete and the session has time to analyse it.


###

<div style="background: linear-gradient(90deg, #150029, #240046); border-left: 3px solid #9d4edd; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #e0aaff; font-size: 1em; margin: 0;">🔢 RQ1 dan RQ2</h3> <p style="display: inline; color: #c8a2e0; font-size: 0.9em; margin: 0;">· Efek rung, TOST 8→12, uji-J K lawan K_eff; β₁ gap K1–K8 menurut umur model dengan MDE.</p></div>

In [38]:
if not ANALYSIS_READY:
    print("RQ1 and RQ2: skipped until the 900-run grid is complete and the session has time to analyse it.")
else:
    rq1, rq2 = NUMBERS["rq1"], NUMBERS["rq2"]
    print(pl.DataFrame(rq1["rung_effects"]))
    print(f"RelMSE change K4 to K8 {rq1['delta_4_to_8']:+.6f}; K8 to K12 {rq1['delta_8_to_12']:+.6f}")
    print(rq1["tost"])
    print(f"J test: K plus K_eff p = {rq1['j_test_k_augmented_by_keff']['p']:.4f}; "
          f"K_eff plus K p = {rq1['j_test_keff_augmented_by_k']['p']:.4f}")
    print(f"\nRQ2: beta1 {rq2['beta1']:+.6f} (cluster SE {rq2['cluster_se']:.6f}, G = {rq2['G']}); "
          f"wild bootstrap p {rq2['p_rademacher']:.4f} (Rademacher), {rq2['p_webb']:.4f} (Webb); "
          f"minimum detectable slope {rq2['minimum_detectable_beta1']:+.6f}")
    print(pl.DataFrame(rq2["stride5"]))
    print(f"coverage covariate: {rq2['coverage_covariate']}")


RQ1 and RQ2: skipped until the 900-run grid is complete and the session has time to analyse it.


###

<div style="background: linear-gradient(90deg, #001a1a, #002b2b); border-left: 3px solid #48cae4; border-radius: 6px; padding: 6px 14px;"><h3 style="display: inline; color: #90e0ef; font-size: 1em; margin: 0;">🧭 Akurasi arah</h3> <p style="display: inline; color: #ade8f4; font-size: 0.9em; margin: 0;">· DA-1h, DA-24h, dan DA-kumulatif dikurangi baseline mayoritas per origin.</p></div>

In [39]:
if not ANALYSIS_READY:
    print("directional accuracy: skipped until the 900-run grid is complete and the session has time to analyse it.")
else:
    print(pl.DataFrame(NUMBERS["directional_accuracy"]).select(
        "model", "k", "n_origins",
        *[f"{part}_{v}" for v in ("1h", "24h", "cum") for part in ("dda", "wins")]))
    print("dda = DA minus each origin's majority-sign rate, from test-period frequencies, so the "
          "comparison is conservative; Pesaran-Timmermann counts (pt_*) are diagnostic only.")


directional accuracy: skipped until the 900-run grid is complete and the session has time to analyse it.


##

<a id="section-11"></a>

<div style="background: linear-gradient(135deg, #001a0d, #003317); border-left: 4px solid #52b788; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #95d5b2; margin: 0 0 4px; font-size: 1.35em;">💾 11 · Simpan hasil, tabel, dan figure</h2>
  <p style="color: #b7e4c7; margin: 0; font-size: 0.95em;">paper_numbers.json, tabel LaTeX, figure, dan panel, semuanya dari prediksi tersimpan.</p>
</div>

<div style="background: #0e0e12; border-left: 3px solid #6c757d; border-radius: 0 6px 6px 0; padding: 4px 14px; font-size: 0.82em; color: #cfcfcf;"><span style="color: #ffd166; font-weight: 600;">MENULIS</span> <code>paper/paper_numbers.json</code> <code>paper/tables/*.tex</code> <code>paper/figures/*.pdf</code> <code>paper/figures/*.png</code> <code>paper/panels/*.parquet</code></div>

In [40]:
if not ANALYSIS_READY:
    print("tables and figures: skipped until the 900-run grid is complete and the session has time to analyse it.")
else:
    PAPER = WORK / "paper"
    PAPER.mkdir(parents=True, exist_ok=True)
    (PAPER / "paper_numbers.json").write_text(json.dumps(NUMBERS, indent=2, default=float), encoding="utf-8")
    for _table in render_tables(NUMBERS, PAPER / "tables"):
        print(f"  {_table.name}")
    render_figures(REPORT, PAPER / "figures", log=print)
    (PAPER / "panels").mkdir(parents=True, exist_ok=True)
    for _name, _frame in (("seed_averaged_cells", REPORT.seed_avg), ("amplification_panel", REPORT.amplification),
                          ("rolling_pr", REPORT.rolling_pr), ("rolling_ols_r2", REPORT.rolling_r2),
                          ("directional_accuracy", REPORT.da_summary)):
        _frame.write_parquet(PAPER / "panels" / f"{_name}.parquet")
    print(f"wrote {PAPER}")


tables and figures: skipped until the 900-run grid is complete and the session has time to analyse it.


##

<a id="section-12"></a>

<div style="background: linear-gradient(135deg, #101010, #1c1c1c); border-left: 4px solid #9e9e9e; border-radius: 8px; padding: 10px 18px;">
  <h2 style="color: #d0d0d0; margin: 0 0 4px; font-size: 1.35em;">🔁 12 · Lampiran — sinkronisasi lokal</h2>
  <p style="color: #a8a8a8; margin: 0; font-size: 0.95em;">Menulis src/ dari sel definisi notebook yang sudah disimpan; nonaktif, hanya untuk checkout lokal.</p>
</div>

In [41]:
# ============================================================================
# SINKRON KE src/ - NONAKTIF. Hapus '# ' pada empat baris terakhir untuk memakai.
# ============================================================================
#
# Menulis src/itransformer_btc/ dari sel definisi notebook yang SUDAH DISIMPAN,
# lalu memeriksa bahwa hasilnya mem-flatten kembali byte-identik; jika tidak,
# berkas dipulihkan. Sel vendor tidak pernah ditulis, hanya dicocokkan.
#
# Syarat: checkout lokal (Kaggle tidak membawa tools/ maupun src/). Impor baru
# ditambahkan di sel Library dan di metadata itbtc.projection_imports sel modulnya.
# Sesudahnya: python tools/build_notebook.py --check, lalu commit src/ dan
# notebook bersama.
#
# import subprocess, sys
# _sync = subprocess.run([sys.executable, "-X", "utf8", "tools/notebook_to_src.py",
#                         "notebooks/btc_walkforward_3model.ipynb"],
#                        capture_output=True, text=True, encoding="utf-8")
# print(_sync.stdout or _sync.stderr)
